In [2]:
"""
ArcFlowText Total-Text Full-Image Text Spotting Implementation
======================================================
This file contains the Total-Text-only full-image text spotting pipeline.
It trains and evaluates a single scene-text spotting system:
    full image -> backbone -> FPN P2/P3/P4 -> DB++ textness -> polygon proposals
    -> ArcFlowAlign -> ArcBridge -> CTC/CE recognizer -> Det/E2E evaluation
Only the Total-Text training and evaluation path is present. The STR recognizer
components are retained because the Total-Text crop recognizer uses
the same OCR head family as the full-image text spotting path.
Default Colab run:
    python arcflowtext_totaltext_camera_ready_v4_vitae_fixed.py
Backbone comparison run, same Total-Text architecture and schedule:
    python arcflowtext_totaltext_camera_ready_v4_vitae_fixed.py --compare_backbones resnet50,vitaev2_s
Dataset tar expected at:
    /content/drive/MyDrive/ArcFlowText/ArcFlowText_DATASET.tar
"""
# =============================================================================
# 0. DEPENDENCIES AND GLOBAL CONTROL PANEL
# =============================================================================
import sys
import argparse
import csv
import gc
import importlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tarfile
import time
import unicodedata
import warnings
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
warnings.filterwarnings("ignore", category=UserWarning)
try:
    import psutil  # type: ignore
except Exception:
    psutil = None  # type: ignore

def ensure_package(import_name: str, pip_name: Optional[str] = None) -> None:
    """Import a package; install it if missing. Colab-safe."""
    try:
        __import__(import_name)
    except Exception:
        pip_name = pip_name or import_name
        print(f"[setup] Installing missing package: {pip_name}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
def apply_numpy_compatibility_aliases() -> None:
    """Restore NumPy aliases expected by older research backbones."""
    alias_map = {
        "int": int,
        "float": float,
        "bool": bool,
        "complex": complex,
        "object": object,
    }
    for alias, typ in alias_map.items():
        if not hasattr(np, alias):
            setattr(np, alias, typ)
def run_shell_command(cmd: Sequence[str], cwd: Optional[str] = None) -> None:
    """Run a shell command with a visible one-line log."""
    print("+ " + " ".join(str(x) for x in cmd), flush=True)
    subprocess.run(list(cmd), cwd=cwd, check=True)
def ensure_vitaev2_environment() -> Any:
    """Prepare the official ViTAEv2-S implementation for dense-feature use.
    ViTAEv2-S is not loaded through the generic timm ``features_only`` path.
    The official repository registers the model name ``ViTAEv2_S`` with timm.
    We import that repository, build the model, and manually expose spatial
    feature maps from the internal stages.
    """
    apply_numpy_compatibility_aliases()
    required_timm = "0.4.12"
    try:
        import timm  # type: ignore
        current_timm = getattr(timm, "__version__", "unknown")
    except Exception:
        current_timm = None
    if current_timm != required_timm:
        print(f"[setup] Installing timm=={required_timm} for the official ViTAEv2-S registry", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"timm=={required_timm}"])
        for mod_name in list(sys.modules.keys()):
            if mod_name == "timm" or mod_name.startswith("timm."):
                sys.modules.pop(mod_name, None)
        importlib.invalidate_caches()
    import timm  # type: ignore
    repo_dir = Path("/content/ViTAE-Transformer")
    code_dir = repo_dir / "Image-Classification"
    if not repo_dir.exists():
        run_shell_command([
            "git", "clone", "--depth", "1",
            "https://github.com/ViTAE-Transformer/ViTAE-Transformer.git",
            str(repo_dir),
        ])
    if not code_dir.exists():
        raise RuntimeError(
            f"ViTAE-Transformer repository was found at {repo_dir}, but {code_dir} is missing."
        )
    if str(code_dir) not in sys.path:
        sys.path.insert(0, str(code_dir))
    # Importing vitaev2 runs the official @register_model decorators.
    import vitaev2  # noqa: F401
    registered = [m for m in timm.list_models() if "ViTAE" in m or "vitae" in m or "VITAE" in m]
    if "ViTAEv2_S" not in registered:
        raise RuntimeError(
            "The official ViTAEv2_S model was not registered with timm. "
            f"Registered ViTAE-like names: {registered}"
        )
    log(f"[model] ViTAEv2-S environment ready | timm={timm.__version__} | registered={registered}")
    return timm
ensure_package("numpy")
ensure_package("PIL", "pillow")
ensure_package("torch")
try:
    ensure_package("regex")
except Exception:
    print("[setup][warn] regex package unavailable; using fallback grapheme splitter.", flush=True)
import numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageFile, ImageFilter
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
try:
    import torchvision
except Exception as e:
    torchvision = None  # type: ignore
    print(f"[setup][warn] torchvision unavailable in this runtime: {repr(e)}", flush=True)
try:
    import regex as regex_lib
except Exception:
    regex_lib = None
try:
    import cv2
except Exception:
    try:
        ensure_package("cv2", "opencv-python-headless")
        import cv2
    except Exception:
        cv2 = None
@dataclass
class CFG:
    # -------------------------------------------------------------------------
    # Paths
    # -------------------------------------------------------------------------
    DRIVE_TAR: str = "/content/drive/MyDrive/ArcFlowText/ArcFlowText_DATASET.tar"
    LOCAL_ROOT: str = "/content/data/ArcFlowText_TotalText_extracted"
    TAR_CACHE_DIR: str = "/content/data/arcflowtext_tar_cache"
    OUTPUT_ROOT: str = "/content/arcflow_runs/ArcFlowText_TotalText_CameraReady"
    DRIVE_SYNC_ROOT: str = "/content/drive/MyDrive/ArcFlowText/ArcFlowText_TotalText_CameraReady"
    # -------------------------------------------------------------------------
    # Total-Text-only training schedule
    # -------------------------------------------------------------------------
    # Default training schedule: 10 crop warm-up epochs and
    # 10 DB++/ArcFlow/ArcBridge full-image epochs. For the final longer paper
    # run, change only these two values.
    EPOCHS_TT_CROP_WARMUP: int = 10
    EPOCHS_TT_DB_ARCFLOW: int = 10
    # Total-Text final geometry/alignment controls.
    ENABLE_ARCFLOW_ALIGN: bool = True
    ARCFLOW_K_POINTS: int = 25
    ARCFLOW_TRANSVERSE_SAMPLES: int = 5
    ARCFLOW_DENOISE_NOISE_STD: float = 0.025
    ARCFLOW_DELTA_SCALE: float = 0.08
    ARCFLOW_MAX_INSTANCES_PER_IMAGE: int = 48
    ARCFLOW_WEIGHT_CURVE: float = 0.50
    ARCFLOW_WEIGHT_BOUNDARY: float = 0.25
    ARCFLOW_WEIGHT_TAU: float = 0.25
    ARCFLOW_WEIGHT_WIDTH: float = 0.10
    ARCFLOW_WEIGHT_ALIGN: float = 0.05
    ARCFLOW_REFINE_PRED_POLYS_AT_EVAL: bool = True
    ARCFLOW_DELAY_RECOGNITION_GEOM_GRAD_EPOCHS: int = 1
    # Total-Text final repair: zero-output ArcBridge feature adapter.
    # It is initialized as an identity path and trained only as a gentle
    # feature-alignment bridge, so it starts as a safe residual path at step 0.
    ENABLE_ARCBRIDGE: bool = True
    ARCBRIDGE_WEIGHT: float = 0.10
    ARCBRIDGE_DELAY_EPOCHS: int = 3
    ARCBRIDGE_HIDDEN: int = 256
    # Small text repair: keep official protocol, but train/evaluate with explicit
    # small/medium/large diagnostics and area-aware positive weighting.
    TT_ENABLE_P2_FPN: bool = True
    TT_ENABLE_MULTI_SCALE_TRAINING: bool = True
    TT_TRAIN_IMAGE_SIZES: Tuple[int, ...] = (640, 736)
    TT_SMALL_TEXT_MIN_SIDE_PX: int = 32
    TT_MEDIUM_TEXT_MIN_SIDE_PX: int = 96
    TT_SMALL_TEXT_POS_BOOST: float = 2.0
    TT_SMALL_TEXT_OVERSAMPLE: bool = True
    TT_SMALL_TEXT_OVERSAMPLE_BOOST: float = 2.0
    # Fail-fast policy for the Total-Text reproduction path.
    NO_FAKE_SUCCESS_SUMMARY: bool = True
    FAIL_IF_TOTALTEXT_OFFICIAL_ZERO_AFTER_FINAL: bool = True
    STOP_IF_TOTALTEXT_FAILS: bool = True
    SHORT_RUN_MODE: bool = True
    # -------------------------------------------------------------------------
    # Reproducibility / runtime
    # -------------------------------------------------------------------------
    SEED: int = 1337
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    CPU_THREADS: int = 2
    # V7.15 T4 paper-full fast profile: target runtime has 1 physical CPU core,
    # 2 logical threads, ~12.7GB RAM and a 15GB Tesla T4. Use both logical threads
    # for data loading while keeping queues bounded; push speed through GPU batches
    # and adaptive fallbacks instead of disabling paper diagnostics.
    AUTO_T4_SAFE_PROFILE: bool = True
    NUM_WORKERS: int = 1
    PIN_MEMORY: bool = False
    PERSISTENT_WORKERS: bool = False
    PREFETCH_FACTOR: int = 1
    DATALOADER_LOG_ON_CREATE: bool = True
    DATALOADER_LOG_LIMIT: int = 40
    USE_TORCH_COMPILE: bool = False
    TORCH_COMPILE_MODE: str = "reduce-overhead"
    # V7/T4 stability: prevent CUDNN from reserving very large alternate workspaces
    # right before validation. This keeps the full validation sweep logic intact,
    # but makes the runtime less likely to disconnect silently on Colab T4.
    CUDNN_BENCHMARK: bool = True
    USE_CHANNELS_LAST: bool = True
    RAM_GUARD_MIN_AVAILABLE_GB: float = 3.00
    VRAM_GUARD_MIN_FREE_GB: float = 1.25
    RAM_CRITICAL_MIN_AVAILABLE_GB: float = 1.25
    VRAM_CRITICAL_MIN_FREE_GB: float = 0.40
    STREAM_EVAL_LOG_EVERY: int = 10
    CROP_EVAL_CLEAN_EVERY: int = 999
    DB_VAL_CLEAN_EVERY: int = 999
    ENABLE_GEOMETRY_GAP_EVAL: bool = True  # V7.12: RAM-safe streaming gap monitor enabled for paper diagnostics.
    STRICT_GAP_EVAL: bool = False  # If gap monitor fails, save partial report and continue final run.
    ENABLE_STREAMING_OFFICIAL_FULL: bool = True
    STORE_STREAM_EVAL_EXAMPLES: bool = False
    AMP_DTYPE: str = "float16"
    SYNC_TO_DRIVE_AT_END: bool = True
    CLEAN_OUTPUT_ROOT: bool = True
    RESUME: bool = False
    ALLOW_UNSAFE_RESUME: bool = False
    STOP_ON_NAN: bool = True
    # -------------------------------------------------------------------------
    # Total-Text dataset configurations
    # -------------------------------------------------------------------------
    REQUIRE_EXPECTED_DATA_COUNTS: bool = False
    # -------------------------------------------------------------------------
    # Total-Text OCR tokenizer configuration. The vocabulary is built from
    # non-ignored Total-Text training transcriptions only.
    # -------------------------------------------------------------------------
    TOKENIZER_INCLUDE_TT_TRAIN_WORDS: bool = True
    VOCAB_MIN_FREQ: int = 1
    EXPECTED_VOCAB_MIN: int = 40
    EXPECTED_VOCAB_MAX: int = 80
    REQUIRE_TOKENIZER_RANGE_CHECK: bool = False
    MAX_TEXT_LEN: int = 34
    TEXT_NORM_LOWER_ASCII: bool = True
    # -------------------------------------------------------------------------
    # OCR crop preprocessing / loader compatibility settings
    # -------------------------------------------------------------------------
    STR_HEIGHT: int = 64
    OCR_FIXED_W: int = 320
    # v5: dataset-calibrated OCR crop width. Total-Text/ICDAR stay compact;
    # CTW1500 gets wider crops because its annotations are curved text-lines / phrases.
    DEFAULT_OCR_FIXED_W: int = 320
    CTW_OCR_FIXED_W: int = 640
    DEFAULT_CROP_PAD_FRAC: float = 0.06
    CTW_GT_CROP_PAD_FRAC: float = 0.12
    CTW_EVAL_POLY_EXPAND_PAD: float = 1.22
    OFFICIAL_LEXICON_MAX_ABS_EDIT: int = 2
    OFFICIAL_CTW_FULL_NED_THRESH: float = 0.30
    STR_USE_MULTI_WIDTH_BUCKETS: bool = True
    STR_WIDTH_BUCKETS: Tuple[int, ...] = (160, 224, 320)
    STR_TRAIN_RANDOM_BUCKET: bool = True
    # -------------------------------------------------------------------------
    # OCR recognizer architecture configurations
    # -------------------------------------------------------------------------
    BACKBONE_NAME: str = "resnet50"
    PRETRAIN_IMAGENET: bool = True
    D_MODEL: int = 256
    TRANSFORMER_HEADS: int = 8
    STR_TRANSFORMER_LAYERS: int = 4
    STR_CE_DECODER_LAYERS: int = 2
    DROPOUT: float = 0.15
    STR_USE_DOMAIN_CONDITIONING: bool = True
    # -------------------------------------------------------------------------
    # OCR recognizer loss / decode schedule
    # -------------------------------------------------------------------------
    STR_CE_SELECT_START_EPOCH: int = 9
    STR_SCHEDULE_END_EPOCH: int = 20
    STR_CTC_WEIGHT_START: float = 1.00
    STR_CTC_WEIGHT_END: float = 0.60
    STR_CE_WEIGHT_START: float = 0.20
    STR_CE_WEIGHT_END: float = 0.85
    STR_SEM_WEIGHT_START: float = 0.00
    STR_SEM_WEIGHT_END: float = 0.08
    STR_DOMAIN_WEIGHT: float = 0.05
    OCR_CE_LABEL_SMOOTHING: float = 0.05
    STR_SEMANTIC_STOPGRAD_CE: bool = True
    OCR_SEM_TEACHER: str = "ce"
    OCR_DECODE_MODE: str = "auto"
    STR_SELECT_CE_CONF_RATIO: float = 0.70
    STR_SELECT_MAX_LEN_RATIO: float = 1.85
    # -------------------------------------------------------------------------
    # Optimizer / checkpoint / monitoring
    # -------------------------------------------------------------------------
    LR_BACKBONE: float = 2e-5
    LR_STR: float = 2e-4
    WEIGHT_DECAY: float = 5e-2
    MAX_GRAD_NORM: float = 1.0
    USE_EMA: bool = True
    EMA_DECAY: float = 0.997
    SAVE_LAST_EVERY_EPOCH: bool = True
    LOGS_PER_EPOCH: int = 10
    MIN_LOG_STEPS: int = 10
    PRINT_GPU_EVERY_LOG: bool = True
    DEBUG_EXAMPLE_COUNT: int = 40
    TT_FIXED_VIS_IDS: Tuple[int, ...] = tuple(range(230, 240))
    VIS_EPOCHS: Tuple[int, ...] = (1, 5, 10)
    # -------------------------------------------------------------------------
    # Total-Text: crop recognizer + DB++ dense textness detector
    # -------------------------------------------------------------------------
    TOTALTEXT_IMG_SIZE: int = 640
    TOTALTEXT_POLY_POINTS: int = 16
    TOTALTEXT_MAX_INSTANCES: int = 100
    EXPECTED_TT_TRAIN_IMAGES: int = 1255
    EXPECTED_TT_TEST_IMAGES: int = 300
    EXPECTED_TT_TRAIN_VALID_INSTANCES: int = 9289
    EXPECTED_TT_TEST_VALID_INSTANCES: int = 2212
    TT_CROP_BATCH_SIZE: int = 160
    TT_CROP_EVAL_BATCH_SIZE: int = 256
    TT_CROP_MODES: Tuple[str, ...] = ("rectified",)
    # Full-image E2E recognizes rectified predicted polygons, so the crop warm-up is
    # restricted to rectified crops for this quick reproduction because full-image E2E recognizes rectified predicted polygons; bbox/masked robustness is reserved for the full 35/50 epoch paper run if needed.
    TT_CROP_MODE_WEIGHTS: Tuple[float, ...] = (1.0,)
    TT_CROP_BEST_METRIC: str = "rectified_WA"  # rectified_WA aligns with final predicted-polygon OCR path
    TT_CROP_EVAL_RECTIFIED_ONLY: bool = True
    TT_CROP_DECODE_MODE: str = "ctc"
    TT_CROP_MIN_WA_HEALTH: float = 20.0
    FAIL_ON_TT_CROP_WEAK: bool = False
    TT_DB_BATCH_SIZE: int = 3
    # V7.18: keep DB training batch conservative because backward pass peaks high on T4;
    # use moderately large eval/inference batches because CPU postprocess/RAM, not VRAM, is the bottleneck.
    TT_DB_EVAL_BATCH_SIZE: int = 24
    # V7: restore v4-style detector supervision by default. Ignore regions are still
    # respected in OCR/evaluation; hard-neutral DB loss is available as an ablation only.
    DB_IGNORE_MODE: str = "v4_default"  # choices: v4_default, soft_neutral, hard_neutral
    DB_IGNORE_SOFT_WEIGHT: float = 0.05
    GRAD_ACCUM_STEPS: int = 1
    TT_DB_LR_BACKBONE: float = 8e-5
    TT_DB_LR_HEAD: float = 8e-4
    TT_DB_WEIGHT_DECAY: float = 1e-4
    TT_DB_POS_WEIGHT: float = 6.0
    TT_DB_BCE_WEIGHT: float = 1.0
    TT_DB_DICE_WEIGHT: float = 1.0
    TT_DB_FOCAL_WEIGHT: float = 0.35
    TT_DB_FOCAL_ALPHA: float = 0.25
    TT_DB_FOCAL_GAMMA: float = 2.0
    TT_DB_MASK_SHRINK_RATIO: float = 0.58
    TT_DB_INTERNAL_VAL_IMAGES: int = 50
    TT_DB_SWEEP_THRESHOLDS: Tuple[float, ...] = (0.65, 0.70, 0.75, 0.80)
    TT_DB_SWEEP_UNSHRINK: Tuple[float, ...] = (1.35, 1.55, 1.75)
    TT_DB_SWEEP_MODES: Tuple[str, ...] = ("minrect", "contour")
    TT_DB_SWEEP_MIN_AREAS: Tuple[int, ...] = (24, 32, 48, 64)
    # Geometry-aware geometry/alignment sweep additions. These do not change DB++ learning;
    # they improve polygon extraction, small-text recall, and rectified crop alignment.
    TT_DB_GEOM_SWEEP_THRESHOLDS: Tuple[float, ...] = (0.65, 0.70, 0.75, 0.80)
    TT_DB_GEOM_SWEEP_UNSHRINK: Tuple[float, ...] = (1.55, 1.75)
    TT_DB_GEOM_SWEEP_MODES: Tuple[str, ...] = ("contour", "minrect")
    TT_DB_GEOM_SWEEP_MIN_AREAS: Tuple[int, ...] = (24, 32, 48, 64)
    TT_DB_GEOM_SWEEP_MORPH: Tuple[str, ...] = ("none", "close")
    TT_DB_GEOM_SWEEP_CROP_PAD: Tuple[float, ...] = (1.00, 1.08, 1.15)
    TT_DB_GEOM_TOPK_E2E: int = 8
    TT_DB_GEOM_E2E_BETA: float = 0.35
    # v6: use train-internal validation E2E on the top-K geometry settings.
    # This never touches official test data and only selects threshold/unshrink/crop_pad.
    TT_DB_ENABLE_E2E_SELECTION: bool = True
    TT_DB_E2E_SELECTION_FINAL_ONLY: bool = True
    TT_DB_E2E_SELECTION_EVERY: int = 999
    # V7: keep the same validation sweep logic, but speed it through caching and
    # bbox rejection inside matching. Do not reduce grid/image coverage.
    ENABLE_VALIDATION_CACHES: bool = True
    # V7.1: keep sweep logic identical, but avoid unbounded CTW pred-cache RAM blow-up.
    CTW_DISABLE_DET_SWEEP_CACHE: bool = True
    SWEEP_PROGRESS_EVERY: int = 50
    # V7: optional final OCR adaptation to predicted crops. Validation-gated, detector frozen.
    ENABLE_PRED_CROP_ADAPT: bool = True
    # V7.5: paper visualizations are useful but should not dominate fast debug runs.
    ENABLE_PAPER_VISUALS: bool = True
    PRED_CROP_ADAPT_EPOCHS: int = 2
    PRED_CROP_ADAPT_MAX_IMAGES: int = 0  # 0 means use all available train/internal-val images; streaming builder prevents RAM blow-up.
    # Turbo-safe PredCropAdapt: keep the stage enabled, but batch detector inference
    # aggressively during no-grad stages. OOM fallback recursively splits the batch.
    PRED_CROP_ADAPT_BUILDER_BATCH_SIZE: int = 24
    PRED_CROP_ADAPT_BUILD_LOG_EVERY: int = 25
    PRED_CROP_ADAPT_BUILD_CLEAN_EVERY: int = 999
    PRED_CROP_ADAPT_TRAIN_BATCH_CAP: int = 192
    FUSE_GAP_WITH_OFFICIAL_EVAL: bool = True
    PRED_CROP_ADAPT_IOU: float = 0.50
    PRED_CROP_ADAPT_LR_FACTOR: float = 0.50
    TT_CROP_RECTIFY_PAD: float = 1.06
    TT_DB_MAX_PREDS: int = 100
    TT_DB_DEFAULT_THRESHOLD: float = 0.65
    TT_DB_DEFAULT_UNSHRINK: float = 1.75
    TT_DB_DEFAULT_MODE: str = "contour"
    TT_DB_DEFAULT_MIN_AREA: int = 64
    TT_MATCH_IOU: float = 0.50
    TT_IGNORE_IOU: float = 0.50
    # E2E/OCR evaluation is expensive because each predicted polygon must be cropped,
    # rectified, recognized, and then matched. Keep detection validation every epoch,
    # but run validation E2E only every N DB epochs plus the final epoch.
    # For a 50-epoch run this gives epochs 10/20/30/40/50.
    TT_DB_E2E_EVAL_INTERVAL: int = 1
    TT_DB_E2E_EVAL_FINAL_EPOCH: bool = True
    # Longer-run safety: train normally, but evaluate final official metrics using
    # the best train-internal validation checkpoints. This avoids reporting a late
    # overfitted epoch when a better checkpoint was already saved. It does not tune
    # on official test data and does not change the architecture.
    USE_BEST_TT_CROP_CHECKPOINT_FOR_DB: bool = True
    USE_BEST_DB_CHECKPOINT_FOR_FINAL_TEST: bool = True
    TOTALTEXT_TARGET_DETF: float = 38.0
    TOTALTEXT_TARGET_E2E: float = 13.0
    TOTALTEXT_TARGET_GT_E2E: float = 40.0
    # Fast confirmation fast confirmation controls: keep the same architecture, but reduce
    # wall-cconfiguration by using larger crop batches, rectified-only crop training,
    # a smaller internal validation split, and cached postprocess settings on
    # non-sweep epochs. These are monitoring/training-schedule choices, not
    # architecture changes.
    TT_DB_POSTPROCESS_SWEEP_EVERY: int = 2
    TT_DB_FORCE_SWEEP_EPOCHS: Tuple[int, ...] = (1, 5, 10)
    TOTALTEXT_ARCH_TARGET_DETF: float = 38.0
    TOTALTEXT_ARCH_TARGET_E2E: float = 10.0
    TOTALTEXT_ARCH_TARGET_GT_E2E: float = 25.0
    TT_CROP_TARGET_RECT_WA: float = 30.0
    TT_CROP_TARGET_RECT_E1: float = 55.0
    TOTALTEXT_DETF_HEALTH_EPOCH: int = 4
    TOTALTEXT_MIN_DETF_AFTER_HEALTH_EPOCH: float = 1.0
    FAIL_FAST_TOTALTEXT_LOW_DETF: bool = False
    TOTALTEXT_TARGET_DETF_HEALTH: float = 10.0
    TOTALTEXT_TARGET_E2E_HEALTH: float = 0.0
    # -------------------------------------------------------------------------
    ARCH_VERSION: str = "totaltext_fullimage_10crop_10db_arcflow_arcbridge_vitae_official"
cfg = CFG()
TOTAL_TEXT_DOMAIN = "total_text_latin"
TOTAL_TEXT_DOMAIN_ID = 0
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IGNORE_TEXTS = {"#", "###", "####", "*", "", "do not care", "dontcare", "don't care"}
# =============================================================================
# 1. LOGGING, SEEDING, IO, AND HARDWARE CHECKS
# =============================================================================
def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")
def log(msg: str) -> None:
    print(f"[{now()}] {msg}", flush=True)
def section(title: str) -> None:
    print("\n" + "=" * 110, flush=True)
    print(title, flush=True)
    print("=" * 110, flush=True)
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = bool(getattr(cfg, "CUDNN_BENCHMARK", False))
    try:
        torch.set_num_threads(max(1, int(cfg.CPU_THREADS)))
        torch.set_num_interop_threads(1)
    except Exception:
        pass
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
    # OpenCV contour extraction dominates validation sweeps and is CPU-bound.
    # Keep the same sweep logic, but let OpenCV use the two available CPU threads.
    try:
        if cv2 is not None:
            cv2.setUseOptimized(True)
            cv2.setNumThreads(max(1, min(2, int(getattr(cfg, "CPU_THREADS", 2)))))
    except Exception:
        pass
def get_amp_dtype() -> torch.dtype:
    if cfg.DEVICE == "cuda" and str(cfg.AMP_DTYPE).lower() == "float16":
        return torch.float16
    if cfg.DEVICE == "cuda" and str(cfg.AMP_DTYPE).lower() == "bfloat16":
        return torch.bfloat16
    return torch.float32
def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)
def write_csv(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    keys: List[str] = []
    seen = set()
    for r in rows:
        for k in r.keys():
            if k not in seen:
                keys.append(k)
                seen.add(k)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)
def _cpu_memory_snapshot() -> Dict[str, float]:
    """Return CPU RAM stats in GB. Works even if psutil is unavailable."""
    if psutil is not None:
        vm = psutil.virtual_memory()
        return {
            "total_gb": float(vm.total) / (1024.0 ** 3),
            "available_gb": float(vm.available) / (1024.0 ** 3),
            "used_gb": float(vm.used) / (1024.0 ** 3),
            "used_pct": float(vm.percent),
        }
    return {"total_gb": 0.0, "available_gb": 0.0, "used_gb": 0.0, "used_pct": 0.0}


def _gpu_memory_snapshot() -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"total_gb": 0.0, "free_gb": 0.0, "allocated_gb": 0.0, "reserved_gb": 0.0, "peak_allocated_gb": 0.0}
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "total_gb": float(total_b) / (1024.0 ** 3),
        "free_gb": float(free_b) / (1024.0 ** 3),
        "allocated_gb": float(torch.cuda.memory_allocated()) / (1024.0 ** 3),
        "reserved_gb": float(torch.cuda.memory_reserved()) / (1024.0 ** 3),
        "peak_allocated_gb": float(torch.cuda.max_memory_allocated()) / (1024.0 ** 3),
    }


def runtime_profile_snapshot() -> Dict[str, Any]:
    gpu_name = None
    capability = None
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        gpu_name = props.name
        capability = f"{props.major}.{props.minor}"
    return {
        "cpu": _cpu_memory_snapshot(),
        "gpu": _gpu_memory_snapshot(),
        "gpu_name": gpu_name,
        "cuda_capability": capability,
        "num_workers": int(getattr(cfg, "NUM_WORKERS", 0)),
        "pin_memory": bool(getattr(cfg, "PIN_MEMORY", False)),
        "persistent_workers": bool(getattr(cfg, "PERSISTENT_WORKERS", False)),
        "prefetch_factor": int(getattr(cfg, "PREFETCH_FACTOR", 0)),
        "db_batch_size": int(getattr(cfg, "TT_DB_BATCH_SIZE", 0)),
        "db_eval_batch_size": int(getattr(cfg, "TT_DB_EVAL_BATCH_SIZE", 0)),
        "crop_batch_size": int(getattr(cfg, "TT_CROP_BATCH_SIZE", 0)),
        "crop_eval_batch_size": int(getattr(cfg, "TT_CROP_EVAL_BATCH_SIZE", 0)),
        "pred_crop_builder_batch_size": int(getattr(cfg, "PRED_CROP_ADAPT_BUILDER_BATCH_SIZE", 0)),
        "pred_crop_train_batch_cap": int(getattr(cfg, "PRED_CROP_ADAPT_TRAIN_BATCH_CAP", 0)),
        "grad_accum_steps": int(getattr(cfg, "GRAD_ACCUM_STEPS", 1)),
        "amp_dtype": str(getattr(cfg, "AMP_DTYPE", "float16")),
        "dataset": current_dataset_key() if "current_dataset_key" in globals() else None,
    }


def clean_memory(aggressive: bool = False) -> None:
    """Systematic CPU/GPU cleanup.

    Non-aggressive mode avoids constantly flushing CUDA's caching allocator because
    that hurts throughput. Aggressive mode is used at phase boundaries and after OOM
    recovery, where stability matters more than allocator reuse.
    """
    gc.collect()
    if torch.cuda.is_available():
        try:
            if aggressive:
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
            else:
                free_b, _ = torch.cuda.mem_get_info()
                free_gb = float(free_b) / (1024.0 ** 3)
                if free_gb < float(getattr(cfg, "VRAM_GUARD_MIN_FREE_GB", 1.25)):
                    torch.cuda.empty_cache()
        except Exception:
            pass


def gpu_mem(prefix: str = "gpu") -> str:
    cpu = _cpu_memory_snapshot()
    if not torch.cuda.is_available():
        return f"[{prefix}] cuda unavailable | CPU available={cpu['available_gb']:.2f}GB used={cpu['used_pct']:.1f}%"
    g = _gpu_memory_snapshot()
    return (
        f"[{prefix}] VRAM allocated={g['allocated_gb']:.2f}GB reserved={g['reserved_gb']:.2f}GB "
        f"free={g['free_gb']:.2f}GB peak={g['peak_allocated_gb']:.2f}GB / {g['total_gb']:.2f}GB | "
        f"CPU available={cpu['available_gb']:.2f}GB used={cpu['used_pct']:.1f}%"
    )


def optimize_image_tensor_for_cuda(x: torch.Tensor) -> torch.Tensor:
    if isinstance(x, torch.Tensor) and x.ndim == 4 and cfg.DEVICE == "cuda" and bool(getattr(cfg, "USE_CHANNELS_LAST", True)):
        try:
            return x.contiguous(memory_format=torch.channels_last)
        except Exception:
            return x
    return x


def is_cuda_oom(exc: BaseException) -> bool:
    msg = str(exc).lower()
    return isinstance(exc, RuntimeError) and ("out of memory" in msg or "cuda error: out of memory" in msg or "cublas_status_alloc_failed" in msg)


def recover_from_oom(tag: str) -> None:
    log(f"[oom-guard][{tag}] CUDA/RAM pressure detected; clearing stale tensors and CUDA cache")
    clean_memory(aggressive=True)
    log(gpu_mem(f"oom-guard:{tag}"))


def memory_safety_barrier(tag: str, aggressive: bool = False, raise_on_critical: bool = False) -> None:
    """Reusable CPU+GPU sanity check used around phase boundaries and evaluations."""
    acted = False
    parts: List[str] = []
    cpu = _cpu_memory_snapshot()
    if cpu.get("available_gb", 0.0) and cpu["available_gb"] < float(getattr(cfg, "RAM_GUARD_MIN_AVAILABLE_GB", 2.0)):
        gc.collect()
        acted = True
        parts.append(f"availRAM={cpu['available_gb']:.2f}GB")
    if torch.cuda.is_available():
        try:
            g = _gpu_memory_snapshot()
            if g["free_gb"] < float(getattr(cfg, "VRAM_GUARD_MIN_FREE_GB", 1.25)) or aggressive:
                torch.cuda.empty_cache()
                acted = True
                parts.append(f"freeVRAM={g['free_gb']:.2f}GB")
        except Exception:
            pass
    if acted or aggressive:
        log(f"[mem-guard][{tag}] cleanup " + " ".join(parts))
        log(gpu_mem(f"mem-guard:{tag}"))
    if raise_on_critical:
        cpu_after = _cpu_memory_snapshot()
        if cpu_after.get("available_gb", 0.0) and cpu_after["available_gb"] < float(getattr(cfg, "RAM_CRITICAL_MIN_AVAILABLE_GB", 0.75)):
            raise RuntimeError(f"Critical CPU RAM before {tag}: available={cpu_after['available_gb']:.2f}GB")
        if torch.cuda.is_available():
            g_after = _gpu_memory_snapshot()
            if g_after["free_gb"] < float(getattr(cfg, "VRAM_CRITICAL_MIN_FREE_GB", 0.40)):
                raise RuntimeError(f"Critical GPU VRAM before {tag}: free={g_after['free_gb']:.2f}GB")


def apply_t4_runtime_profile(dataset_key: Optional[str] = None) -> None:
    """Hardware-aware profile for Colab Simple T4 without changing model/loss logic.

    V7.18 correction: the previous max-throughput profile was GPU-aggressive but
    RAM-hostile on Colab Simple T4. The logs showed CPU available RAM dropping
    below 0.6GB during CTW crop warm-up while the GPU still had free memory.
    Therefore the bottleneck is CPU RAM / worker queues / pinned-memory pressure,
    not model capacity. This profile keeps every modeling/eval stage enabled, but
    uses RAM-aware loader settings and medium-large inference batches that are
    faster than serial streaming without starving/crashing the 12.7GB host RAM.
    """
    if not bool(getattr(cfg, "AUTO_T4_SAFE_PROFILE", True)):
        return
    gpu_name = ""
    if torch.cuda.is_available():
        gpu_name = str(torch.cuda.get_device_properties(0).name).lower()
    cpu = _cpu_memory_snapshot()
    low_ram = (cpu.get("total_gb", 99.0) > 0.0 and cpu.get("total_gb", 99.0) <= 14.0)
    low_cpu = (os.cpu_count() or 1) <= 2
    is_t4 = "t4" in gpu_name
    if not (is_t4 or low_ram or low_cpu):
        return

    ds_key = str(dataset_key or current_dataset_key()).lower()
    avail = float(cpu.get("available_gb", 99.0) or 99.0)

    cfg.CPU_THREADS = 2
    # Critical RAM fix: persistent workers + pinned-memory queues were the source
    # of the slow RAM death across sequential datasets. Keep worker count small,
    # queues shallow, and do not pin host pages on a 12.7GB runtime.
    cfg.PERSISTENT_WORKERS = False
    cfg.PREFETCH_FACTOR = 1
    cfg.PIN_MEMORY = False
    # Use 2 workers only when there is enough host RAM and the crop width is not CTW-wide.
    # Otherwise 1 worker is faster than a crash and avoids duplicated worker queues.
    if ds_key == "ctw1500" or avail < 5.5:
        cfg.NUM_WORKERS = 1
    else:
        cfg.NUM_WORKERS = min(max(int(getattr(cfg, "NUM_WORKERS", 1)), 1), 2)

    cfg.RAM_GUARD_MIN_AVAILABLE_GB = max(float(getattr(cfg, "RAM_GUARD_MIN_AVAILABLE_GB", 3.0)), 3.0)
    cfg.RAM_CRITICAL_MIN_AVAILABLE_GB = max(float(getattr(cfg, "RAM_CRITICAL_MIN_AVAILABLE_GB", 1.25)), 1.25)
    cfg.VRAM_GUARD_MIN_FREE_GB = max(float(getattr(cfg, "VRAM_GUARD_MIN_FREE_GB", 1.25)), 1.25)

    # Medium-large batches are empirically faster than batch=1/3 but safer than 64:
    # postprocess/OCR is CPU-heavy, so huge image batches increase RAM without improving throughput.
    if ds_key == "ctw1500":
        cfg.TT_DB_BATCH_SIZE = min(int(getattr(cfg, "TT_DB_BATCH_SIZE", 2)), 2)
        cfg.GRAD_ACCUM_STEPS = max(int(getattr(cfg, "GRAD_ACCUM_STEPS", 1)), 2)
        cfg.TT_DB_EVAL_BATCH_SIZE = min(max(int(getattr(cfg, "TT_DB_EVAL_BATCH_SIZE", 16)), 8), 16)
        cfg.PRED_CROP_ADAPT_BUILDER_BATCH_SIZE = min(max(int(getattr(cfg, "PRED_CROP_ADAPT_BUILDER_BATCH_SIZE", 12)), 8), 12)
        cfg.TT_CROP_EVAL_BATCH_SIZE = min(max(int(getattr(cfg, "TT_CROP_EVAL_BATCH_SIZE", 128)), 64), 128)
        cfg.PRED_CROP_ADAPT_TRAIN_BATCH_CAP = min(max(int(getattr(cfg, "PRED_CROP_ADAPT_TRAIN_BATCH_CAP", 64)), 48), 64)
    else:
        cfg.TT_DB_BATCH_SIZE = min(int(getattr(cfg, "TT_DB_BATCH_SIZE", 3)), 3)
        cfg.TT_DB_EVAL_BATCH_SIZE = min(max(int(getattr(cfg, "TT_DB_EVAL_BATCH_SIZE", 24)), 12), 24)
        cfg.PRED_CROP_ADAPT_BUILDER_BATCH_SIZE = min(max(int(getattr(cfg, "PRED_CROP_ADAPT_BUILDER_BATCH_SIZE", 24)), 12), 24)
        cfg.TT_CROP_EVAL_BATCH_SIZE = min(max(int(getattr(cfg, "TT_CROP_EVAL_BATCH_SIZE", 256)), 128), 256)
        cfg.PRED_CROP_ADAPT_TRAIN_BATCH_CAP = min(max(int(getattr(cfg, "PRED_CROP_ADAPT_TRAIN_BATCH_CAP", 192)), 128), 192)

    cfg.CROP_EVAL_CLEAN_EVERY = max(int(getattr(cfg, "CROP_EVAL_CLEAN_EVERY", 999)), 999)
    cfg.DB_VAL_CLEAN_EVERY = max(int(getattr(cfg, "DB_VAL_CLEAN_EVERY", 999)), 999)
    cfg.PRED_CROP_ADAPT_BUILD_CLEAN_EVERY = max(int(getattr(cfg, "PRED_CROP_ADAPT_BUILD_CLEAN_EVERY", 999)), 999)
    log(f"[runtime-profile] T4/RAM-stable throughput profile active for {dataset_key or current_dataset_key()} | " + gpu_mem("profile"))

def print_hardware() -> None:
    section("HARDWARE / RUNTIME CHECK")
    log(f"Python       : {sys.version.split()[0]}")
    log(f"PyTorch      : {torch.__version__}")
    log(f"Torchvision  : {getattr(torchvision, '__version__', 'unavailable')}")
    log(f"Device       : {cfg.DEVICE}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        log(f"GPU          : {props.name}")
        log(f"VRAM         : {props.total_memory / 1024 ** 3:.2f} GB")
        log(f"Capability   : {props.major}.{props.minor}")
        log(f"AMP dtype    : {get_amp_dtype()}")
    log(f"CPU count    : {os.cpu_count()}")
    log(f"Output root  : {cfg.OUTPUT_ROOT}")
    log(f"Arch version : {cfg.ARCH_VERSION}")
def maybe_mount_drive() -> None:
    if not str(cfg.DRIVE_TAR).startswith("/content/drive"):
        return
    if Path("/content/drive/MyDrive").exists():
        return
    try:
        from google.colab import drive  # type: ignore
        log("[drive] Mounting Google Drive...")
        drive.mount("/content/drive")
    except Exception as e:
        log(f"[drive][warn] Drive auto-mount failed: {repr(e)}")
# =============================================================================
# 2. DATA LOADING AND DATASET DISCOVERY
# =============================================================================
def safe_extract_tar(tar_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:*") as tar:
        members = tar.getmembers()
        dest_resolved = dest.resolve()
        for m in members:
            target = (dest / m.name).resolve()
            if not str(target).startswith(str(dest_resolved)):
                raise RuntimeError(f"Unsafe tar path blocked: {m.name}")
        log(f"[extract] extracting {len(members)} members to {dest}")
        tar.extractall(dest)
def copy_and_extract_dataset() -> Path:
    section("DATASET IMPORT / EXTRACTION")
    maybe_mount_drive()
    drive_tar = Path(cfg.DRIVE_TAR)
    if not drive_tar.exists():
        raise FileNotFoundError(f"Dataset tar not found: {drive_tar}")
    local_root = Path(cfg.LOCAL_ROOT)
    marker = local_root / ".extraction_complete"
    if marker.exists():
        log(f"[extract] using existing extracted dataset: {local_root}")
        return local_root
    cache_dir = Path(cfg.TAR_CACHE_DIR)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cached_tar = cache_dir / drive_tar.name
    if not cached_tar.exists() or cached_tar.stat().st_size != drive_tar.stat().st_size:
        log(f"[copy] copying tar to local disk")
        log(f"       src: {drive_tar}")
        log(f"       dst: {cached_tar}")
        t0 = time.time()
        shutil.copy2(drive_tar, cached_tar)
        log(f"[copy] done in {time.time() - t0:.1f}s | size={cached_tar.stat().st_size / 1024 ** 3:.2f}GB")
    else:
        log(f"[copy] local tar cache valid: {cached_tar}")
    if local_root.exists():
        shutil.rmtree(local_root)
    t0 = time.time()
    safe_extract_tar(cached_tar, local_root)
    marker.write_text("ok", encoding="utf-8")
    log(f"[extract] complete in {(time.time() - t0) / 60:.1f}min")
    return local_root
def recursive_find(root: Path, predicate) -> List[Path]:
    out: List[Path] = []
    for p in root.rglob("*"):
        try:
            if predicate(p):
                out.append(p)
        except Exception:
            pass
    return out
def _looks_like_totaltext_path(p: Path) -> bool:
    s = str(p).lower().replace("_", "-").replace(" ", "-")
    return "total-text" in s or ("total" in s and "text" in s)
def discover_dataset_paths(local_root: Path) -> Dict[str, Path]:
    """Discover only the Total-Text folders inside the ArcFlowText dataset tar."""
    section("TOTAL-TEXT DATASET DISCOVERY")
    paths: Dict[str, Path] = {}
    tt_train_dirs = recursive_find(local_root, lambda p: p.is_dir() and p.name.lower() == "train" and _looks_like_totaltext_path(p))
    tt_test_dirs = recursive_find(local_root, lambda p: p.is_dir() and p.name.lower() == "test" and _looks_like_totaltext_path(p))
    tt_train_img_dirs = [p for p in tt_train_dirs if any(x.suffix.lower() in IMAGE_EXTS for x in p.iterdir())]
    tt_test_img_dirs = [p for p in tt_test_dirs if any(x.suffix.lower() in IMAGE_EXTS for x in p.iterdir())]
    if not tt_train_img_dirs or not tt_test_img_dirs:
        raise FileNotFoundError("Could not find Total-Text Train/Test image folders inside the extracted tar.")
    paths["tt_train_img"] = tt_train_img_dirs[0]
    paths["tt_test_img"] = tt_test_img_dirs[0]
    poly_train_dirs = recursive_find(
        local_root,
        lambda p: p.is_dir()
        and p.name.lower() == "train"
        and "groundtruth_polygonal_annotation" in str(p).lower()
        and _looks_like_totaltext_path(p),
    )
    poly_test_dirs = recursive_find(
        local_root,
        lambda p: p.is_dir()
        and p.name.lower() == "test"
        and "groundtruth_polygonal_annotation" in str(p).lower()
        and _looks_like_totaltext_path(p),
    )
    if not poly_train_dirs or not poly_test_dirs:
        raise FileNotFoundError("Could not find Total-Text polygon annotation Train/Test folders inside the extracted tar.")
    paths["tt_train_poly"] = poly_train_dirs[0]
    paths["tt_test_poly"] = poly_test_dirs[0]
    for k, v in paths.items():
        log(f"[path] {k:14s}: {v}")
    return paths
# =============================================================================
# 3. SHARED TEXT, TOKENIZER, RECORDS, AND EVALUATION UTILITIES
# =============================================================================
def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", str(s))
def split_graphemes(text: str) -> List[str]:
    text = nfc(text)
    if regex_lib is not None:
        return regex_lib.findall(r"\X", text)
    out: List[str] = []
    for ch in text:
        if not out or unicodedata.combining(ch) == 0:
            out.append(ch)
        else:
            out[-1] += ch
    return out
def normalize_for_eval(s: str, ascii_lower: bool = True) -> str:
    s = nfc(str(s)).strip()
    s = " ".join(s.split())
    if ascii_lower:
        s = "".join(ch.lower() if "A" <= ch <= "Z" else ch for ch in s)
    return s
def normalize_alnum_for_eval(s: str, ascii_lower: bool = True) -> str:
    s = normalize_for_eval(s, ascii_lower=ascii_lower)
    return "".join(ch for ch in s if ch.isalnum())
def levenshtein(a: str, b: str) -> int:
    a = nfc(a)
    b = nfc(b)
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]
class TextTokenizer:
    """Grapheme-level OCR tokenizer.
    Special ids:
      0 = blank/pad, 1 = unk, 2 = eos, 3 = bos
    The vocabulary is built only from training records. Do not include test labels.
    """
    def __init__(self, min_freq: int = 1):
        self.min_freq = int(min_freq)
        self.blank = 0
        self.pad = 0
        self.unk = 1
        self.eos = 2
        self.bos = 3
        self.tokens: List[str] = ["<blank_pad>", "<unk>", "<eos>", "<bos>"]
        self.stoi: Dict[str, int] = {t: i for i, t in enumerate(self.tokens)}
        self.itos: Dict[int, str] = {i: t for t, i in self.stoi.items()}
    @property
    def vocab_size(self) -> int:
        return len(self.tokens)
    def _add(self, token: str) -> int:
        if token not in self.stoi:
            self.stoi[token] = len(self.tokens)
            self.tokens.append(token)
        return self.stoi[token]
    def build(self, texts: Iterable[str], domains: Iterable[str]) -> None:
        counter: Counter = Counter()
        for text in texts:
            counter.update(split_graphemes(text))
        for domain_name in sorted(set(domains)):
            self._add(f"<domain:{domain_name}>")
        for tok, cnt in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
            if cnt >= self.min_freq:
                self._add(tok)
        self.itos = {i: t for t, i in self.stoi.items()}
        log(f"[tokenizer] vocab_size={self.vocab_size} | grapheme/fallback tokens={self.vocab_size - 4}")
    def encode_ctc(self, text: str) -> List[int]:
        return [self.stoi.get(g, self.unk) for g in split_graphemes(text)]
    def encode_ce(self, text: str, max_len: int) -> List[int]:
        ids = [self.stoi.get(g, self.unk) for g in split_graphemes(text)]
        ids = ids[: max_len - 1] + [self.eos]
        if len(ids) < max_len:
            ids += [self.pad] * (max_len - len(ids))
        return ids
    def decode_ctc(self, ids: Sequence[int]) -> str:
        out: List[str] = []
        prev: Optional[int] = None
        for idx in ids:
            idx = int(idx)
            if idx == self.blank or idx == prev:
                prev = idx
                continue
            tok = self.itos.get(idx, "")
            if tok.startswith("<") and tok.endswith(">"):
                prev = idx
                continue
            out.append(tok)
            prev = idx
        return nfc("".join(out))
    def decode_ce(self, ids: Sequence[int]) -> str:
        out: List[str] = []
        for idx in ids:
            idx = int(idx)
            if idx in [self.pad, self.bos]:
                continue
            if idx == self.eos:
                break
            tok = self.itos.get(idx, "")
            if tok.startswith("<") and tok.endswith(">"):
                continue
            out.append(tok)
        return nfc("".join(out))
    def to_dict(self) -> Dict[str, Any]:
        return {"min_freq": self.min_freq, "tokens": self.tokens}
    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "TextTokenizer":
        tok = cls(min_freq=int(d.get("min_freq", 1)))
        tok.tokens = list(d["tokens"])
        tok.stoi = {t: i for i, t in enumerate(tok.tokens)}
        tok.itos = {i: t for t, i in tok.stoi.items()}
        return tok
def polygon_area(poly: np.ndarray) -> float:
    p = np.asarray(poly, dtype=np.float32)
    if p.ndim != 2 or p.shape[0] < 3:
        return 0.0
    x, y = p[:, 0], p[:, 1]
    return float(abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))) * 0.5)
def build_ann_map(poly_dir: Path) -> Dict[str, Path]:
    mp: Dict[str, Path] = {}
    for p in poly_dir.glob("*.txt"):
        s = p.stem.lower()
        keys = {s, s.replace("poly_gt_", ""), s.replace("gt_", ""), s.replace("poly_", "")}
        for k in keys:
            mp[k] = p
    return mp
def parse_totaltext_line(line: str) -> Optional[Dict[str, Any]]:
    line = line.strip().replace("\ufeff", "")
    if not line:
        return None
    xm = re.search(r"x\s*:\s*\[\[(.*?)\]\]", line, flags=re.IGNORECASE)
    ym = re.search(r"y\s*:\s*\[\[(.*?)\]\]", line, flags=re.IGNORECASE)
    tm = re.search(r"transcriptions?\s*:\s*\[\s*(?:u|b)?[\"'](.*?)[\"']\s*\]", line, flags=re.IGNORECASE)
    if xm and ym:
        xs = [float(v) for v in re.findall(r"-?\d+(?:\.\d+)?", xm.group(1))]
        ys = [float(v) for v in re.findall(r"-?\d+(?:\.\d+)?", ym.group(1))]
        if len(xs) >= 3 and len(xs) == len(ys):
            text = tm.group(1) if tm else ""
            poly = np.asarray(list(zip(xs, ys)), dtype=np.float32)
            return {"polygon": poly, "text": nfc(text), "ignore": normalize_for_eval(text).lower() in IGNORE_TEXTS}
    parts = [p.strip() for p in re.split(r",\s*", line)]
    nums: List[float] = []
    text_parts: List[str] = []
    for p in parts:
        try:
            nums.append(float(p))
        except Exception:
            text_parts.append(p.strip().strip("'\""))
    if len(nums) >= 8 and len(nums) % 2 == 0:
        poly = np.asarray(list(zip(nums[0::2], nums[1::2])), dtype=np.float32)
        text = ",".join(text_parts).strip()
        return {"polygon": poly, "text": nfc(text), "ignore": normalize_for_eval(text).lower() in IGNORE_TEXTS}
    return None
def parse_totaltext_annotation(path: Path) -> List[Dict[str, Any]]:
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        text = path.read_text(encoding="latin-1", errors="ignore")
    out: List[Dict[str, Any]] = []
    for line in text.splitlines():
        obj = parse_totaltext_line(line)
        if obj is None:
            continue
        poly = obj["polygon"]
        if len(poly) < 3 or polygon_area(poly) <= 1.0:
            continue
        out.append(obj)
    return out
def build_totaltext_records(img_dir: Path, poly_dir: Path, split: str) -> List[Dict[str, Any]]:
    ann_map = build_ann_map(poly_dir)
    images = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
    records: List[Dict[str, Any]] = []
    missing_ann = 0
    total_inst = 0
    ignore_inst = 0
    for img in images:
        stem = img.stem.lower()
        ann = ann_map.get(stem) or ann_map.get(stem.replace("img", ""))
        if ann is None:
            candidates = [p for k, p in ann_map.items() if stem in k or k in stem]
            ann = candidates[0] if candidates else None
        if ann is None:
            missing_ann += 1
            instances: List[Dict[str, Any]] = []
        else:
            instances = parse_totaltext_annotation(ann)
        total_inst += len(instances)
        ignore_inst += sum(1 for x in instances if x.get("ignore", False))
        records.append({"image": str(img), "instances": instances, "split": split, "source": "TotalText-real"})
    log(f"[totaltext:{split}] images={len(records)} instances={total_inst} ignore={ignore_inst} missing_ann={missing_ann}")
    return records
def build_all_real_records(paths: Dict[str, Path]) -> Dict[str, Any]:
    """Build only Total-Text records.
    Returned keys intentionally match the Total-Text functions so the training path
    remains identical to from the Total-Text loader onward.
    """
    tt_train = build_totaltext_records(paths["tt_train_img"], paths["tt_train_poly"], "train")
    tt_test = build_totaltext_records(paths["tt_test_img"], paths["tt_test_poly"], "test")
    return {"tt_train": tt_train, "tt_test": tt_test}
def _to_jsonable(x: Any) -> Any:
    """Convert numpy / torch / Path scalar containers into JSON-safe Python objects.
    This is intentionally used only for manifest/report export. Training still keeps
    the original in-memory numpy arrays and tensors where the dataloaders expect them.
    """
    if x is None or isinstance(x, (str, int, float, bool)):
        return x
    if isinstance(x, Path):
        return str(x)
    # numpy arrays/scalars
    if hasattr(x, "tolist"):
        try:
            return x.tolist()
        except Exception:
            pass
    # torch tensors
    if hasattr(x, "detach") and hasattr(x, "cpu"):
        try:
            return x.detach().cpu().tolist()
        except Exception:
            pass
    if isinstance(x, dict):
        return {str(k): _to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_to_jsonable(v) for v in x]
    return str(x)
def _json_safe_record(rec: Dict[str, Any]) -> Dict[str, Any]:
    """Return a compact manifest-safe record for reproducibility.
    v6 exports manifests before training. Some parsers keep polygons as numpy
    arrays for fast geometric operations, so every exported field must be converted
    to builtin Python types before json.dumps.
    """
    out = {
        "image_path": _to_jsonable(rec.get("image_path") or rec.get("image")),
        "image_id": _to_jsonable(rec.get("image_id")),
        "width": _to_jsonable(rec.get("width")),
        "height": _to_jsonable(rec.get("height")),
        "split": _to_jsonable(rec.get("split")),
        "source": _to_jsonable(rec.get("source")),
        "instances": [],
    }
    for obj in rec.get("instances", []):
        out["instances"].append({
            "polygon": _to_jsonable(obj.get("polygon")),
            "bbox_xywh": _to_jsonable(obj.get("bbox_xywh")),
            "bezier_pts": _to_jsonable(obj.get("bezier_pts")),
            "text": _to_jsonable(obj.get("text")),
            "text_raw": _to_jsonable(obj.get("text_raw", obj.get("text"))),
            "text_norm": _to_jsonable(obj.get("text_norm")),
            "ignore": bool(obj.get("ignore", False)),
            "text_ignore": bool(obj.get("text_ignore", False)),
            "text_ignore_reason": _to_jsonable(obj.get("text_ignore_reason", "")),
            "source": _to_jsonable(obj.get("source")),
        })
    return out
def write_dataset_manifests(records: Dict[str, Any], out_dir: Path) -> None:
    """v6 manifest-first export. These files freeze the exact parser output used by training."""
    manifest_dir = out_dir / "manifests"
    manifest_dir.mkdir(parents=True, exist_ok=True)
    for key, recs in sorted(records.items()):
        path = manifest_dir / f"{key}.jsonl"
        with path.open("w", encoding="utf-8") as f:
            for rec in recs:
                f.write(json.dumps(_json_safe_record(rec), ensure_ascii=False) + "\n")
        log(f"[manifest] {key}: {len(recs)} records -> {path}")
def sanity_check_records(records: Dict[str, Any]) -> Dict[str, Any]:
    """Total-Text-only dataset count check.
    The local dataset documentation has shown slight parser/count variation across
    versions, so this function logs exact counts and warns on large deviations
    instead of blocking the run unless the image split itself is wrong.
    """
    section("TOTAL-TEXT COUNT / PROTOCOL CHECK")
    counts = {
        "tt_train_images": len(records["tt_train"]),
        "tt_test_images": len(records["tt_test"]),
        "tt_train_instances": sum(len(rec.get("instances", [])) for rec in records["tt_train"]),
        "tt_test_instances": sum(len(rec.get("instances", [])) for rec in records["tt_test"]),
        "tt_train_valid_instances": sum(1 for rec in records["tt_train"] for obj in rec.get("instances", []) if not obj.get("ignore", False)),
        "tt_test_valid_instances": sum(1 for rec in records["tt_test"] for obj in rec.get("instances", []) if not obj.get("ignore", False)),
        "tt_train_ignored_instances": sum(1 for rec in records["tt_train"] for obj in rec.get("instances", []) if obj.get("ignore", False)),
        "tt_test_ignored_instances": sum(1 for rec in records["tt_test"] for obj in rec.get("instances", []) if obj.get("ignore", False)),
    }
    log(json.dumps(counts, ensure_ascii=False, indent=2))
    problems = []
    if counts["tt_train_images"] != int(cfg.EXPECTED_TT_TRAIN_IMAGES):
        problems.append(f"tt_train_images={counts['tt_train_images']} expected={cfg.EXPECTED_TT_TRAIN_IMAGES}")
    if counts["tt_test_images"] != int(cfg.EXPECTED_TT_TEST_IMAGES):
        problems.append(f"tt_test_images={counts['tt_test_images']} expected={cfg.EXPECTED_TT_TEST_IMAGES}")
    if problems:
        raise RuntimeError("Total-Text image split configuration failed: " + "; ".join(problems))
    return counts
def build_tokenizer_and_domain_ids(records: Dict[str, Any], out_root: Path) -> Tuple[TextTokenizer, Dict[str, int]]:
    """Build the OCR tokenizer from non-ignored Total-Text training transcriptions only."""
    section("TOTAL-TEXT OCR TOKENIZER BUILD")
    tok_path = out_root / "tokenizer_totaltext.json"
    domain_path = out_root / "domain_to_id.json"
    if cfg.RESUME and tok_path.exists() and domain_path.exists():
        tokenizer = TextTokenizer.from_dict(read_json(tok_path))
        domain_to_id = {k: int(v) for k, v in read_json(domain_path).items()}
        log(f"[tokenizer] loaded existing tokenizer: vocab={tokenizer.vocab_size}")
        return tokenizer, domain_to_id
    texts: List[str] = []
    for rec in records["tt_train"]:
        for obj in rec.get("instances", []):
            if not is_text_usable_instance(obj):
                continue
            word = normalize_for_eval(obj.get("text", ""), ascii_lower=True)
            if word:
                texts.append(word)
    tokenizer = TextTokenizer(min_freq=cfg.VOCAB_MIN_FREQ)
    tokenizer.build(texts, ["total_text_latin"] * len(texts))
    domain_to_id = {"total_text_latin": 0}
    save_json(tokenizer.to_dict(), tok_path)
    save_json(domain_to_id, domain_path)
    log(f"[tokenizer] Total-Text train words={len(texts)} vocab_size={tokenizer.vocab_size}")
    return tokenizer, domain_to_id
# =============================================================================
# 4. TOTAL-TEXT OCR AUGMENTATION, LOADER, AND COLLATE
# =============================================================================
def load_image_rgb(path: Path) -> Image.Image:
    with Image.open(path) as im:
        return im.convert("RGB")
def pil_to_tensor(im: Image.Image) -> torch.Tensor:
    arr = np.asarray(im).astype(np.float32) / 255.0
    if arr.ndim == 2:
        arr = np.repeat(arr[..., None], 3, axis=-1)
    t = torch.from_numpy(arr).permute(2, 0, 1)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t - mean) / std
def str_bucket_size(w: int, h: int, train: bool = False) -> Tuple[int, int]:
    if not cfg.STR_USE_MULTI_WIDTH_BUCKETS:
        return cfg.STR_HEIGHT, cfg.OCR_FIXED_W
    ratio = float(w) / max(1.0, float(h))
    need_w = int(max(cfg.STR_WIDTH_BUCKETS[0], min(cfg.STR_WIDTH_BUCKETS[-1], round(cfg.STR_HEIGHT * ratio * 1.25))))
    buckets = list(cfg.STR_WIDTH_BUCKETS)
    chosen = buckets[-1]
    for bw in buckets:
        if need_w <= bw:
            chosen = bw
            break
    if train and cfg.STR_TRAIN_RANDOM_BUCKET and len(buckets) > 1:
        idx = buckets.index(chosen)
        candidates = [chosen]
        if idx + 1 < len(buckets):
            candidates.append(buckets[idx + 1])
        if idx > 0 and random.random() < 0.15:
            candidates.append(buckets[idx - 1])
        chosen = random.choice(candidates)
    return cfg.STR_HEIGHT, int(chosen)
def letterbox_rgb(im: Image.Image, size_hw: Tuple[int, int], fill: int = 128) -> Image.Image:
    out_h, out_w = size_hw
    w, h = im.size
    scale = min(out_w / max(1, w), out_h / max(1, h))
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    resized = im.resize((nw, nh), Image.BICUBIC)
    canvas = Image.new("RGB", (out_w, out_h), (fill, fill, fill))
    canvas.paste(resized, ((out_w - nw) // 2, (out_h - nh) // 2))
    return canvas
def _safe_grid_distort(im: Image.Image, strength: float = 0.018) -> Image.Image:
    if cv2 is None:
        return im
    arr = np.asarray(im).astype(np.uint8)
    h, w = arr.shape[:2]
    if h < 16 or w < 16:
        return im
    amp_x = max(1.0, strength * w)
    amp_y = max(1.0, strength * h)
    xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    phase_x = random.random() * math.pi * 2
    phase_y = random.random() * math.pi * 2
    map_x = xs + amp_x * np.sin(2.0 * math.pi * ys / max(8.0, h) + phase_x).astype(np.float32)
    map_y = ys + amp_y * np.sin(2.0 * math.pi * xs / max(8.0, w) + phase_y).astype(np.float32)
    warped = cv2.remap(arr, map_x, map_y, interpolation=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(warped)
def augment_str_image(im: Image.Image) -> Image.Image:
    if random.random() < 0.45:
        im = ImageEnhance.Contrast(im).enhance(random.uniform(0.70, 1.45))
    if random.random() < 0.45:
        im = ImageEnhance.Brightness(im).enhance(random.uniform(0.75, 1.30))
    if random.random() < 0.20:
        im = ImageEnhance.Sharpness(im).enhance(random.uniform(0.75, 1.50))
    if random.random() < 0.22:
        im = im.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.15, 0.75)))
    if random.random() < 0.24:
        angle = random.uniform(-2.5, 2.5)
        im = im.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor=(128, 128, 128))
    if random.random() < 0.14:
        im = _safe_grid_distort(im, strength=0.010)
    if random.random() < 0.24:
        arr = np.asarray(im).astype(np.float32)
        noise = np.random.normal(0.0, random.uniform(1.5, 8.0), size=arr.shape).astype(np.float32)
        im = Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))
    return im
def collate_totaltext_ocr(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    """OCR batch collate used only by the Total-Text crop recognizer."""
    max_h = max(x["image"].shape[1] for x in batch)
    max_w = max(x["image"].shape[2] for x in batch)
    imgs = []
    for x in batch:
        im = x["image"]
        imgs.append(F.pad(im, (0, max_w - im.shape[2], 0, max_h - im.shape[1]), value=0.0))
    return {
        "images": torch.stack(imgs, dim=0),
        "ctc_targets": torch.cat([x["ctc_ids"] for x in batch], dim=0) if batch else torch.zeros((0,), dtype=torch.long),
        "target_lengths": torch.tensor([len(x["ctc_ids"]) for x in batch], dtype=torch.long),
        "ce_targets": torch.stack([x["ce_ids"] for x in batch], dim=0),
        "domain_ids": torch.stack([x["domain_id"] for x in batch], dim=0),
        "texts": [x["text"] for x in batch],
        "domains": [x.get("domain", "total_text_latin") for x in batch],
        "paths": [x["path"] for x in batch],
        "crop_modes": [x.get("crop_mode", "rectified") for x in batch],
    }
_DATALOADER_CREATE_COUNT = 0

def _effective_loader_workers() -> Tuple[int, int, bool]:
    workers = max(0, int(getattr(cfg, "NUM_WORKERS", 0)))
    prefetch = max(1, int(getattr(cfg, "PREFETCH_FACTOR", 1)))
    persistent = bool(getattr(cfg, "PERSISTENT_WORKERS", False))
    # The target Colab Simple T4 has 1 physical core / 2 logical threads and 12.7GB RAM.
    # V7.15 uses both logical threads but keeps queues bounded. This preserves all
    # samples/labels/losses/metrics while avoiding the V7.14 worker=1 starvation.
    cpu = _cpu_memory_snapshot()
    if (os.cpu_count() or 1) <= 2 or (cpu.get("total_gb", 99.0) and cpu.get("total_gb", 99.0) <= 14.0):
        workers = min(max(workers, 1), 2)
        prefetch = min(max(prefetch, 1), 2)
        persistent = bool(workers > 0 and getattr(cfg, "PERSISTENT_WORKERS", True))
    if workers <= 0:
        persistent = False
    return workers, prefetch, persistent

def make_loader(dataset: Dataset, batch_size: int, shuffle: bool, sampler=None, collate_fn=None, drop_last: bool = False) -> DataLoader:
    global _DATALOADER_CREATE_COUNT
    workers, prefetch, persistent = _effective_loader_workers()
    kwargs = dict(
        dataset=dataset,
        batch_size=max(1, int(batch_size)),
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=workers,
        pin_memory=bool(cfg.PIN_MEMORY and torch.cuda.is_available()),
        collate_fn=collate_fn,
        drop_last=drop_last,
    )
    if workers > 0:
        kwargs["persistent_workers"] = persistent
        kwargs["prefetch_factor"] = prefetch
    _DATALOADER_CREATE_COUNT += 1
    if bool(getattr(cfg, "DATALOADER_LOG_ON_CREATE", True)) and _DATALOADER_CREATE_COUNT <= int(getattr(cfg, "DATALOADER_LOG_LIMIT", 40)):
        try:
            log(f"[loader] dataset={type(dataset).__name__} n={len(dataset)} batch={kwargs['batch_size']} workers={workers} prefetch={prefetch if workers > 0 else 0} persistent={persistent} pin={kwargs['pin_memory']} drop_last={drop_last}")
        except Exception:
            pass
    return DataLoader(**kwargs)
# =============================================================================
# 5. TOTAL-TEXT OCR RECOGNIZER: STR HEAD FAMILY
# =============================================================================
class ViTAEv2SStageBackbone(nn.Module):
    """Official ViTAEv2-S adapter that exposes C2-C5 spatial feature maps.
    The official ViTAEv2-S model returns a pooled vector for classification.
    For Total-Text spotting, dense feature maps are required. This adapter runs
    the official internal stages manually and reshapes each token sequence back
    to a spatial map:
        stage 1 -> C2: [B,  64, H/4,  W/4]
        stage 2 -> C3: [B, 128, H/8,  W/8]
        stage 3 -> C4: [B, 256, H/16, W/16]
        stage 4 -> C5: [B, 512, H/32, W/32]
    These maps are then consumed by the same FPN, DB++ head, ArcFlowAlign,
    ArcBridge, recognizer, losses, and evaluator used by ResNet-50.
    """
    stage_names = ("c2", "c3", "c4", "c5")
    stage_strides = (4, 8, 16, 32)
    stage_channels = (64, 128, 256, 512)
    def __init__(self, pretrained: bool = True):
        super().__init__()
        timm = ensure_vitaev2_environment()
        self.name = "ViTAEv2_S"
        self.out_channels = {
            "c2": 64,
            "c3": 128,
            "c4": 256,
            "c5": 512,
        }
        try:
            self.backbone = timm.create_model("ViTAEv2_S", pretrained=pretrained, num_classes=0)
            self.loaded_with_pretrained_flag = bool(pretrained)
        except Exception as e:
            if pretrained:
                log(f"[model][warn] ViTAEv2_S pretrained load failed ({repr(e)}). Falling back to pretrained=False.")
                self.backbone = timm.create_model("ViTAEv2_S", pretrained=False, num_classes=0)
                self.loaded_with_pretrained_flag = False
            else:
                log(f"[model][warn] timm ViTAEv2_S construction failed ({repr(e)}). Trying direct official constructor.")
                from vitaev2.ViTAEv2 import ViTAEv2_S  # type: ignore
                self.backbone = ViTAEv2_S(pretrained=False, num_classes=0)
                self.loaded_with_pretrained_flag = False
        if not hasattr(self.backbone, "layers"):
            raise RuntimeError("Official ViTAEv2_S model does not expose .layers; cannot recover spatial stages.")
        if len(self.backbone.layers) != 4:
            raise RuntimeError(f"Expected ViTAEv2_S to expose 4 layers, found {len(self.backbone.layers)}.")
        total_params = sum(p.numel() for p in self.backbone.parameters())
        trainable_params = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        log(
            f"[model] ViTAEv2_S official adapter ready | pretrained_loaded={self.loaded_with_pretrained_flag} "
            f"params={total_params/1e6:.2f}M trainable={trainable_params/1e6:.2f}M "
            f"out_channels={self.out_channels}"
        )
    @staticmethod
    def _choose_factor_pair(n: int, target_h: int, target_w: int) -> Tuple[int, int]:
        """Find a factor pair whose aspect ratio is closest to the target."""
        if n <= 0:
            raise RuntimeError(f"Invalid token count for spatial reshape: {n}")
        best = None
        target_ratio = float(target_h) / max(1.0, float(target_w))
        root = int(math.sqrt(n))
        for h in range(1, root + 1):
            if n % h != 0:
                continue
            w = n // h
            score = abs((float(h) / float(w)) - target_ratio)
            if best is None or score < best[0]:
                best = (score, h, w)
        if best is None:
            raise RuntimeError(f"Could not factor token count {n} into a spatial map.")
        return int(best[1]), int(best[2])
    def _tokens_to_spatial(self, tokens: torch.Tensor, input_h: int, input_w: int, stage_idx: int) -> torch.Tensor:
        if tokens.dim() != 3:
            raise RuntimeError(
                f"ViTAEv2_S stage {stage_idx + 1} should return tokens [B, N, C], got {list(tokens.shape)}"
            )
        b, n, ch = tokens.shape
        stride = int(self.stage_strides[stage_idx])
        expected_ch = int(self.stage_channels[stage_idx])
        target_h = max(1, int(input_h) // stride)
        target_w = max(1, int(input_w) // stride)
        if target_h * target_w != int(n):
            # Some official implementations infer square token grids internally.
            # Use a safe factor-pair fallback instead of silently producing an
            # invalid reshape. The selected shape is logged once by the caller.
            target_h, target_w = self._choose_factor_pair(int(n), target_h, target_w)
        feat = tokens.transpose(1, 2).contiguous().reshape(b, ch, target_h, target_w)
        if int(ch) != expected_ch:
            log(
                f"[model][warn] ViTAEv2_S stage {stage_idx + 1} channel count {ch} "
                f"differs from expected {expected_ch}. The FPN will adapt using 1x1 projections."
            )
        return feat
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        b, c, h, w = x.shape
        tokens: torch.Tensor = x
        outputs: Dict[str, torch.Tensor] = {}
        for idx, layer in enumerate(self.backbone.layers):
            tokens = layer(tokens)
            outputs[self.stage_names[idx]] = self._tokens_to_spatial(tokens, h, w, idx)
        return outputs
class TotalTextBackbone(nn.Module):
    """Visual backbone returning C2-C5 feature maps for Total-Text spotting.
    Supported public backbones:
      - ``resnet50``: torchvision ResNet-50, C2-C5 from residual stages.
      - ``vitaev2_s`` / ``ViTAEv2_S``: official ViTAEv2-S repository adapter.
    The downstream architecture remains identical for both backbones.
    """
    def __init__(self, name: str = "resnet50", pretrained: bool = True):
        super().__init__()
        raw_name = str(name)
        name_l = raw_name.lower().replace("-", "_")
        log(f"[model] Building backbone={raw_name}, imagenet_pretrained={pretrained}")
        self.backend = "torchvision_resnet"
        self.selected_name = raw_name
        self._shape_logged = False
        if name_l in {"resnet18", "resnet34", "resnet50"}:
            if torchvision is None:
                raise RuntimeError("torchvision is required for ResNet backbones in this file. Use a Colab/PyTorch runtime with torchvision installed.")
            try:
                if name_l == "resnet18":
                    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
                    base = torchvision.models.resnet18(weights=weights)
                    self.out_channels = {"c2": 64, "c3": 128, "c4": 256, "c5": 512}
                elif name_l == "resnet34":
                    weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
                    base = torchvision.models.resnet34(weights=weights)
                    self.out_channels = {"c2": 64, "c3": 128, "c4": 256, "c5": 512}
                else:
                    weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
                    base = torchvision.models.resnet50(weights=weights)
                    self.out_channels = {"c2": 256, "c3": 512, "c4": 1024, "c5": 2048}
            except Exception as e:
                log(f"[model][warn] pretrained weights unavailable ({repr(e)}). Falling back to random init.")
                if name_l == "resnet18":
                    base = torchvision.models.resnet18(weights=None)
                    self.out_channels = {"c2": 64, "c3": 128, "c4": 256, "c5": 512}
                elif name_l == "resnet34":
                    base = torchvision.models.resnet34(weights=None)
                    self.out_channels = {"c2": 64, "c3": 128, "c4": 256, "c5": 512}
                else:
                    base = torchvision.models.resnet50(weights=None)
                    self.out_channels = {"c2": 256, "c3": 512, "c4": 1024, "c5": 2048}
            self.selected_name = name_l
            log(f"[model] backbone_backend={self.backend} selected={self.selected_name} out_channels={self.out_channels}")
            self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
            self.layer1 = base.layer1
            self.layer2 = base.layer2
            self.layer3 = base.layer3
            self.layer4 = base.layer4
            return
        if name_l in {"vitaev2_s", "vitaev2s", "vitae_v2_s", "vitae2_s", "vitae_s", "vitae", "vitaev2_s_224", "vitaev2_s_224_in1k"} or raw_name == "ViTAEv2_S":
            self.backend = "official_vitaev2_s"
            self.selected_name = "ViTAEv2_S"
            self.vitae = ViTAEv2SStageBackbone(pretrained=pretrained)
            self.out_channels = dict(self.vitae.out_channels)
            log(f"[model] backbone_backend={self.backend} selected={self.selected_name} out_channels={self.out_channels}")
            return
        raise ValueError(
            f"Unsupported backbone '{raw_name}'. Supported options are: resnet50 and vitaev2_s."
        )
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        if self.backend == "official_vitaev2_s":
            out = self.vitae(x)
        else:
            x = self.stem(x)
            c2 = self.layer1(x)
            c3 = self.layer2(c2)
            c4 = self.layer3(c3)
            c5 = self.layer4(c4)
            out = {"c2": c2, "c3": c3, "c4": c4, "c5": c5}
        if set(out.keys()) != {"c2", "c3", "c4", "c5"}:
            raise RuntimeError(f"Backbone returned invalid feature keys: {sorted(out.keys())}")
        if not self._shape_logged:
            shape_map = {k: list(v.shape) for k, v in out.items()}
            log(f"[model] backbone_feature_shapes backend={self.backend} selected={self.selected_name} shapes={shape_map}")
            self._shape_logged = True
        return out
class PositionalEncoding1D(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)].to(dtype=x.dtype, device=x.device)
class STRHead(nn.Module):
    def __init__(self, in_channels: int, vocab_size: int, num_domains: int):
        super().__init__()
        self.num_domains = max(1, int(num_domains))
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, cfg.D_MODEL, kernel_size=1),
            nn.BatchNorm2d(cfg.D_MODEL),
            nn.ReLU(inplace=True),
            nn.Conv2d(cfg.D_MODEL, cfg.D_MODEL, kernel_size=3, padding=1),
            nn.BatchNorm2d(cfg.D_MODEL),
            nn.GELU(),
        )
        self.domain_embed = nn.Embedding(self.num_domains, cfg.D_MODEL)
        self.domain_gate = nn.Sequential(nn.Linear(cfg.D_MODEL, cfg.D_MODEL), nn.Sigmoid())
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.TRANSFORMER_HEADS,
            dim_feedforward=cfg.D_MODEL * 4,
            dropout=cfg.DROPOUT,
            batch_first=True,
            activation="gelu",
        )
        self.pos = PositionalEncoding1D(cfg.D_MODEL, max_len=512)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.STR_TRANSFORMER_LAYERS)
        self.ce_query = nn.Embedding(cfg.MAX_TEXT_LEN, cfg.D_MODEL)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.TRANSFORMER_HEADS,
            dim_feedforward=cfg.D_MODEL * 4,
            dropout=cfg.DROPOUT,
            batch_first=True,
            activation="gelu",
        )
        self.ce_decoder = nn.TransformerDecoder(dec_layer, num_layers=cfg.STR_CE_DECODER_LAYERS)
        self.semantic_gate = nn.Sequential(nn.Linear(cfg.D_MODEL, cfg.D_MODEL), nn.Sigmoid())
        self.ctc = nn.Linear(cfg.D_MODEL, vocab_size)
        self.ce = nn.Linear(cfg.D_MODEL, vocab_size)
        self.domain_classifier = nn.Linear(cfg.D_MODEL, self.num_domains)
    def _ctc_rearrange(self, x: torch.Tensor) -> torch.Tensor:
        avg = F.adaptive_avg_pool2d(x, (1, x.shape[-1])).squeeze(2)
        mx = F.adaptive_max_pool2d(x, (1, x.shape[-1])).squeeze(2)
        return (avg + mx).transpose(1, 2)  # B,W,D
    def forward(self, feat: torch.Tensor, domain_ids: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        x2d = self.proj(feat)
        x = self._ctc_rearrange(x2d)
        if cfg.STR_USE_DOMAIN_CONDITIONING and domain_ids is not None:
            domain_ids = domain_ids.clamp(min=0, max=self.num_domains - 1)
            le = self.domain_embed(domain_ids).to(dtype=x.dtype)
            gate = self.domain_gate(le).to(dtype=x.dtype).unsqueeze(1)
            x = x + gate * le.unsqueeze(1)
        x = self.pos(x)
        x = self.encoder(x)
        pooled = x.mean(dim=1)
        sg = self.semantic_gate(pooled).unsqueeze(1).to(dtype=x.dtype)
        x = x + sg * pooled.unsqueeze(1)
        ctc_logits = self.ctc(x).transpose(0, 1)  # T,B,V
        q = self.ce_query.weight.unsqueeze(0).expand(x.shape[0], -1, -1).to(dtype=x.dtype)
        if cfg.STR_USE_DOMAIN_CONDITIONING and domain_ids is not None:
            le = self.domain_embed(domain_ids.clamp(min=0, max=self.num_domains - 1)).to(dtype=x.dtype)
            q = q + le.unsqueeze(1)
        ce_feat = self.ce_decoder(q, x)
        return {
            "ctc_logits": ctc_logits,
            "ce_logits": self.ce(ce_feat),
            "domain_logits": self.domain_classifier(pooled),
            "ctc_features": x,
            "ce_features": ce_feat,
            "pooled_features": pooled,
        }
class ArcFlowSTRModel(nn.Module):
    def __init__(self, vocab_size: int, num_domains: int):
        super().__init__()
        self.backbone = TotalTextBackbone(cfg.BACKBONE_NAME, cfg.PRETRAIN_IMAGENET)
        self.head = STRHead(self.backbone.out_channels["c3"], vocab_size, num_domains)
    def forward(self, images: torch.Tensor, domain_ids: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        feats = self.backbone(images)
        return self.head(feats["c3"], domain_ids=domain_ids)
# =============================================================================
# 6. OCR LOSS, DECODER, AND CROP DIAGNOSTIC EVALUATION
# =============================================================================
def _cosine_progress(epoch: int, end_epoch: int) -> float:
    if end_epoch <= 1:
        return 1.0
    x = min(1.0, max(0.0, float(epoch - 1) / float(end_epoch - 1)))
    return 0.5 - 0.5 * math.cos(math.pi * x)
def str_curriculum_weights(epoch: int) -> Dict[str, float]:
    r = _cosine_progress(int(epoch), int(cfg.STR_SCHEDULE_END_EPOCH))
    sem_r = _cosine_progress(int(epoch), max(1, min(int(cfg.STR_SCHEDULE_END_EPOCH), 15)))
    return {
        "ctc_w": cfg.STR_CTC_WEIGHT_START + r * (cfg.STR_CTC_WEIGHT_END - cfg.STR_CTC_WEIGHT_START),
        "ce_w": cfg.STR_CE_WEIGHT_START + r * (cfg.STR_CE_WEIGHT_END - cfg.STR_CE_WEIGHT_START),
        "sem_w": cfg.STR_SEM_WEIGHT_START + sem_r * (cfg.STR_SEM_WEIGHT_END - cfg.STR_SEM_WEIGHT_START),
        "domain_w": float(cfg.STR_DOMAIN_WEIGHT),
    }
def str_loss(outputs: Dict[str, torch.Tensor], batch: Dict[str, Any], epoch: int) -> Tuple[torch.Tensor, Dict[str, float]]:
    ctc_logits = outputs["ctc_logits"]
    ce_logits = outputs["ce_logits"]
    domain_logits = outputs["domain_logits"]
    t, b, v = ctc_logits.shape
    input_lengths = torch.full((b,), t, dtype=torch.long, device=ctc_logits.device)
    target_lengths = batch["target_lengths"].to(ctc_logits.device)
    ctc_targets = batch["ctc_targets"].to(ctc_logits.device)
    ce_targets = batch["ce_targets"].to(ctc_logits.device)
    domain_ids = batch["domain_ids"].to(ctc_logits.device)
    weights = str_curriculum_weights(epoch)
    loss_ctc = F.ctc_loss(
        F.log_softmax(ctc_logits.float(), dim=-1),
        ctc_targets,
        input_lengths,
        target_lengths,
        blank=0,
        zero_infinity=True,
    )
    loss_ce = F.cross_entropy(
        ce_logits.reshape(-1, v).float(),
        ce_targets.reshape(-1),
        ignore_index=0,
        label_smoothing=float(cfg.OCR_CE_LABEL_SMOOTHING),
    )
    loss_domain = F.cross_entropy(domain_logits.float(), domain_ids)
    loss_sem = torch.zeros((), device=ctc_logits.device, dtype=torch.float32)
    if weights["sem_w"] > 0 and "ctc_features" in outputs and "ce_features" in outputs:
        h_ctc = F.normalize(outputs["ctc_features"].float().mean(dim=1), dim=-1)
        h_ce = F.normalize(outputs["ce_features"].float().mean(dim=1), dim=-1)
        h_teacher = h_ce.detach() if cfg.STR_SEMANTIC_STOPGRAD_CE else h_ce
        loss_sem = (1.0 - (h_ctc * h_teacher).sum(dim=-1)).mean()
    loss = (
        weights["ctc_w"] * loss_ctc
        + weights["ce_w"] * loss_ce
        + weights["domain_w"] * loss_domain
        + weights["sem_w"] * loss_sem
    )
    stats = {
        "loss": float(loss.detach().cpu()),
        "ctc": float(loss_ctc.detach().cpu()),
        "ce": float(loss_ce.detach().cpu()),
        "domain": float(loss_domain.detach().cpu()),
        "sem": float(loss_sem.detach().cpu()),
        "ctc_w": float(weights["ctc_w"]),
        "ce_w": float(weights["ce_w"]),
        "sem_w": float(weights["sem_w"]),
    }
    return loss, stats
def _decode_str_batch(outputs: Dict[str, torch.Tensor], tokenizer: TextTokenizer, epoch: int) -> Dict[str, List[Dict[str, Any]]]:
    ctc_probs = outputs["ctc_logits"].float().softmax(dim=-1).transpose(0, 1)
    ctc_conf, ctc_ids = ctc_probs.max(dim=-1)
    ce_probs = outputs["ce_logits"].float().softmax(dim=-1)
    ce_conf, ce_ids = ce_probs.max(dim=-1)
    decoded: Dict[str, List[Dict[str, Any]]] = {"ctc": [], "ce": [], "selected": []}
    for b in range(ctc_ids.shape[0]):
        ctc_seq = [int(x) for x in ctc_ids[b].detach().cpu().tolist()]
        ctc_conf_list = [float(x) for x in ctc_conf[b].detach().cpu().tolist()]
        ctc_text = tokenizer.decode_ctc(ctc_seq)
        blank_count = sum(1 for x in ctc_seq if x == tokenizer.blank)
        unk_count = sum(1 for x in ctc_seq if x == tokenizer.unk)
        repeat_count = sum(1 for i in range(1, len(ctc_seq)) if ctc_seq[i] == ctc_seq[i - 1] and ctc_seq[i] != tokenizer.blank)
        special_count = 0
        kept_conf: List[float] = []
        prev = None
        for idx, cf in zip(ctc_seq, ctc_conf_list):
            tok = tokenizer.itos.get(int(idx), "")
            if idx == tokenizer.blank or idx == prev:
                prev = idx
                continue
            if tok.startswith("<") and tok.endswith(">"):
                special_count += 1
                prev = idx
                continue
            kept_conf.append(float(cf))
            prev = idx
        ctc_score = float(np.mean(kept_conf)) if kept_conf else float(np.mean(ctc_conf_list))
        ctc_diag = {
            "blank_rate": 100.0 * blank_count / max(1, len(ctc_seq)),
            "repeat_rate": 100.0 * repeat_count / max(1, len(ctc_seq) - 1),
            "unk_rate": 100.0 * unk_count / max(1, len(ctc_seq)),
            "special_rate": 100.0 * special_count / max(1, len(ctc_seq)),
        }
        ce_seq = [int(x) for x in ce_ids[b].detach().cpu().tolist()]
        ce_conf_list = [float(x) for x in ce_conf[b].detach().cpu().tolist()]
        ce_text = tokenizer.decode_ce(ce_seq)
        pad_count = sum(1 for x in ce_seq if x == tokenizer.pad)
        unk_count_ce = sum(1 for x in ce_seq if x == tokenizer.unk)
        eos_count = sum(1 for x in ce_seq if x == tokenizer.eos)
        ce_scores: List[float] = []
        for idx, cf in zip(ce_seq, ce_conf_list):
            if idx == tokenizer.pad or idx == tokenizer.bos:
                continue
            ce_scores.append(float(cf))
            if idx == tokenizer.eos:
                break
        ce_score = float(np.mean(ce_scores)) if ce_scores else float(np.mean(ce_conf_list))
        ce_diag = {
            "blank_rate": 0.0,
            "repeat_rate": 0.0,
            "unk_rate": 100.0 * unk_count_ce / max(1, len(ce_seq)),
            "special_rate": 100.0 * (pad_count + eos_count) / max(1, len(ce_seq)),
            "eos_rate": 100.0 * eos_count / max(1, len(ce_seq)),
            "pad_rate": 100.0 * pad_count / max(1, len(ce_seq)),
        }
        decode_mode = str(cfg.OCR_DECODE_MODE).lower().strip()
        if decode_mode == "auto":
            decode_mode = "ctc" if int(epoch) < int(cfg.STR_CE_SELECT_START_EPOCH) else "select"
        if decode_mode == "ce":
            sel_text, sel_score, sel_src = ce_text, ce_score, "ce"
        elif decode_mode == "select":
            max_len = max(1, int(cfg.MAX_TEXT_LEN * cfg.STR_SELECT_MAX_LEN_RATIO))
            ce_len_ok = 0 < len(ce_text) <= max_len
            ce_conf_ok = ce_score >= float(cfg.STR_SELECT_CE_CONF_RATIO) * max(1e-6, ctc_score)
            if ce_len_ok and (ce_conf_ok or not ctc_text):
                sel_text, sel_score, sel_src = ce_text, ce_score, "ce"
            else:
                sel_text, sel_score, sel_src = ctc_text, ctc_score, "ctc"
        else:
            sel_text, sel_score, sel_src = ctc_text, ctc_score, "ctc"
        decoded["ctc"].append({"text": ctc_text, "conf": ctc_score, "src": "ctc", **ctc_diag})
        decoded["ce"].append({"text": ce_text, "conf": ce_score, "src": "ce", **ce_diag})
        decoded["selected"].append({"text": sel_text, "conf": sel_score, "src": sel_src, **(ce_diag if sel_src == "ce" else ctc_diag)})
    return decoded
def _positional_char_correct(pred: str, gt: str) -> Tuple[int, int]:
    pred_chars = list(pred)
    gt_chars = list(gt)
    correct = sum(1 for i, ch in enumerate(gt_chars) if i < len(pred_chars) and pred_chars[i] == ch)
    return correct, max(1, len(gt_chars))
def _edit_char_correct(pred: str, gt: str) -> Tuple[int, int]:
    total = max(1, len(gt))
    dist = levenshtein(pred, gt)
    return max(0, total - dist), total
def evaluate_str(
    model: nn.Module,
    loader: DataLoader,
    tokenizer: TextTokenizer,
    out_json: Optional[Path],
    epoch: int,
) -> Dict[str, Any]:
    model.eval()
    device = cfg.DEVICE
    modes = ["ctc", "ce", "selected"]
    def new_stats() -> Dict[str, float]:
        return {
            "n": 0,
            "word_correct": 0,
            "word_correct_ci": 0,
            "word_correct_alnum": 0,
            "char_correct": 0,
            "char_total": 0,
            "edit_char_correct": 0,
            "edit_char_total": 0,
            "empty": 0,
            "edit_le1": 0,
            "edit_le2": 0,
            "pred_len": 0,
            "gt_len": 0,
            "conf_correct_sum": 0.0,
            "conf_correct_n": 0,
            "conf_wrong_sum": 0.0,
            "conf_wrong_n": 0,
            "blank_rate_sum": 0.0,
            "repeat_rate_sum": 0.0,
            "unk_rate_sum": 0.0,
            "special_rate_sum": 0.0,
        }
    per_mode = {mode: defaultdict(new_stats) for mode in modes}
    global_mode = {mode: new_stats() for mode in modes}
    selected_src_counter: Counter = Counter()
    examples: List[Dict[str, Any]] = []
    def update_stats(s: Dict[str, float], pred: str, gt: str, conf: float, d: Dict[str, Any]) -> None:
        ok = pred == gt
        ok_ci = normalize_for_eval(pred, ascii_lower=True) == normalize_for_eval(gt, ascii_lower=True)
        ok_alnum = normalize_alnum_for_eval(pred, ascii_lower=True) == normalize_alnum_for_eval(gt, ascii_lower=True)
        pc, pt = _positional_char_correct(pred, gt)
        ec, et = _edit_char_correct(pred, gt)
        s["n"] += 1
        s["word_correct"] += int(ok)
        s["word_correct_ci"] += int(ok_ci)
        s["word_correct_alnum"] += int(ok_alnum)
        s["char_correct"] += pc
        s["char_total"] += pt
        s["edit_char_correct"] += ec
        s["edit_char_total"] += et
        s["empty"] += int(len(pred) == 0)
        dist = levenshtein(pred, gt)
        s["edit_le1"] += int(dist <= 1)
        s["edit_le2"] += int(dist <= 2)
        s["pred_len"] += len(pred)
        s["gt_len"] += len(gt)
        if ok:
            s["conf_correct_sum"] += float(conf)
            s["conf_correct_n"] += 1
        else:
            s["conf_wrong_sum"] += float(conf)
            s["conf_wrong_n"] += 1
        s["blank_rate_sum"] += float(d.get("blank_rate", 0.0))
        s["repeat_rate_sum"] += float(d.get("repeat_rate", 0.0))
        s["unk_rate_sum"] += float(d.get("unk_rate", 0.0))
        s["special_rate_sum"] += float(d.get("special_rate", 0.0))
    # V7.3: crop evaluation is the silent Colab-kill point for CTW1500 after
    # Total-Text + ICDAR have already run. Use inference_mode and explicit tensor
    # lifetime control. This does not change evaluation logic or metrics.
    with torch.inference_mode():
        for batch_idx, batch in enumerate(loader, start=1):
            images = batch["images"].to(device, non_blocking=True)
            domain_ids = batch["domain_ids"].to(device, non_blocking=True)
            with autocast(device_type="cuda", dtype=get_amp_dtype(), enabled=(cfg.DEVICE == "cuda")):
                outputs = model(images, domain_ids=domain_ids)
            decoded = _decode_str_batch(outputs, tokenizer, epoch=epoch)
            for i, (gt, domain_name, path) in enumerate(zip(batch["texts"], batch["domains"], batch["paths"])):
                gt_norm = normalize_for_eval(gt, ascii_lower=False)
                row = {"path": path, "domain": domain_name, "gt": gt_norm}
                preds_for_row: Dict[str, str] = {}
                for mode in modes:
                    d = decoded[mode][i]
                    pred = normalize_for_eval(d["text"], ascii_lower=False)
                    preds_for_row[mode] = pred
                    conf = float(d.get("conf", 0.0))
                    update_stats(per_mode[mode][domain_name], pred, gt_norm, conf, d)
                    update_stats(global_mode[mode], pred, gt_norm, conf, d)
                    row[f"pred_{mode}"] = pred
                    row[f"conf_{mode}"] = conf
                    row[f"src_{mode}"] = d.get("src", mode)
                    row[f"blank_{mode}"] = d.get("blank_rate", 0.0)
                    row[f"unk_{mode}"] = d.get("unk_rate", 0.0)
                selected_src_counter[str(decoded["selected"][i].get("src", "selected"))] += 1
                if len(examples) < cfg.DEBUG_EXAMPLE_COUNT:
                    row["ok"] = preds_for_row["selected"] == gt_norm
                    row["edit_distance_selected"] = levenshtein(preds_for_row["selected"], gt_norm)
                    row["edit_distance_ctc"] = levenshtein(preds_for_row["ctc"], gt_norm)
                    row["edit_distance_ce"] = levenshtein(preds_for_row["ce"], gt_norm)
                    examples.append(row)
            # Release large CTW 64x640 feature tensors promptly. This is intentionally
            # conservative because Colab reports "session crashed" instead of a Python OOM.
            del images, domain_ids, outputs, decoded
            if batch_idx % max(1, int(getattr(cfg, "CROP_EVAL_CLEAN_EVERY", 10))) == 0:
                memory_safety_barrier(f"{current_dataset_tag()}-CropEval-b{batch_idx}")
                if current_dataset_key() == "ctw1500":
                    log(f"[{current_dataset_tag()}-CropEval] batch {batch_idx}/{len(loader)} completed safely")
    def summarize_stats(s: Dict[str, float]) -> Dict[str, float]:
        n = max(1, int(s["n"]))
        return {
            "n": int(s["n"]),
            "WRR": 100.0 * s["word_correct"] / n,
            "WRR_case_insensitive_aux": 100.0 * s["word_correct_ci"] / n,
            "WRR_alnum_aux": 100.0 * s["word_correct_alnum"] / n,
            "CRR_positional": 100.0 * s["char_correct"] / max(1, s["char_total"]),
            "CRR_edit_aux": 100.0 * s["edit_char_correct"] / max(1, s["edit_char_total"]),
            "empty_rate": 100.0 * s["empty"] / n,
            "edit_le1_acc": 100.0 * s["edit_le1"] / n,
            "edit_le2_acc": 100.0 * s["edit_le2"] / n,
            "avg_pred_len": float(s["pred_len"] / n),
            "avg_gt_len": float(s["gt_len"] / n),
            "conf_correct": float(s["conf_correct_sum"] / max(1, s["conf_correct_n"])),
            "conf_wrong": float(s["conf_wrong_sum"] / max(1, s["conf_wrong_n"])),
            "blank_rate": float(s["blank_rate_sum"] / n),
            "repeat_rate": float(s["repeat_rate_sum"] / n),
            "unk_rate": float(s["unk_rate_sum"] / n),
            "special_rate": float(s["special_rate_sum"] / n),
        }
    def summarize_mode(mode: str) -> Dict[str, Any]:
        rows = []
        total = new_stats()
        for domain_name in sorted(per_mode[mode].keys()):
            s = per_mode[mode][domain_name]
            row = summarize_stats(s)
            row["domain"] = domain_name
            rows.append(row)
            for k in total:
                total[k] += s.get(k, 0)
        g = summarize_stats(total)
        return {
            "macro_WRR": float(np.mean([r["WRR"] for r in rows])) if rows else 0.0,
            "macro_CRR": float(np.mean([r["CRR_positional"] for r in rows])) if rows else 0.0,
            "macro_CRR_edit_aux": float(np.mean([r["CRR_edit_aux"] for r in rows])) if rows else 0.0,
            "micro_WRR": g["WRR"],
            "micro_CRR": g["CRR_positional"],
            "micro_CRR_edit_aux": g["CRR_edit_aux"],
            "empty_pred_rate": g["empty_rate"],
            "edit_distance_le1_acc": g["edit_le1_acc"],
            "edit_distance_le2_acc": g["edit_le2_acc"],
            "avg_pred_len": g["avg_pred_len"],
            "avg_gt_len": g["avg_gt_len"],
            "conf_correct": g["conf_correct"],
            "conf_wrong": g["conf_wrong"],
            "blank_rate": g["blank_rate"],
            "repeat_rate": g["repeat_rate"],
            "unk_rate": g["unk_rate"],
            "special_rate": g["special_rate"],
            "per_domain": rows,
        }
    mode_summaries = {mode: summarize_mode(mode) for mode in modes}
    result = dict(mode_summaries["selected"])
    result.update({
        "epoch": int(epoch),
        "decode_mode": str(cfg.OCR_DECODE_MODE),
        "selected_source_counts": dict(selected_src_counter),
        "decode_summary": {
            "ctc_WRR": mode_summaries["ctc"].get("macro_WRR", 0.0),
            "ce_WRR": mode_summaries["ce"].get("macro_WRR", 0.0),
            "selected_WRR": mode_summaries["selected"].get("macro_WRR", 0.0),
            "winner": "ce" if mode_summaries["ce"].get("macro_WRR", 0.0) > mode_summaries["ctc"].get("macro_WRR", 0.0) else "ctc",
        },
        "selected_metrics": mode_summaries["selected"],
        "ctc_metrics": mode_summaries["ctc"],
        "ce_metrics": mode_summaries["ce"],
        "examples": examples,
        "official_protocol": "Total-Text auxiliary crop-recognition diagnostic only; no detection, polygon IoU, lexicon, fuzzy match, or transliteration.",
    })
    if out_json is not None:
        save_json(result, out_json)
    model.train()
    return result
# =============================================================================
# 7. CHECKPOINTS, OPTIMIZER, EMA, AND COMPACT LOGGING
# =============================================================================
def unwrap_model(model: nn.Module) -> nn.Module:
    return getattr(model, "_orig_mod", model)
def count_params(model: nn.Module) -> Tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable
def parameter_groups(model: nn.Module, lr_backbone: float, lr_head: float) -> List[Dict[str, Any]]:
    back, head = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "backbone" in name:
            back.append(p)
        else:
            head.append(p)
    return [{"params": back, "lr": lr_backbone}, {"params": head, "lr": lr_head}]
def save_ckpt(path: Path, model: nn.Module, optimizer, scaler, epoch: int, best_metric: float, extra: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model": unwrap_model(model).state_dict(),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_metric": float(best_metric),
        "extra": extra,
        "cfg": asdict(cfg),
        "arch_version": "final_DBPP_ArcFlowAlign_P2P4_SpecialistInit",
    }, path)
    log(f"[ckpt] saved: {path}")
def load_ckpt(path: Path, model: nn.Module, optimizer=None, scaler=None) -> Tuple[int, float, Dict[str, Any]]:
    if not path.exists() or not cfg.RESUME:
        return 0, -1e9, {}
    ckpt = torch.load(path, map_location="cpu")
    ckpt_arch = ckpt.get("arch_version", ckpt.get("cfg", {}).get("ARCH_VERSION", "unknown"))
    if ckpt_arch != cfg.ARCH_VERSION and not cfg.ALLOW_UNSAFE_RESUME:
        log(f"[ckpt][blocked] arch mismatch ckpt={ckpt_arch} current={cfg.ARCH_VERSION}; starting clean")
        return 0, -1e9, {}
    missing, unexpected = unwrap_model(model).load_state_dict(ckpt["model"], strict=False)
    log(f"[ckpt] loaded {path} | missing={len(missing)} unexpected={len(unexpected)}")
    if optimizer is not None and ckpt.get("optimizer") is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scaler is not None and ckpt.get("scaler") is not None:
        scaler.load_state_dict(ckpt["scaler"])
    return int(ckpt.get("epoch", 0)) + 1, float(ckpt.get("best_metric", -1e9)), ckpt.get("extra", {})
def load_model_weights_for_eval(path: Path, model: nn.Module, label: str) -> Dict[str, Any]:
    """Load only model weights for final evaluation/checkpoint selection.
    This intentionally ignores cfg.RESUME because it is used after a clean training
    run to evaluate the best train-internal validation checkpoint. It never looks
    at official test metrics, so it remains protocol-safe.
    """
    if not path.exists():
        log(f"[ckpt][warn] {label} checkpoint not found: {path}")
        return {"loaded": False, "path": str(path), "reason": "missing"}
    ckpt = torch.load(path, map_location="cpu")
    missing, unexpected = unwrap_model(model).load_state_dict(ckpt["model"], strict=False)
    info = {
        "loaded": True,
        "path": str(path),
        "epoch": int(ckpt.get("epoch", -1)),
        "best_metric": float(ckpt.get("best_metric", -1e9)),
        "missing": len(missing),
        "unexpected": len(unexpected),
    }
    log(f"[ckpt] loaded {label} weights for final/e2e use: {path} | epoch={info['epoch']} best={info['best_metric']:.3f} missing={len(missing)} unexpected={len(unexpected)}")
    return info
class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.997):
        self.decay = float(decay)
        self.shadow: Dict[str, torch.Tensor] = {}
        for k, v in unwrap_model(model).state_dict().items():
            if torch.is_floating_point(v):
                self.shadow[k] = v.detach().clone()
        self.backup: Dict[str, torch.Tensor] = {}
    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        msd = unwrap_model(model).state_dict()
        for k, v in msd.items():
            if k in self.shadow and torch.is_floating_point(v):
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
    @torch.no_grad()
    def apply_to(self, model: nn.Module) -> None:
        base = unwrap_model(model)
        self.backup = {}
        msd = base.state_dict()
        for k, v in msd.items():
            if k in self.shadow:
                self.backup[k] = v.detach().clone()
                v.copy_(self.shadow[k].to(device=v.device, dtype=v.dtype))
    @torch.no_grad()
    def restore(self, model: nn.Module) -> None:
        if not self.backup:
            return
        msd = unwrap_model(model).state_dict()
        for k, v in msd.items():
            if k in self.backup:
                v.copy_(self.backup[k].to(device=v.device, dtype=v.dtype))
        self.backup = {}
def maybe_compile_model(model: nn.Module, name: str) -> nn.Module:
    if not cfg.USE_TORCH_COMPILE or cfg.DEVICE != "cuda" or not hasattr(torch, "compile"):
        return model
    try:
        log(f"[{name}] torch.compile enabled: mode={cfg.TORCH_COMPILE_MODE}")
        return torch.compile(model, mode=cfg.TORCH_COMPILE_MODE)
    except Exception as e:
        log(f"[{name}][warn] torch.compile skipped: {repr(e)}")
        return model
def check_loss(loss: torch.Tensor, name: str) -> None:
    if torch.isnan(loss) or torch.isinf(loss):
        msg = f"[{name}] NaN/Inf loss: {loss.item()}"
        if cfg.STOP_ON_NAN:
            raise FloatingPointError(msg)
        log("[warn] " + msg)
def print_first_batch_debug(name: str, batch: Any) -> None:
    log(f"[debug:{name}] first batch")
    if isinstance(batch, dict):
        log(f"  images={tuple(batch['images'].shape)} ctc_targets={tuple(batch['ctc_targets'].shape)} ce={tuple(batch['ce_targets'].shape)}")
        log(f"  sample text={batch['texts'][0]} domain={batch['domains'][0]} path={batch['paths'][0]}")
    else:
        images, targets = batch
        log(f"  images={tuple(images.shape)} targets={len(targets)}")
        if targets:
            n_inst = [int(t['polys'].shape[0]) for t in targets[: min(3, len(targets))]]
            log(f"  sample path={targets[0].get('path', '')} instances(first)={n_inst}")
# =============================================================================
# 8. UNUSED VERIFICATION AND PAPER VISUALIZATION HELPERS
# =============================================================================
# =============================================================================
# 9. UNUSED TRAINING LOOP
# =============================================================================
# =============================================================================
# 8. TOTAL-TEXT FULL-IMAGE TEXT SPOTTING
# =============================================================================
def bbox_iou_np(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float32).reshape(4)
    b = np.asarray(b, dtype=np.float32).reshape(4)
    x1 = max(float(a[0]), float(b[0])); y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2])); y2 = min(float(a[3]), float(b[3]))
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, float(a[2] - a[0])) * max(0.0, float(a[3] - a[1]))
    area_b = max(0.0, float(b[2] - b[0])) * max(0.0, float(b[3] - b[1]))
    return float(inter / max(1e-6, area_a + area_b - inter))
def poly_to_bbox(poly: np.ndarray) -> np.ndarray:
    p = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    return np.array([p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max()], dtype=np.float32)
def resample_polygon(poly: np.ndarray, n: int) -> np.ndarray:
    p = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    if p.shape[0] == 0:
        return np.zeros((n, 2), dtype=np.float32)
    if p.shape[0] == 1:
        return np.repeat(p, n, axis=0)
    closed = np.concatenate([p, p[:1]], axis=0)
    seg = np.sqrt(((closed[1:] - closed[:-1]) ** 2).sum(axis=1))
    total = float(seg.sum())
    if total <= 1e-6:
        return np.repeat(p[:1], n, axis=0)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    samples = np.linspace(0.0, total, n, endpoint=False)
    out = []
    for s in samples:
        idx = int(np.searchsorted(cum, s, side="right") - 1)
        idx = min(max(idx, 0), len(seg) - 1)
        t = (s - cum[idx]) / max(float(seg[idx]), 1e-6)
        out.append(closed[idx] * (1.0 - t) + closed[idx + 1] * t)
    return np.asarray(out, dtype=np.float32)
def polygon_iou_mask(poly_a: np.ndarray, poly_b: np.ndarray, size: int = 256) -> float:
    pa = np.asarray(poly_a, dtype=np.float32).reshape(-1, 2).copy()
    pb = np.asarray(poly_b, dtype=np.float32).reshape(-1, 2).copy()
    if pa.shape[0] < 3 or pb.shape[0] < 3:
        return 0.0
    if pa.shape == pb.shape and np.allclose(pa, pb, atol=1e-4):
        return 1.0
    if cv2 is None:
        return bbox_iou_np(poly_to_bbox(pa), poly_to_bbox(pb))
    if max(float(np.nanmax(pa)), float(np.nanmax(pb))) <= 1.01:
        pa = pa * float(size - 1)
        pb = pb * float(size - 1)
    allp = np.concatenate([pa, pb], axis=0)
    mn = np.nanmin(allp, axis=0)
    mx = np.nanmax(allp, axis=0)
    wh = np.maximum(mx - mn, 1.0)
    pad = 2.0
    scale = (size - 1 - 2 * pad) / float(max(wh[0], wh[1], 1.0))
    pa2 = np.round((pa - mn) * scale + pad).astype(np.int32)
    pb2 = np.round((pb - mn) * scale + pad).astype(np.int32)
    ma = np.zeros((size, size), dtype=np.uint8)
    mb = np.zeros((size, size), dtype=np.uint8)
    cv2.fillPoly(ma, [pa2], 1)
    cv2.fillPoly(mb, [pb2], 1)
    inter = int(np.logical_and(ma, mb).sum())
    union = int(np.logical_or(ma, mb).sum())
    return float(inter / max(1, union))
def letterbox_image_and_polys(im: Image.Image, polygons: List[np.ndarray], out_size: int) -> Tuple[Image.Image, List[np.ndarray], Dict[str, float]]:
    w, h = im.size
    scale = min(out_size / max(1, w), out_size / max(1, h))
    nw, nh = int(round(w * scale)), int(round(h * scale))
    resized = im.resize((nw, nh), Image.BICUBIC)
    canvas = Image.new("RGB", (out_size, out_size), (128, 128, 128))
    pad_x = (out_size - nw) // 2
    pad_y = (out_size - nh) // 2
    canvas.paste(resized, (pad_x, pad_y))
    out_polys: List[np.ndarray] = []
    for poly in polygons:
        p = np.asarray(poly, dtype=np.float32).copy()
        p[:, 0] = p[:, 0] * scale + pad_x
        p[:, 1] = p[:, 1] * scale + pad_y
        p[:, 0] = np.clip(p[:, 0] / float(out_size), 0.0, 1.0)
        p[:, 1] = np.clip(p[:, 1] / float(out_size), 0.0, 1.0)
        out_polys.append(p.astype(np.float32))
    meta = {"orig_w": w, "orig_h": h, "scale": scale, "pad_x": pad_x, "pad_y": pad_y, "out_size": out_size}
    return canvas, out_polys, meta
def crop_polygon_bbox(im: Image.Image, poly: np.ndarray, pad_frac: float = 0.08) -> Image.Image:
    p = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    x1, y1 = float(p[:, 0].min()), float(p[:, 1].min())
    x2, y2 = float(p[:, 0].max()), float(p[:, 1].max())
    w, h = im.size
    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    pad = pad_frac * max(bw, bh)
    x1 = max(0, int(math.floor(x1 - pad))); y1 = max(0, int(math.floor(y1 - pad)))
    x2 = min(w, int(math.ceil(x2 + pad))); y2 = min(h, int(math.ceil(y2 + pad)))
    if x2 <= x1 or y2 <= y1:
        return im.copy().convert("RGB")
    return im.crop((x1, y1, x2, y2)).convert("RGB")
def crop_polygon_rectified(im: Image.Image, poly: np.ndarray, pad_frac: float = 0.06) -> Image.Image:
    if cv2 is None:
        return crop_polygon_bbox(im, poly, pad_frac)
    p = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    if p.shape[0] < 4:
        return crop_polygon_bbox(im, p, pad_frac)
    arr = np.asarray(im.convert("RGB"))
    rect = cv2.minAreaRect(p.astype(np.float32))
    (cx, cy), (rw, rh), angle = rect
    rw, rh = max(2.0, float(rw)) * (1.0 + pad_frac), max(2.0, float(rh)) * (1.0 + pad_frac)
    box = cv2.boxPoints(((cx, cy), (rw, rh), angle)).astype(np.float32)
    ssum = box.sum(axis=1)
    diff = np.diff(box, axis=1).reshape(-1)
    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = box[np.argmin(ssum)]
    ordered[2] = box[np.argmax(ssum)]
    ordered[1] = box[np.argmin(diff)]
    ordered[3] = box[np.argmax(diff)]
    out_w = int(round(max(np.linalg.norm(ordered[1] - ordered[0]), np.linalg.norm(ordered[2] - ordered[3]))))
    out_h = int(round(max(np.linalg.norm(ordered[3] - ordered[0]), np.linalg.norm(ordered[2] - ordered[1]))))
    out_w, out_h = max(8, out_w), max(8, out_h)
    if out_h > out_w * 1.15:
        out_w, out_h = out_h, out_w
        ordered = np.array([ordered[1], ordered[2], ordered[3], ordered[0]], dtype=np.float32)
    dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]], dtype=np.float32)
    try:
        M = cv2.getPerspectiveTransform(ordered, dst)
        warped = cv2.warpPerspective(arr, M, (out_w, out_h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        return Image.fromarray(warped).convert("RGB")
    except Exception:
        return crop_polygon_bbox(im, p, pad_frac)
def crop_polygon_masked(im: Image.Image, poly: np.ndarray, pad_frac: float = 0.10) -> Image.Image:
    p = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    if p.shape[0] < 3:
        return crop_polygon_bbox(im, p, pad_frac)
    base = im.convert("RGB")
    w, h = base.size
    x1, y1 = float(p[:, 0].min()), float(p[:, 1].min())
    x2, y2 = float(p[:, 0].max()), float(p[:, 1].max())
    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    pad = pad_frac * max(bw, bh)
    ix1 = max(0, int(math.floor(x1 - pad))); iy1 = max(0, int(math.floor(y1 - pad)))
    ix2 = min(w, int(math.ceil(x2 + pad))); iy2 = min(h, int(math.ceil(y2 + pad)))
    if ix2 <= ix1 or iy2 <= iy1:
        return crop_polygon_bbox(im, p, pad_frac)
    crop = base.crop((ix1, iy1, ix2, iy2))
    local = p.copy(); local[:, 0] -= ix1; local[:, 1] -= iy1
    mask = Image.new("L", crop.size, 0)
    ImageDraw.Draw(mask).polygon([tuple(map(float, xy)) for xy in local], fill=255)
    gray = Image.new("RGB", crop.size, (128, 128, 128))
    return Image.composite(crop, gray, mask).convert("RGB")
def _dataset_crop_pad_frac() -> float:
    # v5: CTW1500 text-line crops need more context around ascenders, descenders,
    # curved baselines, and phrase-level punctuation. This is preprocessing only;
    # it does not change the model architecture.
    if str(getattr(cfg, "CURRENT_DATASET_KEY", "total_text")) == "ctw1500":
        return float(getattr(cfg, "CTW_GT_CROP_PAD_FRAC", 0.12))
    return float(getattr(cfg, "DEFAULT_CROP_PAD_FRAC", 0.06))
def crop_totaltext_by_mode(im: Image.Image, poly: np.ndarray, mode: str) -> Image.Image:
    mode = str(mode).lower()
    pad_frac = _dataset_crop_pad_frac()
    if mode == "bbox":
        return crop_polygon_bbox(im, poly, pad_frac=max(0.08, pad_frac))
    if mode in {"rectified", "rotated", "minarea", "ctw_line_rectified"}:
        return crop_polygon_rectified(im, poly, pad_frac=pad_frac)
    if mode in {"masked", "mask", "polygon"}:
        return crop_polygon_masked(im, poly, pad_frac=max(0.10, pad_frac))
    return crop_polygon_rectified(im, poly, pad_frac=pad_frac)
class TotalTextFullImageDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]], train: bool, img_size: Optional[int] = None):
        self.records = records
        self.train = train
        self.img_size = int(img_size or cfg.TOTALTEXT_IMG_SIZE)
    def __len__(self) -> int:
        return len(self.records)
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict[str, Any]]:
        r = self.records[idx]
        im = load_image_rgb(Path(r["image"]))
        inst = list(r.get("instances", []))
        if self.train and len(inst) > cfg.TOTALTEXT_MAX_INSTANCES:
            valid = [x for x in inst if not x.get("ignore", False)]
            ign = [x for x in inst if x.get("ignore", False)]
            inst = valid[: cfg.TOTALTEXT_MAX_INSTANCES] + ign[: max(0, cfg.TOTALTEXT_MAX_INSTANCES - len(valid))]
        raw_polys = [np.asarray(x["polygon"], dtype=np.float32) for x in inst]
        im_lb, norm_polys, meta = letterbox_image_and_polys(im, raw_polys, self.img_size)
        boxes, polys, texts, ignore_flags, text_ignore_flags = [], [], [], [], []
        arc_center, arc_upper, arc_lower, arc_tau, arc_width, arc_vis = [], [], [], [], [], []
        for obj, p in zip(inst, norm_polys):
            if p.shape[0] < 3:
                continue
            rp = resample_polygon(p, cfg.TOTALTEXT_POLY_POINTS)
            boxes.append(poly_to_bbox(rp))
            polys.append(rp)
            texts.append(normalize_for_eval(obj.get("text", ""), ascii_lower=True))
            ignore_flags.append(bool(obj.get("ignore", False)))
            text_ignore_flags.append(bool(obj.get("text_ignore", False)))
            g = instance_to_arcflow_geometry(obj, rp, meta)
            arc_center.append(g["centerline"])
            arc_upper.append(g["upper"])
            arc_lower.append(g["lower"])
            arc_tau.append(g["tau"])
            arc_width.append(g["width"])
            arc_vis.append(g["visibility"])
        k = int(cfg.ARCFLOW_K_POINTS)
        target = {
            "boxes": torch.tensor(np.asarray(boxes, dtype=np.float32)) if boxes else torch.zeros((0, 4), dtype=torch.float32),
            "polys": torch.tensor(np.asarray(polys, dtype=np.float32)) if polys else torch.zeros((0, cfg.TOTALTEXT_POLY_POINTS, 2), dtype=torch.float32),
            "texts": texts,
            "ignore": torch.tensor(ignore_flags, dtype=torch.bool) if ignore_flags else torch.zeros((0,), dtype=torch.bool),
            "text_ignore": torch.tensor(text_ignore_flags, dtype=torch.bool) if text_ignore_flags else torch.zeros((0,), dtype=torch.bool),
            "arc_centerline": torch.tensor(np.asarray(arc_center, dtype=np.float32)) if arc_center else torch.zeros((0, k, 2), dtype=torch.float32),
            "arc_upper": torch.tensor(np.asarray(arc_upper, dtype=np.float32)) if arc_upper else torch.zeros((0, k, 2), dtype=torch.float32),
            "arc_lower": torch.tensor(np.asarray(arc_lower, dtype=np.float32)) if arc_lower else torch.zeros((0, k, 2), dtype=torch.float32),
            "arc_tau": torch.tensor(np.asarray(arc_tau, dtype=np.float32)) if arc_tau else torch.zeros((0, k, 2), dtype=torch.float32),
            "arc_width": torch.tensor(np.asarray(arc_width, dtype=np.float32)) if arc_width else torch.zeros((0, k), dtype=torch.float32),
            "arc_visibility": torch.tensor(np.asarray(arc_vis, dtype=np.float32)) if arc_vis else torch.zeros((0, k), dtype=torch.float32),
            "path": r["image"],
            "meta": meta,
            "img_size": self.img_size,
        }
        return pil_to_tensor(im_lb), target
class TotalTextCropDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]], tokenizer: TextTokenizer, train: bool):
        self.items: List[Dict[str, Any]] = []
        self.tokenizer = tokenizer
        self.train = train
        modes = list(cfg.TT_CROP_MODES)
        for rec in records:
            for obj in rec.get("instances", []):
                text = normalize_for_eval(obj.get("text", ""), ascii_lower=True)
                if obj.get("ignore", False) or obj.get("text_ignore", False) or not text:
                    continue
                base = {"image": rec["image"], "polygon": np.asarray(obj["polygon"], dtype=np.float32), "text": text}
                if train:
                    self.items.append(base)
                else:
                    for m in modes:
                        z = dict(base); z["crop_mode"] = m; self.items.append(z)
    def __len__(self) -> int:
        return len(self.items)
    def _choose_mode(self, item: Dict[str, Any]) -> str:
        if not self.train:
            return str(item.get("crop_mode", cfg.TT_CROP_MODES[0]))
        modes = list(cfg.TT_CROP_MODES)
        weights = list(cfg.TT_CROP_MODE_WEIGHTS)
        if len(weights) != len(modes):
            weights = [1.0] * len(modes)
        return random.choices(modes, weights=weights, k=1)[0]
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        r = self.items[idx]
        mode = self._choose_mode(r)
        im = load_image_rgb(Path(r["image"]))
        crop = crop_totaltext_by_mode(im, np.asarray(r["polygon"], dtype=np.float32), mode)
        if self.train:
            crop = augment_str_image(crop)
        crop = letterbox_rgb(crop, (cfg.STR_HEIGHT, cfg.OCR_FIXED_W))
        text = normalize_for_eval(r["text"], ascii_lower=True)
        return {
            "image": pil_to_tensor(crop),
            "ctc_ids": torch.tensor(self.tokenizer.encode_ctc(text), dtype=torch.long),
            "ce_ids": torch.tensor(self.tokenizer.encode_ce(text, cfg.MAX_TEXT_LEN), dtype=torch.long),
            "domain_id": torch.tensor(0, dtype=torch.long),
            "domain": "total_text_latin",
            "crop_mode": mode,
            "text": text,
            "path": r["image"],
        }
def collate_tt_full(batch: List[Tuple[torch.Tensor, Dict[str, Any]]]) -> Tuple[torch.Tensor, List[Dict[str, Any]]]:
    return torch.stack([x[0] for x in batch], dim=0), [x[1] for x in batch]
def make_totaltext_split(records: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    val_n = min(int(cfg.TT_DB_INTERNAL_VAL_IMAGES), max(1, len(records) // 10))
    rng = random.Random(cfg.SEED + 11)
    idx = list(range(len(records)))
    rng.shuffle(idx)
    val_idx = set(idx[:val_n])
    train = [r for i, r in enumerate(records) if i not in val_idx]
    val = [r for i, r in enumerate(records) if i in val_idx]
    return train, val
# =============================================================================
# 8A. final TOTAL-TEXT GEOMETRY: FPN P2-P4 + ArcFlowAlign
# =============================================================================
def _as_np_points(x: Any) -> np.ndarray:
    """Convert list/JSON/tensor point fields into a clean Nx2 float32 array."""
    if x is None:
        return np.zeros((0, 2), dtype=np.float32)
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    if isinstance(x, dict):
        # tau_points may come as [{"x":..., "y":..., "visible":...}, ...]
        if "x" in x and "y" in x:
            return np.asarray([[float(x["x"]), float(x["y"])]], dtype=np.float32)
    if isinstance(x, (list, tuple)) and x and isinstance(x[0], dict):
        pts = []
        for p in x:
            if "x" in p and "y" in p:
                pts.append([float(p["x"]), float(p["y"])])
        return np.asarray(pts, dtype=np.float32)
    arr = np.asarray(x, dtype=np.float32)
    if arr.ndim == 1 and arr.size >= 2:
        arr = arr.reshape(-1, 2)
    if arr.ndim != 2 or arr.shape[1] != 2:
        return np.zeros((0, 2), dtype=np.float32)
    return arr.astype(np.float32)
def resample_polyline_np(points: np.ndarray, k: int) -> np.ndarray:
    """Arc-length resampling of an open polyline."""
    pts = _as_np_points(points)
    k = int(k)
    if pts.shape[0] == 0:
        return np.zeros((k, 2), dtype=np.float32)
    if pts.shape[0] == 1:
        return np.repeat(pts[:1], k, axis=0).astype(np.float32)
    seg = np.linalg.norm(pts[1:] - pts[:-1], axis=1)
    dist = np.concatenate([[0.0], np.cumsum(seg)])
    total = float(dist[-1])
    if total <= 1e-6:
        return np.repeat(pts[:1], k, axis=0).astype(np.float32)
    qs = np.linspace(0.0, total, k, dtype=np.float32)
    out = np.zeros((k, 2), dtype=np.float32)
    out[:, 0] = np.interp(qs, dist, pts[:, 0])
    out[:, 1] = np.interp(qs, dist, pts[:, 1])
    return out.astype(np.float32)
def _transform_raw_points_to_letterbox_norm(points: Any, meta: Dict[str, float]) -> np.ndarray:
    """Map raw image-coordinate available geometry to current letterboxed [0,1] space."""
    pts = _as_np_points(points)
    if pts.shape[0] == 0:
        return pts
    # Already normalized fields stay normalized.
    if float(np.nanmax(np.abs(pts))) <= 1.5:
        return np.clip(pts.astype(np.float32), 0.0, 1.0)
    scale = float(meta.get("scale", 1.0))
    pad_x = float(meta.get("pad_x", 0.0))
    pad_y = float(meta.get("pad_y", 0.0))
    out_size = float(meta.get("out_size", cfg.TOTALTEXT_IMG_SIZE))
    q = pts.copy().astype(np.float32)
    q[:, 0] = (q[:, 0] * scale + pad_x) / max(1.0, out_size)
    q[:, 1] = (q[:, 1] * scale + pad_y) / max(1.0, out_size)
    return np.clip(q, 0.0, 1.0).astype(np.float32)
def polygon_to_arcflow_geometry_np(poly_norm: np.ndarray, k: Optional[int] = None) -> Dict[str, np.ndarray]:
    """Derive approximate reading-flow geometry from a polygon.
    For real Total-Text, only polygons are available. This derives a stable
    centerline, upper/lower boundaries, width and visibility using a PCA split
    followed by arc-length resampling. This function derives the geometry used by the Total-Text training path.
    """
    k = int(k or cfg.ARCFLOW_K_POINTS)
    pts = _as_np_points(poly_norm)
    if pts.shape[0] < 3:
        pts = np.array([[0.1, 0.4], [0.9, 0.4], [0.9, 0.6], [0.1, 0.6]], dtype=np.float32)
    pts = np.clip(pts.astype(np.float32), 0.0, 1.0)
    dense = resample_polygon(pts, max(32, 2 * k)).astype(np.float32)
    mean = dense.mean(axis=0, keepdims=True)
    centered = dense - mean
    try:
        _, _, vh = np.linalg.svd(centered, full_matrices=False)
        axis = vh[0].astype(np.float32)
    except Exception:
        axis = np.array([1.0, 0.0], dtype=np.float32)
    axis = axis / max(1e-6, float(np.linalg.norm(axis)))
    normal = np.array([-axis[1], axis[0]], dtype=np.float32)
    t = (dense - mean) @ axis
    r = (dense - mean) @ normal
    # Split boundary by transverse sign around median. Then interpolate each side.
    med = float(np.median(r))
    upper_pts = dense[r >= med]
    lower_pts = dense[r < med]
    if upper_pts.shape[0] < 2 or lower_pts.shape[0] < 2:
        rect = cv2.minAreaRect((pts * 1000).astype(np.float32)) if cv2 is not None else None
        x1, y1, x2, y2 = poly_to_bbox(pts)
        upper = np.stack([np.linspace(x1, x2, k), np.linspace(y1, y1, k)], axis=1).astype(np.float32)
        lower = np.stack([np.linspace(x1, x2, k), np.linspace(y2, y2, k)], axis=1).astype(np.float32)
    else:
        def _sort_and_resample(side_pts: np.ndarray) -> np.ndarray:
            side_t = (side_pts - mean) @ axis
            order = np.argsort(side_t.reshape(-1))
            return resample_polyline_np(side_pts[order], k)
        upper = _sort_and_resample(upper_pts)
        lower = _sort_and_resample(lower_pts)
    # Make the reading direction left-to-right in image coordinates for stable CTC crops.
    center = (upper + lower) * 0.5
    if center[-1, 0] < center[0, 0]:
        upper = upper[::-1].copy()
        lower = lower[::-1].copy()
        center = center[::-1].copy()
    width = np.linalg.norm(upper - lower, axis=1).astype(np.float32)
    visibility = np.ones((k,), dtype=np.float32)
    tau = center.copy().astype(np.float32)
    return {
        "centerline": np.clip(center, 0.0, 1.0).astype(np.float32),
        "upper": np.clip(upper, 0.0, 1.0).astype(np.float32),
        "lower": np.clip(lower, 0.0, 1.0).astype(np.float32),
        "tau": np.clip(tau, 0.0, 1.0).astype(np.float32),
        "width": np.clip(width, 1e-4, 1.0).astype(np.float32),
        "visibility": visibility,
    }
def instance_to_arcflow_geometry(obj: Dict[str, Any], norm_poly: np.ndarray, meta: Dict[str, float]) -> Dict[str, np.ndarray]:
    """Use exact available geometry when present, otherwise derive from polygon."""
    k = int(cfg.ARCFLOW_K_POINTS)
    if all(key in obj for key in ("centerline_ctrl", "upper_ctrl", "lower_ctrl")):
        center = resample_polyline_np(_transform_raw_points_to_letterbox_norm(obj.get("centerline_ctrl"), meta), k)
        upper = resample_polyline_np(_transform_raw_points_to_letterbox_norm(obj.get("upper_ctrl"), meta), k)
        lower = resample_polyline_np(_transform_raw_points_to_letterbox_norm(obj.get("lower_ctrl"), meta), k)
        tau_raw = obj.get("tau_points", None)
        tau = resample_polyline_np(_transform_raw_points_to_letterbox_norm(tau_raw, meta), k) if tau_raw is not None else center.copy()
        width = np.linalg.norm(upper - lower, axis=1).astype(np.float32)
        vis = np.asarray(obj.get("visibility", np.ones((k,), dtype=np.float32)), dtype=np.float32).reshape(-1)
        if vis.shape[0] != k:
            vis = np.interp(np.linspace(0, max(0, len(vis) - 1), k), np.arange(len(vis)), vis).astype(np.float32) if len(vis) else np.ones((k,), dtype=np.float32)
        return {
            "centerline": np.clip(center, 0.0, 1.0).astype(np.float32),
            "upper": np.clip(upper, 0.0, 1.0).astype(np.float32),
            "lower": np.clip(lower, 0.0, 1.0).astype(np.float32),
            "tau": np.clip(tau, 0.0, 1.0).astype(np.float32),
            "width": np.clip(width, 1e-4, 1.0).astype(np.float32),
            "visibility": np.clip(vis, 0.0, 1.0).astype(np.float32),
        }
    return polygon_to_arcflow_geometry_np(norm_poly, k)
def arcflow_geometry_to_polygon_np(upper: np.ndarray, lower: np.ndarray) -> np.ndarray:
    pts = np.concatenate([_as_np_points(upper), _as_np_points(lower)[::-1]], axis=0)
    if pts.shape[0] > cfg.TOTALTEXT_POLY_POINTS:
        pts = resample_polygon(pts.astype(np.float32), cfg.TOTALTEXT_POLY_POINTS)
    return np.clip(pts.astype(np.float32), 0.0, 1.0)
class SimpleFPNP2P4(nn.Module):
    """Lightweight P2-P4 FPN for DB++ small-text feature fusion.
    The OCR recognizer uses the backbone C3 path. Total-Text
    DB++ heads use P2 high-resolution features.
    """
    def __init__(self, in_channels: Dict[str, int], out_ch: int):
        super().__init__()
        self.lat2 = nn.Conv2d(in_channels["c2"], out_ch, 1)
        self.lat3 = nn.Conv2d(in_channels["c3"], out_ch, 1)
        self.lat4 = nn.Conv2d(in_channels["c4"], out_ch, 1)
        self.smooth2 = nn.Sequential(nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.smooth3 = nn.Sequential(nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.smooth4 = nn.Sequential(nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self._shape_logged = False
    def forward(self, feats: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        p4 = self.lat4(feats["c4"])
        p3 = self.lat3(feats["c3"]) + F.interpolate(p4, size=feats["c3"].shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lat2(feats["c2"]) + F.interpolate(p3, size=feats["c2"].shape[-2:], mode="bilinear", align_corners=False)
        out = {"p2": self.smooth2(p2), "p3": self.smooth3(p3), "p4": self.smooth4(p4)}
        if not self._shape_logged:
            shape_map = {k: list(v.shape) for k, v in out.items()}
            log(f"[model] fpn_feature_shapes all_outputs_projected_to_{cfg.D_MODEL}_channels shapes={shape_map}")
            self._shape_logged = True
        return out
class ArcFlowAlignHead(nn.Module):
    """Denoising geometry head for ordered arc-length reading flow.
    Input geometry is a noisy or predicted centerline/boundary hypothesis.
    The head samples image features along the centerline, predicts small residuals,
    and is trained to recover the GT exact geometry.
    """
    def __init__(self, in_ch: int):
        super().__init__()
        hidden = cfg.D_MODEL
        self.mlp = nn.Sequential(
            nn.Linear(in_ch + 6, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        # center dxdy, upper dxdy, lower dxdy, tau dxdy, width delta
        self.delta = nn.Linear(hidden, 9)
    def sample(self, feat: torch.Tensor, points01: torch.Tensor, batch_ids: torch.Tensor) -> torch.Tensor:
        """Sample FPN features at normalized arc-flow points.
        Important AMP fix: ``grid_sample`` requires ``input`` and ``grid`` to
        have the same floating dtype on CUDA. During final evaluation the FPN
        feature map can be fp16 because it was produced under autocast, while
        the geometry tensors are fp32. Cast the grid to the selected feature
        dtype before calling ``grid_sample``. This prevents the final-test crash:
        ``RuntimeError: expected scalar type Half but found Float``.
        """
        if points01.numel() == 0:
            return torch.zeros((0, int(cfg.ARCFLOW_K_POINTS), feat.shape[1]), dtype=feat.dtype, device=feat.device)
        points01 = points01.clamp(0.0, 1.0)
        sel = feat[batch_ids.clamp(min=0, max=feat.shape[0] - 1)]
        grid = points01.mul(2.0).sub(1.0).unsqueeze(2).to(device=sel.device, dtype=sel.dtype)  # M,K,1,2
        sampled = F.grid_sample(sel, grid, mode="bilinear", padding_mode="border", align_corners=False)
        return sampled.squeeze(-1).permute(0, 2, 1).contiguous()  # M,K,C
    def forward(self, feat: torch.Tensor, init: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        center = init["centerline"].clamp(0.0, 1.0)
        upper = init["upper"].clamp(0.0, 1.0)
        lower = init["lower"].clamp(0.0, 1.0)
        tau = init["tau"].clamp(0.0, 1.0)
        width = init["width"].clamp(1e-4, 1.0)
        vis = init.get("visibility", torch.ones_like(width)).clamp(0.0, 1.0)
        batch_ids = init["batch_ids"].long()
        sf = self.sample(feat, center, batch_ids)
        # Keep the tiny ArcFlow MLP in parameter dtype (normally fp32). This is
        # safer outside autocast and avoids half-input/float-weight dtype issues.
        mlp_dtype = next(self.mlp.parameters()).dtype
        sf_mlp = sf.to(dtype=mlp_dtype)
        geom = torch.cat([center, tau, width.unsqueeze(-1), vis.unsqueeze(-1)], dim=-1).to(device=sf.device, dtype=mlp_dtype)
        h = self.mlp(torch.cat([sf_mlp, geom], dim=-1))
        delta = torch.tanh(self.delta(h)) * float(cfg.ARCFLOW_DELTA_SCALE)
        center_f = center.to(dtype=delta.dtype)
        upper_f = upper.to(dtype=delta.dtype)
        lower_f = lower.to(dtype=delta.dtype)
        tau_f = tau.to(dtype=delta.dtype)
        width_f = width.to(dtype=delta.dtype)
        pred_center = (center_f + delta[..., 0:2]).clamp(0.0, 1.0)
        pred_upper = (upper_f + delta[..., 2:4]).clamp(0.0, 1.0)
        pred_lower = (lower_f + delta[..., 4:6]).clamp(0.0, 1.0)
        pred_tau = (tau_f + delta[..., 6:8]).clamp(0.0, 1.0)
        pred_width = (width_f + delta[..., 8]).clamp(1e-4, 1.0)
        return {
            "centerline": pred_center,
            "upper": pred_upper,
            "lower": pred_lower,
            "tau": pred_tau,
            "width": pred_width,
            "visibility": vis,
            "sampled_features": sf,
        }
class ArcBridgeAdapter(nn.Module):
    """Zero-output ArcFlow-to-recognition feature bridge.
    This module is intentionally conservative. The final projection is initialized
    to zero, so the adapter output is exactly the base feature at initialization:
        Z_final = Z_base + Bridge([Z_base, Z_arc, geometry])
    This allows testing the bridge without corrupting the stable DB++ or
    crop-recognizer behavior. The bridge is trained with a feature-alignment loss
    between noisy/predicted ArcFlow samples and GT centerline samples. It prepares
    the architecture while keeping the
    official Total-Text E2E protocol unchanged.
    """
    def __init__(self, d_model: int):
        super().__init__()
        hidden = int(getattr(cfg, "ARCBRIDGE_HIDDEN", d_model))
        self.geom = nn.Linear(6, d_model)
        self.net = nn.Sequential(
            nn.LayerNorm(d_model * 3),
            nn.Linear(d_model * 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, d_model),
        )
        # Zero-output initialization: identity residual path at step 0.
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
    def forward(self, z_base: torch.Tensor, z_arc: torch.Tensor, geom: torch.Tensor) -> torch.Tensor:
        if z_base.numel() == 0:
            return z_base
        param_dtype = next(self.parameters()).dtype
        z_base_f = z_base.to(dtype=param_dtype)
        z_arc_f = z_arc.to(dtype=param_dtype)
        geom_f = geom.to(device=z_base.device, dtype=param_dtype)
        g = self.geom(geom_f)
        delta = self.net(torch.cat([z_base_f, z_arc_f, g], dim=-1))
        return (z_base_f + delta).to(dtype=z_base.dtype)
def arcbridge_feature_loss(model: nn.Module, model_out: Dict[str, Any], targets: List[Dict[str, Any]], train: bool = True, epoch: int = 999) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Gentle feature-alignment objective for ArcBridge.
    The loss is intentionally feature-level, not an official metric. It makes the
    bridge learn to convert noisy/predicted ArcFlow-sampled features toward the
    GT centerline-sampled feature distribution. This directly targets the
    geometry/OCR interface without changing Total-Text headline evaluation.
    """
    base = unwrap_model(model) if "unwrap_model" in globals() else model
    if (not bool(getattr(cfg, "ENABLE_ARCBRIDGE", True))) or (not hasattr(base, "arcbridge")) or (not hasattr(base, "arcflow_head")):
        z = model_out["db_logits"].sum() * 0.0
        return z, {"bridge_loss": 0.0, "bridge_instances": 0, "bridge_active": 0}
    if int(epoch) <= int(getattr(cfg, "ARCBRIDGE_DELAY_EPOCHS", 0)):
        z = model_out["db_logits"].sum() * 0.0
        return z, {"bridge_loss": 0.0, "bridge_instances": 0, "bridge_active": 0}
    feat = model_out.get("fpn", {}).get("p2", None)
    if feat is None:
        z = model_out["db_logits"].sum() * 0.0
        return z, {"bridge_loss": 0.0, "bridge_instances": 0, "bridge_active": 0}
    gt = _arcflow_flatten_targets(targets, feat.device)
    if gt is None:
        z = model_out["db_logits"].sum() * 0.0
        return z, {"bridge_loss": 0.0, "bridge_instances": 0, "bridge_active": 0}
    init = _noisy_arcflow_init(gt, train=train)
    with torch.no_grad():
        pred = base.arcflow_head(feat, init)
        z_teacher = base.arcflow_head.sample(feat, gt["centerline"], gt["batch_ids"]).detach()
    z_base = base.arcflow_head.sample(feat, init["centerline"], gt["batch_ids"])
    z_arc = base.arcflow_head.sample(feat, pred["centerline"], gt["batch_ids"])
    geom = torch.cat([gt["centerline"], gt["tau"], gt["width"].unsqueeze(-1), gt["visibility"].unsqueeze(-1)], dim=-1)
    z_bridge = base.arcbridge(z_base, z_arc, geom)
    vis = gt["visibility"].float()
    cos = F.cosine_similarity(z_bridge.float(), z_teacher.float(), dim=-1)
    loss = ((1.0 - cos) * vis).sum() / vis.sum().clamp_min(1.0)
    stats = {
        "bridge_loss": float(loss.detach().cpu()),
        "bridge_instances": int(gt["centerline"].shape[0]),
        "bridge_active": 1,
    }
    return loss, stats
def _arcflow_flatten_targets(targets: List[Dict[str, Any]], device: torch.device) -> Optional[Dict[str, torch.Tensor]]:
    centers = []; uppers = []; lowers = []; taus = []; widths = []; vises = []; batch_ids = []
    max_inst = int(getattr(cfg, "ARCFLOW_MAX_INSTANCES_PER_IMAGE", 48))
    for bi, tgt in enumerate(targets):
        ignore = tgt.get("ignore", torch.zeros((0,), dtype=torch.bool))
        ignore_np = ignore.detach().cpu().numpy().astype(bool) if isinstance(ignore, torch.Tensor) else np.asarray(ignore, dtype=bool)
        for key, store in [("arc_centerline", centers), ("arc_upper", uppers), ("arc_lower", lowers), ("arc_tau", taus), ("arc_width", widths), ("arc_visibility", vises)]:
            if key not in tgt:
                break
        n_added_for_image = 0
        if all(k in tgt for k in ("arc_centerline", "arc_upper", "arc_lower", "arc_tau", "arc_width", "arc_visibility")):
            n = int(tgt["arc_centerline"].shape[0])
            for i in range(n):
                if i < len(ignore_np) and bool(ignore_np[i]):
                    continue
                if n_added_for_image >= max_inst:
                    break
                centers.append(tgt["arc_centerline"][i])
                uppers.append(tgt["arc_upper"][i])
                lowers.append(tgt["arc_lower"][i])
                taus.append(tgt["arc_tau"][i])
                widths.append(tgt["arc_width"][i])
                vises.append(tgt["arc_visibility"][i])
                batch_ids.append(bi)
                n_added_for_image += 1
        else:
            polys = tgt.get("polys", torch.zeros((0, cfg.TOTALTEXT_POLY_POINTS, 2)))
            polys_np = polys.detach().cpu().numpy() if isinstance(polys, torch.Tensor) else np.asarray(polys, dtype=np.float32)
            for i, p in enumerate(polys_np):
                if i < len(ignore_np) and bool(ignore_np[i]):
                    continue
                if n_added_for_image >= max_inst:
                    break
                g = polygon_to_arcflow_geometry_np(p, int(cfg.ARCFLOW_K_POINTS))
                centers.append(torch.tensor(g["centerline"], dtype=torch.float32))
                uppers.append(torch.tensor(g["upper"], dtype=torch.float32))
                lowers.append(torch.tensor(g["lower"], dtype=torch.float32))
                taus.append(torch.tensor(g["tau"], dtype=torch.float32))
                widths.append(torch.tensor(g["width"], dtype=torch.float32))
                vises.append(torch.tensor(g["visibility"], dtype=torch.float32))
                batch_ids.append(bi)
                n_added_for_image += 1
    if not centers:
        return None
    def _stack(xs): return torch.stack([x if isinstance(x, torch.Tensor) else torch.tensor(x, dtype=torch.float32) for x in xs], dim=0).to(device)
    return {
        "centerline": _stack(centers),
        "upper": _stack(uppers),
        "lower": _stack(lowers),
        "tau": _stack(taus),
        "width": _stack(widths),
        "visibility": _stack(vises),
        "batch_ids": torch.tensor(batch_ids, dtype=torch.long, device=device),
    }
def _noisy_arcflow_init(gt: Dict[str, torch.Tensor], train: bool = True) -> Dict[str, torch.Tensor]:
    init = {k: v.clone() if torch.is_tensor(v) else v for k, v in gt.items()}
    if train and float(getattr(cfg, "ARCFLOW_DENOISE_NOISE_STD", 0.0)) > 0:
        std = float(cfg.ARCFLOW_DENOISE_NOISE_STD)
        for key in ("centerline", "upper", "lower", "tau"):
            noise = torch.randn_like(init[key]) * std
            init[key] = (init[key] + noise * init["visibility"].unsqueeze(-1)).clamp(0.0, 1.0)
        init["width"] = (init["width"] + torch.randn_like(init["width"]) * std).clamp(1e-4, 1.0)
    return init
def _masked_smooth_l1(pred: torch.Tensor, target: torch.Tensor, visibility: torch.Tensor) -> torch.Tensor:
    if pred.ndim == 3:
        mask = visibility.unsqueeze(-1).to(dtype=pred.dtype)
    else:
        mask = visibility.to(dtype=pred.dtype)
    loss = F.smooth_l1_loss(pred.float(), target.float(), reduction="none")
    return (loss * mask.float()).sum() / mask.float().sum().clamp_min(1.0)
def arcflow_supervision_loss(model: nn.Module, model_out: Dict[str, Any], targets: List[Dict[str, Any]], train: bool = True) -> Tuple[torch.Tensor, Dict[str, float]]:
    base = unwrap_model(model) if "unwrap_model" in globals() else model
    if not bool(getattr(cfg, "ENABLE_ARCFLOW_ALIGN", True)) or not hasattr(base, "arcflow_head"):
        z = model_out["db_logits"].sum() * 0.0
        return z, {"arc_loss": 0.0, "arc_curve": 0.0, "arc_boundary": 0.0, "arc_tau": 0.0, "arc_width": 0.0, "arc_align": 0.0, "arc_instances": 0}
    feat = model_out.get("fpn", {}).get("p2", None)
    if feat is None:
        # fallback for old checkpoint compatibility; not used in final path
        feat = model_out.get("features", {}).get("c3", None)
    if feat is None:
        z = model_out["db_logits"].sum() * 0.0
        return z, {"arc_loss": 0.0, "arc_curve": 0.0, "arc_boundary": 0.0, "arc_tau": 0.0, "arc_width": 0.0, "arc_align": 0.0, "arc_instances": 0}
    gt = _arcflow_flatten_targets(targets, feat.device)
    if gt is None:
        z = model_out["db_logits"].sum() * 0.0
        return z, {"arc_loss": 0.0, "arc_curve": 0.0, "arc_boundary": 0.0, "arc_tau": 0.0, "arc_width": 0.0, "arc_align": 0.0, "arc_instances": 0}
    init = _noisy_arcflow_init(gt, train=train)
    pred = base.arcflow_head(feat, init)
    vis = gt["visibility"]
    curve = _masked_smooth_l1(pred["centerline"], gt["centerline"], vis)
    boundary = 0.5 * (_masked_smooth_l1(pred["upper"], gt["upper"], vis) + _masked_smooth_l1(pred["lower"], gt["lower"], vis))
    tau = _masked_smooth_l1(pred["tau"], gt["tau"], vis)
    width = _masked_smooth_l1(pred["width"], gt["width"], vis)
    with torch.no_grad():
        teacher = base.arcflow_head.sample(feat, gt["centerline"], gt["batch_ids"]).detach()
    student = base.arcflow_head.sample(feat, pred["centerline"], gt["batch_ids"])
    cos = F.cosine_similarity(student.float(), teacher.float(), dim=-1)
    align = ((1.0 - cos) * vis.float()).sum() / vis.float().sum().clamp_min(1.0)
    total = (
        float(cfg.ARCFLOW_WEIGHT_CURVE) * curve
        + float(cfg.ARCFLOW_WEIGHT_BOUNDARY) * boundary
        + float(cfg.ARCFLOW_WEIGHT_TAU) * tau
        + float(cfg.ARCFLOW_WEIGHT_WIDTH) * width
        + float(cfg.ARCFLOW_WEIGHT_ALIGN) * align
    )
    stats = {
        "arc_loss": float(total.detach().cpu()),
        "arc_curve": float(curve.detach().cpu()),
        "arc_boundary": float(boundary.detach().cpu()),
        "arc_tau": float(tau.detach().cpu()),
        "arc_width": float(width.detach().cpu()),
        "arc_align": float(align.detach().cpu()),
        "arc_instances": int(gt["centerline"].shape[0]),
    }
    return total, stats
def refine_pred_batches_with_arcflow(model: nn.Module, images_cpu: List[torch.Tensor], pred_batches: List[List[Dict[str, Any]]]) -> List[List[Dict[str, Any]]]:
    """Refine DB++ polygons through ArcFlowAlign before OCR/E2E evaluation."""
    if not bool(getattr(cfg, "ARCFLOW_REFINE_PRED_POLYS_AT_EVAL", True)):
        return pred_batches
    base = unwrap_model(model) if "unwrap_model" in globals() else model
    if not hasattr(base, "arcflow_head"):
        return pred_batches
    was_train = base.training
    base.eval()
    refined_batches: List[List[Dict[str, Any]]] = []
    with torch.no_grad():
        for img_t, preds in zip(images_cpu, pred_batches):
            if not preds:
                refined_batches.append([])
                continue
            image_gpu = img_t.unsqueeze(0).to(cfg.DEVICE)
            with autocast(device_type="cuda", dtype=get_amp_dtype(), enabled=(cfg.DEVICE == "cuda")):
                out = base.forward_db(image_gpu) if hasattr(base, "forward_db") else base(image_gpu)
                feat = out.get("fpn", {}).get("p2", None)
            if feat is None:
                refined_batches.append(preds)
                continue
            # Evaluation can produce fp16 FPN features under autocast. ArcFlow
            # refinement is light and geometry-sensitive, so run it in fp32.
            feat = feat.float()
            geoms = [polygon_to_arcflow_geometry_np(np.asarray(p["poly"], dtype=np.float32), int(cfg.ARCFLOW_K_POINTS)) for p in preds]
            init = {
                "centerline": torch.tensor(np.stack([g["centerline"] for g in geoms]), dtype=torch.float32, device=feat.device),
                "upper": torch.tensor(np.stack([g["upper"] for g in geoms]), dtype=torch.float32, device=feat.device),
                "lower": torch.tensor(np.stack([g["lower"] for g in geoms]), dtype=torch.float32, device=feat.device),
                "tau": torch.tensor(np.stack([g["tau"] for g in geoms]), dtype=torch.float32, device=feat.device),
                "width": torch.tensor(np.stack([g["width"] for g in geoms]), dtype=torch.float32, device=feat.device),
                "visibility": torch.tensor(np.stack([g["visibility"] for g in geoms]), dtype=torch.float32, device=feat.device),
                "batch_ids": torch.zeros((len(geoms),), dtype=torch.long, device=feat.device),
            }
            pred = base.arcflow_head(feat, init)
            new_batch: List[Dict[str, Any]] = []
            for j, p in enumerate(preds):
                q = dict(p)
                upper = pred["upper"][j].detach().float().cpu().numpy()
                lower = pred["lower"][j].detach().float().cpu().numpy()
                poly = arcflow_geometry_to_polygon_np(upper, lower)
                q["poly_before_arcflow"] = np.asarray(p["poly"], dtype=np.float32)
                q["poly"] = poly.astype(np.float32)
                q["box"] = poly_to_bbox(q["poly"])
                q["arcflow_refined"] = True
                new_batch.append(q)
            refined_batches.append(new_batch)
    if was_train:
        base.train()
    return refined_batches
def text_size_bucket_from_poly(poly_norm: np.ndarray) -> str:
    x1, y1, x2, y2 = poly_to_bbox(np.asarray(poly_norm, dtype=np.float32))
    side = max((x2 - x1), (y2 - y1)) * float(cfg.TOTALTEXT_IMG_SIZE)
    if side < float(cfg.TT_SMALL_TEXT_MIN_SIDE_PX):
        return "small"
    if side < float(cfg.TT_MEDIUM_TEXT_MIN_SIDE_PX):
        return "medium"
    return "large"
def compute_size_recall_breakdown(pred_batches: List[List[Dict[str, Any]]], targets: List[Dict[str, Any]], out_csv: Optional[Path] = None) -> Dict[str, Any]:
    buckets = {b: {"gt": 0, "matched": 0} for b in ("small", "medium", "large")}
    for preds, tgt in zip(pred_batches, targets):
        gt_polys = tgt.get("polys", torch.zeros((0, cfg.TOTALTEXT_POLY_POINTS, 2))).detach().cpu().numpy()
        ignore = tgt.get("ignore", torch.zeros((len(gt_polys),), dtype=torch.bool)).detach().cpu().numpy().astype(bool)
        used = set()
        for gi, gp in enumerate(gt_polys):
            if gi < len(ignore) and bool(ignore[gi]):
                continue
            b = text_size_bucket_from_poly(gp)
            buckets[b]["gt"] += 1
            best_iou, best_j = 0.0, -1
            for pj, p in enumerate(preds):
                if pj in used:
                    continue
                iou = polygon_iou_mask(np.asarray(p["poly"], dtype=np.float32), gp)
                if iou > best_iou:
                    best_iou, best_j = iou, pj
            if best_iou >= float(cfg.TT_MATCH_IOU) and best_j >= 0:
                buckets[b]["matched"] += 1
                used.add(best_j)
    rows = []
    out = {}
    for b, d in buckets.items():
        recall = 100.0 * d["matched"] / max(1, d["gt"])
        out[f"{b}_gt"] = int(d["gt"])
        out[f"{b}_matched"] = int(d["matched"])
        out[f"{b}_recall"] = float(recall)
        rows.append({"bucket": b, "gt": int(d["gt"]), "matched": int(d["matched"]), "recall": float(recall)})
    if out_csv is not None:
        write_csv(rows, out_csv)
    return out
def make_totaltext_image_sampler(records: List[Dict[str, Any]]) -> Optional[WeightedRandomSampler]:
    if not bool(getattr(cfg, "TT_SMALL_TEXT_OVERSAMPLE", True)):
        return None
    weights = []
    for r in records:
        w = 1.0
        for obj in r.get("instances", []):
            if obj.get("ignore", False):
                continue
            p = np.asarray(obj.get("polygon", []), dtype=np.float32)
            if p.ndim == 2 and p.shape[0] >= 3:
                # Raw image coordinates. Estimate from bbox side length.
                x1, y1, x2, y2 = poly_to_bbox(p)
                if max(x2 - x1, y2 - y1) < float(cfg.TT_SMALL_TEXT_MIN_SIDE_PX):
                    w += float(cfg.TT_SMALL_TEXT_OVERSAMPLE_BOOST)
        weights.append(float(w))
    if not weights:
        return None
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
class DBAuxHead(nn.Module):
    def __init__(self, in_ch: int):
        super().__init__()
        hidden = 128
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, 1, 1),
        )
    def forward(self, feat: torch.Tensor, out_size: Optional[Tuple[int, int]] = None) -> torch.Tensor:
        logits = self.net(feat)
        if out_size is not None and tuple(logits.shape[-2:]) != tuple(out_size):
            logits = F.interpolate(logits, size=out_size, mode="bilinear", align_corners=False)
        return logits
class TotalTextDBModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = TotalTextBackbone(cfg.BACKBONE_NAME, cfg.PRETRAIN_IMAGENET)
        self.fpn = SimpleFPNP2P4(self.backbone.out_channels, cfg.D_MODEL)
        self.db_head = DBAuxHead(cfg.D_MODEL)
        self.arcflow_head = ArcFlowAlignHead(cfg.D_MODEL)
        self.arcbridge = ArcBridgeAdapter(cfg.D_MODEL)
    def forward(self, images: torch.Tensor) -> Dict[str, torch.Tensor]:
        feats = self.backbone(images)
        fpn = self.fpn(feats)
        logits = self.db_head(fpn["p2"], out_size=(images.shape[-2], images.shape[-1]))
        return {"db_logits": logits, "features": feats, "fpn": fpn}
def polys_to_db_mask(targets: List[Dict[str, Any]], h: int, w: int, device: torch.device) -> torch.Tensor:
    if cv2 is None:
        raise RuntimeError("OpenCV/cv2 is required for DB++ mask generation and polygon extraction.")
    masks = []
    for tgt in targets:
        mask_np = np.zeros((h, w), dtype=np.uint8)
        polys = tgt.get("polys", torch.zeros((0, cfg.TOTALTEXT_POLY_POINTS, 2))).detach().cpu().numpy()
        ignore = tgt.get("ignore", torch.zeros((len(polys),), dtype=torch.bool)).detach().cpu().numpy().astype(bool)
        for p, ign in zip(polys, ignore):
            if ign:
                continue
            pts = np.asarray(p, dtype=np.float32).reshape(-1, 2).copy()
            if pts.shape[0] < 3:
                continue
            shrink = float(cfg.TT_DB_MASK_SHRINK_RATIO)
            if shrink < 0.999:
                c = pts.mean(axis=0, keepdims=True)
                pts = c + (pts - c) * max(0.05, shrink)
            pts[:, 0] = np.clip(pts[:, 0] * (w - 1), 0, w - 1)
            pts[:, 1] = np.clip(pts[:, 1] * (h - 1), 0, h - 1)
            cv2.fillPoly(mask_np, [np.round(pts).astype(np.int32)], 1)
        masks.append(torch.from_numpy(mask_np.astype(np.float32)))
    return torch.stack(masks, dim=0).unsqueeze(1).to(device)
def polys_to_db_mask_and_weight(targets: List[Dict[str, Any]], h: int, w: int, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    """DB++ target mask plus optional care-region weights.
    V7 default restores the empirically stronger v4 detector supervision:
    readable text polygons are positives, everything else is background for DB loss.
    Geometry-level ignore regions are still respected in evaluation/OCR, but they are
    NOT hard-neutralized in DB loss by default because that hurt ICDAR/CTW detector
    gradients in v6.
    Ablation modes:
      - v4_default: no hard care mask, preserves v4 detector behavior.
      - soft_neutral: downweights ignore polygons to DB_IGNORE_SOFT_WEIGHT.
      - hard_neutral: zero-weights ignore polygons, the v6 behavior.
    """
    mask = polys_to_db_mask(targets, h, w, device)
    weights = torch.ones_like(mask)
    ignore_mode = str(getattr(cfg, "DB_IGNORE_MODE", "v4_default")).lower()
    if ignore_mode in {"soft_neutral", "hard_neutral"}:
        ignore_weight = 0.0 if ignore_mode == "hard_neutral" else float(getattr(cfg, "DB_IGNORE_SOFT_WEIGHT", 0.05))
        for bi, tgt in enumerate(targets):
            polys = tgt.get("polys", torch.zeros((0, cfg.TOTALTEXT_POLY_POINTS, 2))).detach().cpu().numpy()
            ignore = tgt.get("ignore", torch.zeros((len(polys),), dtype=torch.bool)).detach().cpu().numpy().astype(bool)
            for p, ign in zip(polys, ignore):
                if not ign:
                    continue
                local_mask = np.zeros((h, w), dtype=np.uint8)
                pts = np.asarray(p, dtype=np.float32).copy()
                if pts.shape[0] < 3 or cv2 is None:
                    continue
                pts[:, 0] = np.clip(pts[:, 0] * (w - 1), 0, w - 1)
                pts[:, 1] = np.clip(pts[:, 1] * (h - 1), 0, h - 1)
                cv2.fillPoly(local_mask, [np.round(pts).astype(np.int32)], 1)
                neutral = torch.from_numpy(local_mask.astype(np.float32)).to(device=device)
                weights[bi, 0] = weights[bi, 0] * (1.0 - neutral) + ignore_weight * neutral
    # Optional small-text positive boost, only on positive regions.
    if bool(getattr(cfg, "TT_SMALL_TEXT_OVERSAMPLE", True)) or float(getattr(cfg, "TT_SMALL_TEXT_POS_BOOST", 0.0)) > 0:
        boost = float(getattr(cfg, "TT_SMALL_TEXT_POS_BOOST", 2.0))
        if boost > 0:
            for bi, tgt in enumerate(targets):
                polys = tgt.get("polys", torch.zeros((0, cfg.TOTALTEXT_POLY_POINTS, 2))).detach().cpu().numpy()
                ignore = tgt.get("ignore", torch.zeros((len(polys),), dtype=torch.bool)).detach().cpu().numpy().astype(bool)
                for p, ign in zip(polys, ignore):
                    if ign:
                        continue
                    bucket = text_size_bucket_from_poly(p)
                    if bucket != "small":
                        continue
                    local_mask = np.zeros((h, w), dtype=np.uint8)
                    pts = np.asarray(p, dtype=np.float32).copy()
                    pts[:, 0] = np.clip(pts[:, 0] * (w - 1), 0, w - 1)
                    pts[:, 1] = np.clip(pts[:, 1] * (h - 1), 0, h - 1)
                    if cv2 is not None and pts.shape[0] >= 3:
                        cv2.fillPoly(local_mask, [np.round(pts).astype(np.int32)], 1)
                        weights[bi, 0] = weights[bi, 0] + torch.from_numpy(local_mask.astype(np.float32)).to(device) * boost
    return mask, weights
def db_dice_loss(logits: torch.Tensor, target: torch.Tensor, care: Optional[torch.Tensor] = None) -> torch.Tensor:
    prob = torch.sigmoid(logits.float())
    tgt = target.float()
    if care is None:
        care = torch.ones_like(tgt)
    care = care.float().to(device=logits.device)
    inter = (prob * tgt * care).sum(dim=(1, 2, 3))
    denom = (prob * care).sum(dim=(1, 2, 3)) + (tgt * care).sum(dim=(1, 2, 3)) + 1e-6
    return (1.0 - (2.0 * inter + 1e-6) / denom).mean()
def db_focal_loss(logits: torch.Tensor, target: torch.Tensor, care: Optional[torch.Tensor] = None) -> torch.Tensor:
    prob = torch.sigmoid(logits.float())
    tgt = target.float()
    bce = F.binary_cross_entropy_with_logits(logits.float(), tgt, reduction="none")
    pt = prob * tgt + (1.0 - prob) * (1.0 - tgt)
    alpha_t = float(cfg.TT_DB_FOCAL_ALPHA) * tgt + (1.0 - float(cfg.TT_DB_FOCAL_ALPHA)) * (1.0 - tgt)
    raw = alpha_t * (1.0 - pt).pow(float(cfg.TT_DB_FOCAL_GAMMA)) * bce
    if care is None:
        return raw.mean()
    care = care.to(device=logits.device, dtype=raw.dtype)
    return (raw * care).sum() / care.sum().clamp_min(1.0)
def db_loss(logits: torch.Tensor, mask: torch.Tensor, weight_map: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, float]]:
    pos_weight = torch.tensor(float(cfg.TT_DB_POS_WEIGHT), dtype=torch.float32, device=logits.device)
    if weight_map is None:
        raw_bce = F.binary_cross_entropy_with_logits(logits.float(), mask.float(), pos_weight=pos_weight, reduction="none")
        weight_map = torch.ones_like(raw_bce)
    else:
        raw_bce = F.binary_cross_entropy_with_logits(logits.float(), mask.float(), pos_weight=pos_weight, reduction="none")
        weight_map = weight_map.to(device=logits.device, dtype=raw_bce.dtype)
    # In V7 default this is all ones. In optional soft/hard neutral ablations,
    # pixels with zero weight are excluded from dice/focal as well.
    care = (weight_map > 0).to(dtype=raw_bce.dtype, device=logits.device)
    bce = (raw_bce * weight_map).sum() / weight_map.sum().clamp_min(1.0)
    dice = db_dice_loss(logits, mask, care=care)
    focal = db_focal_loss(logits, mask, care=care)
    loss = float(cfg.TT_DB_BCE_WEIGHT) * bce + float(cfg.TT_DB_DICE_WEIGHT) * dice + float(cfg.TT_DB_FOCAL_WEIGHT) * focal
    with torch.no_grad():
        prob = torch.sigmoid(logits.float())
        pred_pos = float(((prob > 0.5).float() * care).sum().detach().cpu() * 100.0 / care.sum().detach().cpu().clamp_min(1.0))
        target_pos = float((mask.float() * care).sum().detach().cpu() * 100.0 / care.sum().detach().cpu().clamp_min(1.0))
        textness_mean = float((prob * care).sum().detach().cpu() / care.sum().detach().cpu().clamp_min(1.0))
        textness_max = float((prob * care).max().detach().cpu()) if care.sum() > 0 else float(prob.max().detach().cpu())
        ignored_pct = float((1.0 - care).mean().detach().cpu() * 100.0)
    return loss, {"loss": float(loss.detach().cpu()), "bce": float(bce.detach().cpu()), "dice": float(dice.detach().cpu()), "focal": float(focal.detach().cpu()), "pred_pos_pct": pred_pos, "target_pos_pct": target_pos, "ignored_pct": ignored_pct, "textness_mean": textness_mean, "textness_max": textness_max}
def db_postprocess_prob(prob: np.ndarray, settings: Dict[str, Any]) -> List[Dict[str, Any]]:
    """DB++ probability-map postprocess with Geometry-aware geometry/alignment controls.
    The detector remains DB++; these settings only affect polygon extraction and
    crop alignment. They are selected on train-internal validation, never on the
    official test set.
    """
    if cv2 is None:
        raise RuntimeError("cv2 is required for DB postprocessing")
    prob = np.asarray(prob, dtype=np.float32)
    h, w = prob.shape[:2]
    th = float(settings.get("threshold", cfg.TT_DB_DEFAULT_THRESHOLD))
    mode = str(settings.get("mode", cfg.TT_DB_DEFAULT_MODE))
    unshrink = float(settings.get("unshrink", cfg.TT_DB_DEFAULT_UNSHRINK))
    min_area = float(settings.get("min_area", cfg.TT_DB_DEFAULT_MIN_AREA))
    max_preds = int(settings.get("max_preds", cfg.TT_DB_MAX_PREDS))
    morph = str(settings.get("morph", "none"))
    crop_pad = float(settings.get("crop_pad", cfg.TT_CROP_RECTIFY_PAD))
    bitmap = (prob >= th).astype(np.uint8) * 255
    # Tiny text often appears fragmented. A conservative 3x3 close/dilate option
    # is swept on train-internal validation to improve recall without hard-coding.
    if morph != "none":
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        if morph == "close":
            bitmap = cv2.morphologyEx(bitmap, cv2.MORPH_CLOSE, kernel, iterations=1)
        elif morph == "dilate":
            bitmap = cv2.dilate(bitmap, kernel, iterations=1)
    contours, _ = cv2.findContours(bitmap, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    preds = []
    for cnt in contours:
        if cnt is None or len(cnt) < 3:
            continue
        area = float(cv2.contourArea(cnt))
        if area < min_area:
            continue
        if mode == "minrect":
            pts = cv2.boxPoints(cv2.minAreaRect(cnt)).astype(np.float32)
        else:
            # Smaller epsilon preserves curved/irregular text contours better than
            # over-aggressive simplification, which was a known alignment bottleneck.
            eps = 0.0035 * cv2.arcLength(cnt, True)
            pts = cv2.approxPolyDP(cnt, eps, True).reshape(-1, 2).astype(np.float32)
            if pts.shape[0] < 3:
                pts = cnt.reshape(-1, 2).astype(np.float32)
        c = pts.mean(axis=0, keepdims=True)
        pts = c + (pts - c) * unshrink
        pts[:, 0] = np.clip(pts[:, 0] / max(1, w - 1), 0.0, 1.0)
        pts[:, 1] = np.clip(pts[:, 1] / max(1, h - 1), 0.0, 1.0)
        if pts.shape[0] > cfg.TOTALTEXT_POLY_POINTS:
            pts = resample_polygon(pts, cfg.TOTALTEXT_POLY_POINTS)
        x1, y1, x2, y2 = poly_to_bbox(pts * np.array([w - 1, h - 1], dtype=np.float32))
        crop = prob[max(0, int(y1)): min(h, int(y2) + 1), max(0, int(x1)): min(w, int(x2) + 1)]
        score = float(crop.max()) if crop.size else float(prob.max())
        preds.append({
            "poly": pts.astype(np.float32),
            "score": score,
            "box": poly_to_bbox(pts),
            "crop_pad": crop_pad,
            "morph": morph,
        })
    preds.sort(key=lambda x: float(x["score"]), reverse=True)
    return preds[:max_preds]
def tensor_to_pil_unnorm(x: torch.Tensor) -> Image.Image:
    t = x.detach().cpu().float().clone()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    t = (t * std + mean).clamp(0.0, 1.0)
    arr = (t.permute(1, 2, 0).numpy() * 255.0).astype(np.uint8)
    return Image.fromarray(arr).convert("RGB")
def expand_poly_abs_for_crop(poly_abs: np.ndarray, pad: float, width: int, height: int) -> np.ndarray:
    """Expand a predicted polygon around its centroid before rectified OCR.
    Small under-shrink and contour jitter were the main source of the GT-vs-pred
    E2E gap. This padding is selected by internal validation via `crop_pad` and
    is applied only to the OCR crop, not to detection matching.
    """
    pts = np.asarray(poly_abs, dtype=np.float32).copy()
    if pts.ndim != 2 or pts.shape[0] < 3:
        return pts
    pad = float(pad)
    if abs(pad - 1.0) < 1e-6:
        return pts
    c = pts.mean(axis=0, keepdims=True)
    pts = c + (pts - c) * pad
    pts[:, 0] = np.clip(pts[:, 0], 0, max(1, width - 1))
    pts[:, 1] = np.clip(pts[:, 1], 0, max(1, height - 1))
    return pts
def recognize_crops_from_pred_polys(rec_model: nn.Module, images_cpu: List[torch.Tensor], pred_batches: List[List[Dict[str, Any]]], tokenizer: TextTokenizer) -> List[List[Dict[str, Any]]]:
    """Recognize predicted/GT polygon crops with batch-level OCR decoding.

    Earlier versions decoded crops image-by-image, so full-image eval with batch=12
    still launched many tiny OCR forwards. V7.17 keeps the exact crop/rectify/decode
    logic, but flattens all crops in the current full-image batch, decodes them in
    TT_CROP_EVAL_BATCH_SIZE chunks, then restores the original per-image grouping.
    If the T4 overflows, the OCR chunk size is halved and retried.
    """
    rec_model.eval()
    device = cfg.DEVICE
    old_decode = cfg.OCR_DECODE_MODE
    cfg.OCR_DECODE_MODE = str(cfg.TT_CROP_DECODE_MODE)
    flat_crops: List[torch.Tensor] = []
    flat_meta: List[Dict[str, Any]] = []
    counts: List[int] = []
    try:
        for img_t, preds in zip(images_cpu, pred_batches):
            counts.append(len(preds))
            if not preds:
                continue
            pil = tensor_to_pil_unnorm(img_t)
            for p in preds:
                poly_abs = np.asarray(p["poly"], dtype=np.float32).copy()
                poly_abs[:, 0] *= pil.width - 1
                poly_abs[:, 1] *= pil.height - 1
                crop_pad = float(p.get("crop_pad", cfg.TT_CROP_RECTIFY_PAD))
                poly_abs = expand_poly_abs_for_crop(poly_abs, crop_pad, pil.width, pil.height)
                crop = crop_totaltext_by_mode(pil, poly_abs, "rectified")
                crop = letterbox_rgb(crop, (cfg.STR_HEIGHT, cfg.OCR_FIXED_W))
                flat_crops.append(pil_to_tensor(crop))
                flat_meta.append(p)

        if not flat_crops:
            return [[] for _ in pred_batches]

        texts: List[str] = []
        start_i = 0
        crop_eval_bs = max(1, int(cfg.TT_CROP_EVAL_BATCH_SIZE))
        while start_i < len(flat_crops):
            end_i = min(len(flat_crops), start_i + crop_eval_bs)
            try:
                batch = torch.stack(flat_crops[start_i:end_i], dim=0).to(device, non_blocking=True)
                batch = optimize_image_tensor_for_cuda(batch)
                domain_ids = torch.zeros((batch.shape[0],), dtype=torch.long, device=device)
                with torch.inference_mode():
                    with autocast(device_type="cuda", dtype=get_amp_dtype(), enabled=(cfg.DEVICE == "cuda")):
                        outputs = rec_model(batch, domain_ids=domain_ids)
                    dec = _decode_str_batch(outputs, tokenizer, epoch=999)
                texts.extend([normalize_for_eval(x["text"], ascii_lower=True) for x in dec["selected"]])
                start_i = end_i
                del batch, domain_ids, outputs, dec
            except RuntimeError as e:
                if is_cuda_oom(e) and crop_eval_bs > 1:
                    recover_from_oom(f"{current_dataset_tag()}-crop-ocr-split-b{crop_eval_bs}")
                    crop_eval_bs = max(1, crop_eval_bs // 2)
                    continue
                raise

        out_batches: List[List[Dict[str, Any]]] = []
        cursor = 0
        for preds, n in zip(pred_batches, counts):
            batch_out: List[Dict[str, Any]] = []
            for p in preds:
                q = dict(p)
                q["text"] = texts[cursor] if cursor < len(texts) else ""
                batch_out.append(q)
                cursor += 1
            out_batches.append(batch_out)
        return out_batches
    finally:
        cfg.OCR_DECODE_MODE = old_decode
        rec_model.train()

def _bbox_could_overlap(box_a: Any, box_b: Any, eps: float = 1e-6) -> bool:
    """Cheap bbox rejection before expensive polygon IoU. Preserves metric logic."""
    try:
        a = np.asarray(box_a, dtype=np.float32).reshape(4)
        b = np.asarray(box_b, dtype=np.float32).reshape(4)
        return bool((min(a[2], b[2]) - max(a[0], b[0]) > eps) and (min(a[3], b[3]) - max(a[1], b[1]) > eps))
    except Exception:
        return True
def evaluate_polygon_batches(pred_batches: List[List[Dict[str, Any]]], targets: List[Dict[str, Any]], out_json: Optional[Path] = None) -> Dict[str, Any]:
    det_tp = det_fp = det_fn = 0
    e2e_tp = e2e_fp = e2e_fn = 0
    pred_counts, max_scores = [], []
    examples: List[Dict[str, Any]] = []
    for preds, tgt in zip(pred_batches, targets):
        gt_polys = tgt["polys"].detach().cpu().numpy()
        gt_texts = [normalize_for_eval(t, ascii_lower=True) for t in tgt.get("texts", [])]
        ignore = tgt["ignore"].detach().cpu().numpy().astype(bool)
        text_ignore_t = tgt.get("text_ignore", torch.zeros((len(gt_polys),), dtype=torch.bool))
        text_ignore = text_ignore_t.detach().cpu().numpy().astype(bool) if isinstance(text_ignore_t, torch.Tensor) else np.asarray(text_ignore_t, dtype=bool)
        valid_idx = [i for i in range(len(gt_polys)) if not ignore[i]]
        e2e_valid_idx = [i for i in valid_idx if i >= len(text_ignore) or not bool(text_ignore[i])]
        ignore_idx = [i for i in range(len(gt_polys)) if ignore[i]]
        gt_boxes = [poly_to_bbox(g) for g in gt_polys]
        filtered = []
        for p in preds:
            p_box = p.get("box", poly_to_bbox(p["poly"]))
            ign_overlap = 0.0
            for ii in ignore_idx:
                if not _bbox_could_overlap(p_box, gt_boxes[ii]):
                    continue
                ign_overlap = max(ign_overlap, polygon_iou_mask(p["poly"], gt_polys[ii]))
            if ign_overlap >= float(cfg.TT_IGNORE_IOU):
                continue
            filtered.append(p)
        preds = filtered
        pred_counts.append(len(preds))
        max_scores.append(max([float(p.get("score", 0.0)) for p in preds], default=0.0))
        matched_gt, matched_pred = set(), set()
        for pi in sorted(range(len(preds)), key=lambda i: float(preds[i].get("score", 0.0)), reverse=True):
            best_iou, best_gi = 0.0, None
            p_box = preds[pi].get("box", poly_to_bbox(preds[pi]["poly"]))
            for gi in valid_idx:
                if gi in matched_gt:
                    continue
                if not _bbox_could_overlap(p_box, gt_boxes[gi]):
                    continue
                iou = polygon_iou_mask(preds[pi]["poly"], gt_polys[gi])
                if iou > best_iou:
                    best_iou, best_gi = iou, gi
            if best_gi is not None and best_iou >= float(cfg.TT_MATCH_IOU):
                det_tp += 1
                matched_gt.add(best_gi); matched_pred.add(pi)
                pred_text = normalize_for_eval(preds[pi].get("text", ""), ascii_lower=True)
                gt_text = gt_texts[best_gi] if best_gi < len(gt_texts) else ""
                if best_gi in e2e_valid_idx:
                    if pred_text == gt_text:
                        e2e_tp += 1
                    else:
                        e2e_fp += 1; e2e_fn += 1
                # If GT is text_ignore, the detection match is valid but the
                # OCR/E2E comparison is neutral.
        det_fp += max(0, len(preds) - len(matched_pred))
        det_fn += max(0, len(valid_idx) - len(matched_gt))
        e2e_fp += max(0, len(preds) - len(matched_pred))
        e2e_fn += max(0, len([gi for gi in e2e_valid_idx if gi not in matched_gt]))
        if len(examples) < 50:
            examples.append({
                "path": tgt.get("path", ""),
                "gt": [{"text": gt_texts[i], "ignore": bool(ignore[i]), "text_ignore": bool(text_ignore[i]) if i < len(text_ignore) else False} for i in range(len(gt_texts))],
                "pred": [{"text": p.get("text", ""), "score": float(p.get("score", 0.0))} for p in preds[:10]],
            })
    def prf(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
        p = 100.0 * tp / max(1, tp + fp)
        r = 100.0 * tp / max(1, tp + fn)
        f = 2 * p * r / max(1e-9, p + r)
        return float(p), float(r), float(f)
    det_p, det_r, det_f = prf(det_tp, det_fp, det_fn)
    none_p, none_r, none_f = prf(e2e_tp, e2e_fp, e2e_fn)
    result = {
        "det_precision": det_p, "det_recall": det_r, "det_f": det_f,
        "e2e_none_precision": none_p, "e2e_none_recall": none_r, "e2e_none_f": none_f,
        "avg_preds_per_image": float(np.mean(pred_counts)) if pred_counts else 0.0,
        "avg_max_score": float(np.mean(max_scores)) if max_scores else 0.0,
        "counts": {"det_tp": det_tp, "det_fp": det_fp, "det_fn": det_fn, "e2e_none_tp": e2e_tp, "e2e_none_fp": e2e_fp, "e2e_none_fn": e2e_fn},
        "examples": examples,
        "metric_scale_note": "Percent metrics in [0,100]. Total-Text headline is full-image Det/E2E, not crop WA.",
    }
    if out_json is not None:
        save_json(result, out_json)
    return result
# =============================================================================
# v6 OFFICIAL-STYLE LEXICON REPORTING
# =============================================================================
def _target_lexicon_texts(targets: List[Dict[str, Any]], per_image: bool = False) -> Any:
    """Collect normalized evaluation GT text. text_ignore instances are excluded."""
    if per_image:
        per: List[List[str]] = []
        for tgt in targets:
            gt_texts = [normalize_for_eval(t, ascii_lower=True) for t in tgt.get("texts", [])]
            ignore = tgt.get("ignore", torch.zeros((len(gt_texts),), dtype=torch.bool))
            ignore_np = ignore.detach().cpu().numpy().astype(bool) if isinstance(ignore, torch.Tensor) else np.asarray(ignore, dtype=bool)
            text_ignore_t = tgt.get("text_ignore", torch.zeros((len(gt_texts),), dtype=torch.bool))
            text_ignore_np = text_ignore_t.detach().cpu().numpy().astype(bool) if isinstance(text_ignore_t, torch.Tensor) else np.asarray(text_ignore_t, dtype=bool)
            words = sorted({gt_texts[i] for i in range(len(gt_texts)) if i < len(ignore_np) and not ignore_np[i] and (i >= len(text_ignore_np) or not text_ignore_np[i]) and gt_texts[i]})
            per.append(words)
        return per
    words = []
    for tgt in targets:
        gt_texts = [normalize_for_eval(t, ascii_lower=True) for t in tgt.get("texts", [])]
        ignore = tgt.get("ignore", torch.zeros((len(gt_texts),), dtype=torch.bool))
        ignore_np = ignore.detach().cpu().numpy().astype(bool) if isinstance(ignore, torch.Tensor) else np.asarray(ignore, dtype=bool)
        text_ignore_t = tgt.get("text_ignore", torch.zeros((len(gt_texts),), dtype=torch.bool))
        text_ignore_np = text_ignore_t.detach().cpu().numpy().astype(bool) if isinstance(text_ignore_t, torch.Tensor) else np.asarray(text_ignore_t, dtype=bool)
        for i, txt in enumerate(gt_texts):
            if i < len(ignore_np) and ignore_np[i]:
                continue
            if i < len(text_ignore_np) and text_ignore_np[i]:
                continue
            if txt:
                words.append(txt)
    return sorted(set(words))
def _normalized_edit_distance(a: str, b: str) -> float:
    return float(levenshtein(a, b) / max(1, max(len(a), len(b))))
def _lexicon_correct_text(text: str, lexicon: Sequence[str], mode: str, dataset_key: str) -> Tuple[str, Dict[str, Any]]:
    pred = normalize_for_eval(text, ascii_lower=True)
    lex = [normalize_for_eval(x, ascii_lower=True) for x in lexicon if normalize_for_eval(x, ascii_lower=True)]
    if not lex:
        return pred, {"used": False, "reason": "empty_lexicon"}
    best = min(lex, key=lambda w: levenshtein(pred, w))
    dist = int(levenshtein(pred, best))
    if dataset_key == "ctw1500":
        ned = _normalized_edit_distance(pred, best)
        ok = ned <= float(getattr(cfg, "OFFICIAL_CTW_FULL_NED_THRESH", 0.30))
        return (best if ok else "<lexicon-reject>"), {"used": True, "closest": best, "edit": dist, "ned": ned, "accepted": bool(ok), "threshold": "NED<=0.30"}
    ok = dist <= int(getattr(cfg, "OFFICIAL_LEXICON_MAX_ABS_EDIT", 2))
    return (best if ok else "<lexicon-reject>"), {"used": True, "closest": best, "edit": dist, "accepted": bool(ok), "threshold": "ED<=2"}
def _apply_lexicon_to_predictions(pred_batches: List[List[Dict[str, Any]]], lexicons: Any, dataset_key: str) -> List[List[Dict[str, Any]]]:
    out: List[List[Dict[str, Any]]] = []
    per_image = bool(lexicons and isinstance(lexicons, list) and (not lexicons or isinstance(lexicons[0], list)))
    for bi, preds in enumerate(pred_batches):
        lex = lexicons[bi] if per_image and bi < len(lexicons) else lexicons
        batch = []
        for p in preds:
            q = dict(p)
            corrected, info = _lexicon_correct_text(str(q.get("text", "")), lex, "lexicon", dataset_key)
            q["raw_text_before_lexicon"] = q.get("text", "")
            q["text"] = corrected
            q["lexicon_correction"] = info
            batch.append(q)
        out.append(batch)
    return out
def make_official_reporting_bundle(pred_batches: List[List[Dict[str, Any]]], targets: List[Dict[str, Any]], dataset_key: str, out_dir: Optional[Path] = None, prefix: str = "official") -> Dict[str, Any]:
    """Create dataset-specific official-style reporting.
    Implemented exactly inside the unified evaluator path:
    - all datasets: Det P/R/H and E2E None.
    - Total-Text: E2E Full using all evaluation GT text instances, ED<=2.
    - CTW1500: E2E Full using all evaluation GT phrase instances, NED<=0.3.
    - ICDAR2015: E2E None plus official hooks. If official S/W/G lexicon files are
      absent, safe diagnostic proxies are exported separately and clearly labeled.
    """
    dataset_key = str(dataset_key)
    report: Dict[str, Any] = {
        "dataset": dataset_key,
        "metric": "polygon/quad IoU>=0.5 + normalized transcription match; Hmean/F1 in percent",
        "detection": evaluate_polygon_batches(pred_batches, targets, None),
        "e2e_none": evaluate_polygon_batches(pred_batches, targets, None),
        "settings": {},
    }
    # Keep only compact copy of none metrics under a standard table-friendly section.
    report["none"] = {k: report["e2e_none"].get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]}
    full_lex = _target_lexicon_texts(targets, per_image=False)
    per_image_lex = _target_lexicon_texts(targets, per_image=True)
    report["lexicon_stats"] = {"full_size": len(full_lex), "per_image_avg_size": float(np.mean([len(x) for x in per_image_lex])) if per_image_lex else 0.0}
    if dataset_key in {"total_text", "ctw1500"}:
        lex_preds = _apply_lexicon_to_predictions(pred_batches, full_lex, dataset_key)
        full_ev = evaluate_polygon_batches(lex_preds, targets, None)
        report["full"] = {k: full_ev.get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]}
        report["full"]["reported_as"] = "E2E Full"
        report["settings"]["full"] = "Full lexicon built from all evaluation text instances; CTW uses NED<=0.3, Total-Text uses ED<=2."
    elif dataset_key == "icdar2015":
        # Diagnostic approximations are deliberately not named as official S/W/G unless real lexicon files are wired in.
        weak_preds = _apply_lexicon_to_predictions(pred_batches, full_lex, dataset_key)
        strong_proxy_preds = _apply_lexicon_to_predictions(pred_batches, per_image_lex, dataset_key)
        weak_ev = evaluate_polygon_batches(weak_preds, targets, None)
        strong_proxy_ev = evaluate_polygon_batches(strong_proxy_preds, targets, None)
        report["weak_proxy"] = {k: weak_ev.get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]}
        report["strong_proxy"] = {k: strong_proxy_ev.get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]}
        report["official_strong"] = None
        report["official_weak"] = None
        report["official_generic"] = None
        report["settings"]["icdar"] = "Official ICDAR S/W/G requires release lexicon files. This run exports None plus clearly labeled strong/weak diagnostic proxies from GT lexicons."
    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)
        save_json(report, out_dir / f"{prefix}_{dataset_key}_official_reporting_bundle.json")
        rows = []
        none = report.get("none", {})
        rows.append({"dataset": dataset_key, "setting": "None", "det_f": none.get("det_f"), "e2e_hmean": none.get("e2e_none_f"), "e2e_p": none.get("e2e_none_precision"), "e2e_r": none.get("e2e_none_recall"), "note": "lexicon-free"})
        if dataset_key in {"total_text", "ctw1500"} and "full" in report:
            full = report["full"]
            rows.append({"dataset": dataset_key, "setting": "Full", "det_f": full.get("det_f"), "e2e_hmean": full.get("e2e_none_f"), "e2e_p": full.get("e2e_none_precision"), "e2e_r": full.get("e2e_none_recall"), "note": full.get("reported_as", "E2E Full")})
        if dataset_key == "icdar2015":
            for setting in ["strong_proxy", "weak_proxy"]:
                ev = report.get(setting, {})
                rows.append({"dataset": dataset_key, "setting": setting, "det_f": ev.get("det_f"), "e2e_hmean": ev.get("e2e_none_f"), "e2e_p": ev.get("e2e_none_precision"), "e2e_r": ev.get("e2e_none_recall"), "note": "diagnostic proxy; official lexicon files not bundled"})
        write_csv(rows, out_dir / f"{prefix}_{dataset_key}_official_reporting_table.csv")
    return report

# =============================================================================
# V7.10 TRUE STREAMING OFFICIAL EVALUATION
# =============================================================================
def _compact_prf(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
    p = 100.0 * tp / max(1, tp + fp)
    r = 100.0 * tp / max(1, tp + fn)
    f = 2 * p * r / max(1e-9, p + r)
    return float(p), float(r), float(f)


def _records_full_lexicon_from_loader(loader: DataLoader) -> List[str]:
    words = []
    ds = getattr(loader, "dataset", None)
    records = getattr(ds, "records", []) if ds is not None else []
    for r in records:
        for obj in r.get("instances", []):
            if obj.get("ignore", False) or obj.get("text_ignore", False):
                continue
            txt = normalize_for_eval(obj.get("text", ""), ascii_lower=True)
            if txt:
                words.append(txt)
    return sorted(set(words))


def _batch_per_image_lexicons_from_targets(targets: List[Dict[str, Any]]) -> List[List[str]]:
    out = []
    for tgt in targets:
        gt_texts = [normalize_for_eval(t, ascii_lower=True) for t in tgt.get("texts", [])]
        ignore = tgt.get("ignore", torch.zeros((len(gt_texts),), dtype=torch.bool))
        ignore_np = ignore.detach().cpu().numpy().astype(bool) if isinstance(ignore, torch.Tensor) else np.asarray(ignore, dtype=bool)
        text_ignore_t = tgt.get("text_ignore", torch.zeros((len(gt_texts),), dtype=torch.bool))
        text_ignore_np = text_ignore_t.detach().cpu().numpy().astype(bool) if isinstance(text_ignore_t, torch.Tensor) else np.asarray(text_ignore_t, dtype=bool)
        out.append(sorted({gt_texts[i] for i in range(len(gt_texts)) if i < len(ignore_np) and not ignore_np[i] and (i >= len(text_ignore_np) or not text_ignore_np[i]) and gt_texts[i]}))
    return out


def _merge_eval_counts(acc: Dict[str, Any], ev: Dict[str, Any], n_img: int, pred_batches: List[List[Dict[str, Any]]]) -> None:
    c = ev.get("counts", {})
    for k in ["det_tp", "det_fp", "det_fn", "e2e_none_tp", "e2e_none_fp", "e2e_none_fn"]:
        acc[k] = int(acc.get(k, 0)) + int(c.get(k, 0))
    acc["images"] = int(acc.get("images", 0)) + int(n_img)
    acc["preds"] = int(acc.get("preds", 0)) + int(sum(len(x) for x in pred_batches))


def _finalize_eval_acc(acc: Dict[str, Any]) -> Dict[str, Any]:
    det_p, det_r, det_f = _compact_prf(int(acc.get("det_tp", 0)), int(acc.get("det_fp", 0)), int(acc.get("det_fn", 0)))
    e2e_p, e2e_r, e2e_f = _compact_prf(int(acc.get("e2e_none_tp", 0)), int(acc.get("e2e_none_fp", 0)), int(acc.get("e2e_none_fn", 0)))
    return {
        "det_precision": det_p,
        "det_recall": det_r,
        "det_f": det_f,
        "e2e_none_precision": e2e_p,
        "e2e_none_recall": e2e_r,
        "e2e_none_f": e2e_f,
        "avg_preds_per_image": float(acc.get("preds", 0)) / max(1, int(acc.get("images", 0))),
        "counts": {
            "det_tp": int(acc.get("det_tp", 0)),
            "det_fp": int(acc.get("det_fp", 0)),
            "det_fn": int(acc.get("det_fn", 0)),
            "e2e_none_tp": int(acc.get("e2e_none_tp", 0)),
            "e2e_none_fp": int(acc.get("e2e_none_fp", 0)),
            "e2e_none_fn": int(acc.get("e2e_none_fn", 0)),
        },
        "metric_scale_note": "Percent metrics in [0,100]. True streaming aggregate; examples omitted for RAM safety.",
    }


def _write_stream_official_reporting(report: Dict[str, Any], dataset_key: str, out_dir: Path, prefix: str) -> None:
    off_dir = out_dir / "official_reporting"
    off_dir.mkdir(parents=True, exist_ok=True)
    save_json(report, off_dir / f"{prefix}_{dataset_key}_official_reporting_bundle.json")
    rows = []
    none = report.get("none", {})
    rows.append({"dataset": dataset_key, "setting": "None", "det_f": none.get("det_f"), "e2e_hmean": none.get("e2e_none_f"), "e2e_p": none.get("e2e_none_precision"), "e2e_r": none.get("e2e_none_recall"), "note": "lexicon-free"})
    if dataset_key in {"total_text", "ctw1500"}:
        full = report.get("full", {})
        rows.append({"dataset": dataset_key, "setting": "Full", "det_f": full.get("det_f"), "e2e_hmean": full.get("e2e_none_f"), "e2e_p": full.get("e2e_none_precision"), "e2e_r": full.get("e2e_none_recall"), "note": "E2E Full"})
    elif dataset_key == "icdar2015":
        # Official S/W/G are present as fields, but remain null unless official lexicon files are wired.
        rows.append({"dataset": dataset_key, "setting": "official_strong", "det_f": None, "e2e_hmean": None, "e2e_p": None, "e2e_r": None, "note": "requires official ICDAR2015 strong lexicon files"})
        rows.append({"dataset": dataset_key, "setting": "official_weak", "det_f": None, "e2e_hmean": None, "e2e_p": None, "e2e_r": None, "note": "requires official ICDAR2015 weak lexicon files"})
        rows.append({"dataset": dataset_key, "setting": "official_generic", "det_f": None, "e2e_hmean": None, "e2e_p": None, "e2e_r": None, "note": "requires official ICDAR2015 generic lexicon files"})
        for setting in ["strong_proxy", "weak_proxy", "generic_proxy"]:
            ev = report.get(setting, {})
            rows.append({"dataset": dataset_key, "setting": setting, "det_f": ev.get("det_f"), "e2e_hmean": ev.get("e2e_none_f"), "e2e_p": ev.get("e2e_none_precision"), "e2e_r": ev.get("e2e_none_recall"), "note": "diagnostic proxy; not official"})
    write_csv(rows, off_dir / f"{prefix}_{dataset_key}_official_reporting_table.csv")

def collect_db_predictions(model: nn.Module, loader: DataLoader) -> Tuple[List[np.ndarray], List[Dict[str, Any]], List[torch.Tensor]]:
    """Collect DB validation predictions with V7 T4-safe memory handling.
    Keeps the exact validation logic intact; only changes runtime hygiene:
    inference_mode, boundary cache cleanup, and explicit tensor deletion.
    """
    model.eval()
    probs: List[np.ndarray] = []
    tgts: List[Dict[str, Any]] = []
    imgs: List[torch.Tensor] = []
    clean_memory()
    memory_safety_barrier(f"{current_dataset_tag()}-DB-val-start")
    log(gpu_mem(f"{current_dataset_tag()}-DB-val-start"))
    try:
        with torch.inference_mode():
            for vi, (images, targets) in enumerate(loader, start=1):
                batch_probs = _forward_db_probs_for_batch(model, images)
                probs.extend(batch_probs)
                tgts.extend(targets)
                imgs.extend([x.detach().cpu() for x in images])
                del batch_probs, images
                if vi == 1 or vi == len(loader) or vi % max(1, int(getattr(cfg, "DB_VAL_CLEAN_EVERY", 10))) == 0:
                    memory_safety_barrier(f"{current_dataset_tag()}-DB-val-b{vi}")
                    log(gpu_mem(f"{current_dataset_tag()}-DB-val batch {vi}/{len(loader)}"))
    except RuntimeError as e:
        log(f"[{current_dataset_tag()}-DB-val][RuntimeError] {repr(e)}")
        log(gpu_mem(f"{current_dataset_tag()}-DB-val-error"))
        raise
    finally:
        model.train()
        clean_memory()
    log(gpu_mem(f"{current_dataset_tag()}-DB-val-done"))
    return probs, tgts, imgs
def _settings_cache_key(settings: Dict[str, Any]) -> Tuple[Any, ...]:
    return (
        float(settings.get("threshold", 0.0)), float(settings.get("unshrink", 0.0)),
        str(settings.get("mode", "")), int(settings.get("min_area", 0)),
        int(settings.get("max_preds", cfg.TT_DB_MAX_PREDS)), str(settings.get("morph", "none")),
        float(settings.get("crop_pad", 1.0)),
    )
def _postprocess_prob_batches_cached(probs: List[np.ndarray], settings: Dict[str, Any], cache: Optional[Dict[Tuple[Any, ...], List[List[Dict[str, Any]]]]] = None) -> List[List[Dict[str, Any]]]:
    key = _settings_cache_key(settings)
    if bool(getattr(cfg, "ENABLE_VALIDATION_CACHES", True)) and cache is not None and key in cache:
        return cache[key]
    preds = [db_postprocess_prob(p, settings) for p in probs]
    if bool(getattr(cfg, "ENABLE_VALIDATION_CACHES", True)) and cache is not None:
        cache[key] = preds
    return preds
def sweep_db_postprocess(probs: List[np.ndarray], targets: List[Dict[str, Any]], out_dir: Path, prefix: str) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    """Full postprocess sweep, optimized for T4/low-RAM Colab.
    V7.1 keeps the exact sweep grid and scoring logic, but fixes the CTW crash
    observed immediately after `[CTW1500-DB-val-done]`. The crash was caused by
    materializing too many CTW postprocess results in an unbounded cache during
    the full curved-line sweep. Detection metrics do not depend on `crop_pad`,
    so this function evaluates each geometry setting once and then emits the
    same row for every crop-pad candidate. This preserves the logical grid and
    all reported rows while avoiding redundant contour extraction and preventing
    RAM blow-up. E2E top-K selection still evaluates crop_pad normally later.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    rows: List[Dict[str, Any]] = []
    best_settings: Dict[str, Any] = {}
    best_score = -1e9
    thresholds = tuple(getattr(cfg, "TT_DB_GEOM_SWEEP_THRESHOLDS", cfg.TT_DB_SWEEP_THRESHOLDS))
    unshrinks = tuple(getattr(cfg, "TT_DB_GEOM_SWEEP_UNSHRINK", cfg.TT_DB_SWEEP_UNSHRINK))
    modes = tuple(getattr(cfg, "TT_DB_GEOM_SWEEP_MODES", cfg.TT_DB_SWEEP_MODES))
    min_areas = tuple(getattr(cfg, "TT_DB_GEOM_SWEEP_MIN_AREAS", cfg.TT_DB_SWEEP_MIN_AREAS))
    morphs = tuple(getattr(cfg, "TT_DB_GEOM_SWEEP_MORPH", ("none",)))
    crop_pads = tuple(getattr(cfg, "TT_DB_GEOM_SWEEP_CROP_PAD", (cfg.TT_CROP_RECTIFY_PAD,)))
    total_geom = max(1, len(thresholds) * len(unshrinks) * len(modes) * len(min_areas) * len(morphs))
    total_rows = total_geom * max(1, len(crop_pads))
    geom_i = 0
    row_i = 0
    log(f"[{current_dataset_tag()}-Sweep][{prefix}] start geometry={total_geom} rows={total_rows} images={len(probs)} pads={list(crop_pads)} cache=grouped_by_geometry")
    for th in thresholds:
        for un in unshrinks:
            for mode in modes:
                for ma in min_areas:
                    for morph in morphs:
                        geom_i += 1
                        # crop_pad does not affect DB contour extraction or Det-F; it only
                        # affects OCR crop expansion later. We attach each pad value to the
                        # emitted rows below.
                        geom_settings = {
                            "threshold": float(th),
                            "unshrink": float(un),
                            "mode": mode,
                            "min_area": int(ma),
                            "max_preds": int(cfg.TT_DB_MAX_PREDS),
                            "morph": morph,
                            "crop_pad": float(crop_pads[0]) if crop_pads else float(cfg.TT_CROP_RECTIFY_PAD),
                        }
                        try:
                            preds = [db_postprocess_prob(p, geom_settings) for p in probs]
                            ev = evaluate_polygon_batches(preds, targets)
                        except Exception as e:
                            log(f"[{current_dataset_tag()}-Sweep][{prefix}] failed at geom {geom_i}/{total_geom} settings={geom_settings} error={repr(e)}")
                            raise
                        base_metrics = {
                            "det_f": ev["det_f"],
                            "det_precision": ev["det_precision"],
                            "det_recall": ev["det_recall"],
                            "avg_preds_per_image": ev["avg_preds_per_image"],
                            "avg_max_score": ev["avg_max_score"],
                        }
                        score = float(ev["det_f"]) + 0.04 * float(ev["det_recall"]) - 0.02 * abs(float(ev["avg_preds_per_image"]) - 8.0)
                        for pad in crop_pads:
                            row_i += 1
                            settings = dict(geom_settings)
                            settings["crop_pad"] = float(pad)
                            row = {**settings, **base_metrics, "selection_score": score}
                            rows.append(row)
                            if score > best_score:
                                best_score = score
                                best_settings = dict(settings)
                                best_settings.update({
                                    "val_det_f": ev["det_f"],
                                    "val_det_precision": ev["det_precision"],
                                    "val_det_recall": ev["det_recall"],
                                    "val_avg_preds_per_image": ev["avg_preds_per_image"],
                                    "selection_score": score,
                                })
                        if geom_i == 1 or geom_i == total_geom or geom_i % max(1, int(getattr(cfg, "SWEEP_PROGRESS_EVERY", 50))) == 0:
                            log(f"[{current_dataset_tag()}-Sweep][{prefix}] geom {geom_i}/{total_geom} rows {row_i}/{total_rows} bestDetF={float(best_settings.get('val_det_f', 0.0)):.3f}")
                            log(gpu_mem(f"{current_dataset_tag()}-Sweep"))
                        del preds, ev
                        if current_dataset_key() == "ctw1500" and geom_i % 25 == 0:
                            gc.collect()
    write_csv(rows, out_dir / f"{prefix}_postprocess_sweep.csv")
    save_json(best_settings, out_dir / f"{prefix}_best_postprocess.json")
    save_json({
        "sweep_rows": len(rows),
        "geometry_settings": total_geom,
        "crop_pad_rows_per_geometry": len(crop_pads),
        "cache_strategy": "geometry_grouped_no_unbounded_pred_cache",
        "note": "Detection sweep logic/grid preserved; crop_pad rows are emitted for later E2E top-k, while contour extraction is shared because crop_pad does not affect DB postprocess."
    }, out_dir / f"{prefix}_sweep_cache_telemetry.json")
    log(f"[{current_dataset_tag()}-Sweep][{prefix}] done rows={len(rows)} best={best_settings}")
    return best_settings, rows
def sweep_geometry_alignment_e2e(
    probs: List[np.ndarray],
    targets: List[Dict[str, Any]],
    imgs: List[torch.Tensor],
    rec_model: nn.Module,
    tokenizer: TextTokenizer,
    out_dir: Path,
    prefix: str,
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    """Run OCR/E2E only on the top-K detection settings from the geometry sweep.
    This directly addresses the GT-vs-pred gap without tuning on official test.
    Full grid detection is cheap; OCR on every setting is expensive, so only the
    top-K train-internal validation settings are evaluated with E2E.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    det_best, det_rows = sweep_db_postprocess(probs, targets, out_dir, prefix=f"{prefix}_det")
    topk = max(1, int(getattr(cfg, "TT_DB_GEOM_TOPK_E2E", 8)))
    det_rows_sorted = sorted(det_rows, key=lambda r: float(r.get("selection_score", r.get("det_f", 0.0))), reverse=True)[:topk]
    e2e_rows: List[Dict[str, Any]] = []
    best_settings = dict(det_best)
    best_score = -1e9
    pred_cache: Dict[Tuple[Any, ...], List[List[Dict[str, Any]]]] = {}
    beta = float(getattr(cfg, "TT_DB_GEOM_E2E_BETA", 0.35))
    for i, row in enumerate(det_rows_sorted, start=1):
        settings = {k: row[k] for k in ["threshold", "unshrink", "mode", "min_area", "max_preds", "morph", "crop_pad"] if k in row}
        pred_batches = _postprocess_prob_batches_cached(probs, settings, pred_cache)
        pred_rec_batches = recognize_crops_from_pred_polys(rec_model, imgs, pred_batches, tokenizer)
        ev = evaluate_polygon_batches(pred_rec_batches, targets)
        dataset_key = current_dataset_key()
        det_h = float(ev.get("det_f", 0.0))
        e2e_h = float(ev.get("e2e_none_f", 0.0))
        rec_h = float(ev.get("det_recall", 0.0))
        if dataset_key == "ctw1500":
            score = 0.55 * det_h + 0.30 * e2e_h + 0.15 * rec_h
            score_formula = "0.55*DetH + 0.30*E2E + 0.15*Recall"
        else:
            score = 0.70 * det_h + 0.20 * e2e_h + 0.10 * rec_h
            score_formula = "0.70*DetH + 0.20*E2E + 0.10*Recall"
        out_row = {**settings, "rank": i, "det_f": ev.get("det_f", 0.0), "det_precision": ev.get("det_precision", 0.0), "det_recall": ev.get("det_recall", 0.0), "e2e_none_f": ev.get("e2e_none_f", 0.0), "e2e_none_precision": ev.get("e2e_none_precision", 0.0), "e2e_none_recall": ev.get("e2e_none_recall", 0.0), "avg_preds_per_image": ev.get("avg_preds_per_image", 0.0), "alignment_selection_score": score, "selection_formula": score_formula}
        e2e_rows.append(out_row)
        if score > best_score:
            best_score = score
            best_settings = dict(settings)
            best_settings.update({
                "val_det_f": ev.get("det_f", 0.0),
                "val_det_precision": ev.get("det_precision", 0.0),
                "val_det_recall": ev.get("det_recall", 0.0),
                "val_e2e_none_f": ev.get("e2e_none_f", 0.0),
                "val_e2e_none_precision": ev.get("e2e_none_precision", 0.0),
                "val_e2e_none_recall": ev.get("e2e_none_recall", 0.0),
                "val_avg_preds_per_image": ev.get("avg_preds_per_image", 0.0),
                "alignment_selection_score": score,
                "selection_note": "train-internal validation top-K E2E alignment sweep; official test untouched",
                "selection_formula": score_formula,
            })
    write_csv(e2e_rows, out_dir / f"{prefix}_alignment_e2e_topk.csv")
    save_json(best_settings, out_dir / f"{prefix}_best_alignment_postprocess.json")
    log(f"[GeomAlign][{prefix}] best DetF={float(best_settings.get('val_det_f',0.0)):.3f} E2E={float(best_settings.get('val_e2e_none_f',0.0)):.3f} settings={best_settings}")
    return best_settings, e2e_rows
def _forward_db_probs_for_batch(model: nn.Module, images: torch.Tensor) -> List[np.ndarray]:
    try:
        with torch.inference_mode():
            images_gpu = images.to(cfg.DEVICE, non_blocking=True)
            images_gpu = optimize_image_tensor_for_cuda(images_gpu)
            with autocast(device_type="cuda", dtype=get_amp_dtype(), enabled=(cfg.DEVICE == "cuda")):
                logits = model(images_gpu)["db_logits"]
            p_np = torch.sigmoid(logits.float()).detach().cpu().numpy()[:, 0]
            probs = [x.astype(np.float32) for x in p_np]
            del images_gpu, logits, p_np
        return probs
    except RuntimeError as e:
        if is_cuda_oom(e) and isinstance(images, torch.Tensor) and images.shape[0] > 1:
            recover_from_oom(f"{current_dataset_tag()}-db-forward-split-b{images.shape[0]}")
            mid = images.shape[0] // 2
            return _forward_db_probs_for_batch(model, images[:mid].contiguous()) + _forward_db_probs_for_batch(model, images[mid:].contiguous())
        raise


def eval_db_with_recognizer(model: nn.Module, rec_model: Optional[nn.Module], loader: DataLoader, tokenizer: TextTokenizer, settings: Dict[str, Any], out_json: Path, with_ocr: bool = True) -> Dict[str, Any]:
    """True streaming official/full evaluation.

    V7.9 still retained all prediction dictionaries and targets until the end.
    V7.10 aggregates metric counts per batch and keeps only compact counters, so
    CTW500 final evaluation cannot exhaust 12GB Colab RAM.
    """
    model.eval()
    if rec_model is not None:
        rec_model.eval()

    dataset_key = current_dataset_key()
    full_lex = _records_full_lexicon_from_loader(loader)
    none_acc: Dict[str, Any] = {}
    full_acc: Dict[str, Any] = {}
    weak_acc: Dict[str, Any] = {}
    strong_acc: Dict[str, Any] = {}
    generic_acc: Dict[str, Any] = {}
    fused_gap_enabled = bool(with_ocr and rec_model is not None and getattr(cfg, "ENABLE_GEOMETRY_GAP_EVAL", False) and getattr(cfg, "FUSE_GAP_WITH_OFFICIAL_EVAL", True))
    gt_gap_acc: Dict[str, Any] = {}
    gap_progress_rows: List[Dict[str, Any]] = []

    memory_safety_barrier(f"{current_dataset_tag()}-stream-eval-start")
    log(f"[{current_dataset_tag()}-StreamEval] start batches={len(loader)} with_ocr={with_ocr} true_stream=True")

    with torch.inference_mode():
        for bi, (images, targets) in enumerate(loader, start=1):
            images_cpu = [x.detach().cpu() for x in images]
            probs = _forward_db_probs_for_batch(model, images)
            pred_batches = [db_postprocess_prob(p, settings) for p in probs]
            if bool(getattr(cfg, "ARCFLOW_REFINE_PRED_POLYS_AT_EVAL", True)) and hasattr(unwrap_model(model), "arcflow_head"):
                pred_batches = refine_pred_batches_with_arcflow(model, images_cpu, pred_batches)
            if with_ocr and rec_model is not None:
                pred_batches = recognize_crops_from_pred_polys(rec_model, images_cpu, pred_batches, tokenizer)
            else:
                for batch in pred_batches:
                    for p in batch:
                        p["text"] = ""

            ev_none_b = evaluate_polygon_batches(pred_batches, targets, None)
            _merge_eval_counts(none_acc, ev_none_b, len(targets), pred_batches)

            if with_ocr and rec_model is not None:
                if dataset_key in {"total_text", "ctw1500"}:
                    lex_pred = _apply_lexicon_to_predictions(pred_batches, full_lex, dataset_key)
                    ev_full_b = evaluate_polygon_batches(lex_pred, targets, None)
                    _merge_eval_counts(full_acc, ev_full_b, len(targets), lex_pred)
                    del lex_pred
                elif dataset_key == "icdar2015":
                    # Proxies only, not official S/W/G.
                    weak_pred = _apply_lexicon_to_predictions(pred_batches, full_lex, dataset_key)
                    ev_weak_b = evaluate_polygon_batches(weak_pred, targets, None)
                    _merge_eval_counts(weak_acc, ev_weak_b, len(targets), weak_pred)
                    generic_pred = weak_pred
                    ev_generic_b = ev_weak_b
                    _merge_eval_counts(generic_acc, ev_generic_b, len(targets), generic_pred)
                    per_img_lex = _batch_per_image_lexicons_from_targets(targets)
                    strong_pred = _apply_lexicon_to_predictions(pred_batches, per_img_lex, dataset_key)
                    ev_strong_b = evaluate_polygon_batches(strong_pred, targets, None)
                    _merge_eval_counts(strong_acc, ev_strong_b, len(targets), strong_pred)
                    del weak_pred, strong_pred, per_img_lex

            if fused_gap_enabled:
                try:
                    gt_batches: List[List[Dict[str, Any]]] = []
                    for tgt in targets:
                        polys = tgt["polys"].detach().cpu().numpy()
                        ignore = tgt["ignore"].detach().cpu().numpy().astype(bool)
                        batch_gt = []
                        for p, ign in zip(polys, ignore):
                            if ign:
                                continue
                            batch_gt.append({"poly": p.astype(np.float32), "box": poly_to_bbox(p), "score": 1.0, "text": ""})
                        gt_batches.append(batch_gt)
                    gt_rec_batches = recognize_crops_from_pred_polys(rec_model, images_cpu, gt_batches, tokenizer)
                    gt_ev_b = evaluate_polygon_batches(gt_rec_batches, targets, None)
                    _merge_eval_counts(gt_gap_acc, gt_ev_b, len(targets), gt_rec_batches)
                    if bi == 1 or bi == len(loader) or bi % max(1, int(getattr(cfg, "STREAM_EVAL_LOG_EVERY", 50))) == 0:
                        pred_now_gap = _finalize_eval_acc(none_acc)
                        gt_now_gap = _finalize_eval_acc(gt_gap_acc)
                        gap_progress_rows.append({
                            "batch": bi,
                            "images": int(none_acc.get("images", 0)),
                            "gt_geometry_e2e_none_f": gt_now_gap.get("e2e_none_f", 0.0),
                            "predicted_geometry_e2e_none_f": pred_now_gap.get("e2e_none_f", 0.0),
                            "gap": float(gt_now_gap.get("e2e_none_f", 0.0)) - float(pred_now_gap.get("e2e_none_f", 0.0)),
                            "pred_det_f": pred_now_gap.get("det_f", 0.0),
                        })
                    del gt_batches, gt_rec_batches
                except Exception as e:
                    if bool(getattr(cfg, "STRICT_GAP_EVAL", False)):
                        raise
                    fused_gap_enabled = False
                    log(f"[{current_dataset_tag()}-Gap][warn] fused gap failed; official eval continues: {repr(e)}")

            if bi == 1 or bi == len(loader) or bi % max(1, int(getattr(cfg, "STREAM_EVAL_LOG_EVERY", 50))) == 0:
                so_far = _finalize_eval_acc(none_acc)
                gap_msg = ""
                if fused_gap_enabled and gt_gap_acc:
                    gt_now = _finalize_eval_acc(gt_gap_acc)
                    gap_msg = f" GT_E2E={float(gt_now.get('e2e_none_f',0.0)):.3f} Gap={float(gt_now.get('e2e_none_f',0.0))-float(so_far.get('e2e_none_f',0.0)):.3f}"
                log(f"[{current_dataset_tag()}-StreamEval] batch {bi}/{len(loader)} images={none_acc.get('images',0)} preds={none_acc.get('preds',0)} DetF={so_far.get('det_f',0.0):.3f}{gap_msg}")
                log(gpu_mem(f"{current_dataset_tag()}-StreamEval"))

            del images_cpu, probs, pred_batches, images, targets
            if bi % 25 == 0:
                memory_safety_barrier(f"{current_dataset_tag()}-stream-eval-b{bi}")

    ev = _finalize_eval_acc(none_acc)
    ev["postprocess"] = settings
    ev["arcflow_refine_pred_polys_at_eval"] = bool(getattr(cfg, "ARCFLOW_REFINE_PRED_POLYS_AT_EVAL", True))
    ev["streaming_eval"] = True
    ev["true_low_ram_streaming"] = True
    ev["small_medium_large_recall"] = {"skipped": True, "reason": "disabled in V7.10 true streaming mode to prevent RAM growth"}

    if with_ocr and rec_model is not None:
        report: Dict[str, Any] = {
            "dataset": dataset_key,
            "metric": "polygon/quad IoU>=0.5 + normalized transcription match; Hmean/F1 in percent",
            "detection": dict(ev),
            "e2e_none": dict(ev),
            "none": {k: ev.get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]},
            "settings": {},
            "lexicon_stats": {"full_size": len(full_lex), "per_image_avg_size": None, "source": "loader.dataset.records"},
            "streaming_official_reporting": True,
        }
        if dataset_key in {"total_text", "ctw1500"}:
            full_ev = _finalize_eval_acc(full_acc)
            report["full"] = {k: full_ev.get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]}
            report["full"]["reported_as"] = "E2E Full"
            report["settings"]["full"] = "Full lexicon built from all evaluation text instances; CTW uses NED<=0.3, Total-Text uses ED<=2."
        elif dataset_key == "icdar2015":
            strong_ev = _finalize_eval_acc(strong_acc)
            weak_ev = _finalize_eval_acc(weak_acc)
            generic_ev = _finalize_eval_acc(generic_acc)
            report["strong_proxy"] = {k: strong_ev.get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]}
            report["weak_proxy"] = {k: weak_ev.get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]}
            report["generic_proxy"] = {k: generic_ev.get(k) for k in ["det_precision", "det_recall", "det_f", "e2e_none_precision", "e2e_none_recall", "e2e_none_f", "avg_preds_per_image", "counts"]}
            report["official_strong"] = None
            report["official_weak"] = None
            report["official_generic"] = None
            report["settings"]["icdar"] = "Official ICDAR S/W/G require release lexicon files. V7.10 exports None plus clearly labeled proxies; official fields remain null unless lexicons are provided."
        official_dir = out_json.parent
        _write_stream_official_reporting(report, dataset_key, official_dir, prefix=out_json.stem)
        ev["official_reporting"] = report

    if fused_gap_enabled:
        gt_ev = _finalize_eval_acc(gt_gap_acc)
        pred_ev = _finalize_eval_acc(none_acc)
        gap_report = {
            "gt_geometry_e2e_none_f": gt_ev.get("e2e_none_f", 0.0),
            "predicted_geometry_e2e_none_f": pred_ev.get("e2e_none_f", 0.0),
            "gt_geometry_det_f": gt_ev.get("det_f", 0.0),
            "predicted_geometry_det_f": pred_ev.get("det_f", 0.0),
            "geometry_to_recognition_gap": float(gt_ev.get("e2e_none_f", 0.0)) - float(pred_ev.get("e2e_none_f", 0.0)),
            "streaming_gap_eval": True,
            "fused_with_official_eval": True,
            "true_low_ram_streaming": False,
            "counts": {"gt": gt_ev.get("counts", {}), "predicted": pred_ev.get("counts", {})},
            "progress_rows": gap_progress_rows,
            "interpretation": "High GT-geometry E2E with low predicted E2E means predicted geometry/alignment is the bottleneck.",
        }
        ev["gt_vs_pred_geometry_gap"] = gap_report
        try:
            save_json(gap_report, out_json.parent / "gt_vs_pred_geometry_gap_report.json")
            write_csv(gap_progress_rows, out_json.parent / "gt_vs_pred_geometry_gap_progress.csv")
        except Exception:
            pass

    save_json(ev, out_json)
    memory_safety_barrier(f"{current_dataset_tag()}-stream-eval-done")
    model.train()
    if rec_model is not None:
        rec_model.train()
    return ev

def evaluate_tt_crop_recognizer(model: nn.Module, loader: DataLoader, tokenizer: TextTokenizer, out_json: Path, epoch: int) -> Dict[str, Any]:
    old_mode = cfg.OCR_DECODE_MODE
    cfg.OCR_DECODE_MODE = str(cfg.TT_CROP_DECODE_MODE)
    try:
        ev = evaluate_str(model, loader, tokenizer, out_json, epoch=epoch)
    finally:
        cfg.OCR_DECODE_MODE = old_mode
    # Rename key aliases for Total-Text crop diagnostic.
    ev["WA"] = ev.get("macro_WRR", 0.0)
    ev["CRR"] = ev.get("macro_CRR", 0.0)
    ev["protocol_note"] = "Auxiliary Total-Text GT crop recognizer diagnostic only; not headline Total-Text metric."
    save_json(ev, out_json)
    return ev
def current_dataset_key() -> str:
    return str(getattr(cfg, "CURRENT_DATASET_KEY", "total_text"))
def current_dataset_pretty() -> str:
    return str(getattr(cfg, "CURRENT_DATASET_PRETTY", "Total-Text"))
def current_dataset_tag() -> str:
    return str(getattr(cfg, "CURRENT_DATASET_TAG", "TotalText"))
def current_dataset_run_dirname() -> str:
    return f"{current_dataset_key()}_full_image_run"
def is_text_usable_instance(obj: Dict[str, Any]) -> bool:
    if obj.get("ignore", False):
        return False
    if obj.get("text_ignore", False):
        return False
    return bool(normalize_for_eval(obj.get("text", ""), True))
def train_totaltext_crop_recognizer(tokenizer: TextTokenizer, records: Dict[str, Any], out_dir: Path) -> Tuple[nn.Module, Dict[str, Any]]:
    section(f"{current_dataset_pretty().upper()} LANE 1: GT CROP RECOGNIZER WARM-UP")
    crop_dir = out_dir / "tt_crop_recognizer"
    crop_dir.mkdir(parents=True, exist_ok=True)
    train_ds = TotalTextCropDataset(records["tt_train"], tokenizer, train=True)
    test_ds = TotalTextCropDataset(records["tt_test"], tokenizer, train=False)
    save_json({"train_crops": len(train_ds), "test_crops_expanded_modes": len(test_ds), "modes": list(cfg.TT_CROP_MODES)}, crop_dir / "crop_dataset_counts.json")
    train_loader = make_loader(train_ds, cfg.TT_CROP_BATCH_SIZE, shuffle=True, collate_fn=collate_totaltext_ocr, drop_last=True)
    test_loader = make_loader(test_ds, cfg.TT_CROP_EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_totaltext_ocr, drop_last=False)
    # Rectified-only diagnostic aligns with the full-image predicted polygon OCR path,
    # where detected polygons are rectified before recognition. The all-mode crop score
    # remains logged as robustness telemetry but no longer drives checkpoint selection.
    rect_test_loader = None
    if bool(getattr(cfg, "TT_CROP_EVAL_RECTIFIED_ONLY", True)):
        old_modes = cfg.TT_CROP_MODES
        cfg.TT_CROP_MODES = ("rectified",)
        rect_test_ds = TotalTextCropDataset(records["tt_test"], tokenizer, train=False)
        cfg.TT_CROP_MODES = old_modes
        rect_test_loader = make_loader(rect_test_ds, cfg.TT_CROP_EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_totaltext_ocr, drop_last=False)
    first = next(iter(train_loader))
    print_first_batch_debug(f"{current_dataset_tag()}-Crop", first)
    del first
    clean_memory()
    model = ArcFlowSTRModel(tokenizer.vocab_size, 1).to(cfg.DEVICE)
    if cfg.DEVICE == "cuda" and bool(getattr(cfg, "USE_CHANNELS_LAST", True)):
        model = model.to(memory_format=torch.channels_last)
    model = maybe_compile_model(model, f"{current_dataset_tag()}-Crop")
    total, trainable = count_params(model)
    log(f"[{current_dataset_tag()}-Crop] params total={total/1e6:.2f}M trainable={trainable/1e6:.2f}M")
    optimizer = torch.optim.AdamW(parameter_groups(model, cfg.LR_BACKBONE, cfg.LR_STR), weight_decay=cfg.WEIGHT_DECAY)
    scaler = GradScaler(enabled=(cfg.DEVICE == "cuda"))
    best_wa = -1.0
    metrics: List[Dict[str, Any]] = []
    log_every = max(int(getattr(cfg, 'MIN_LOG_STEPS', 10)), len(train_loader) // max(1, cfg.LOGS_PER_EPOCH))
    for epoch in range(1, int(cfg.EPOCHS_TT_CROP_WARMUP) + 1):
        model.train(); meter = defaultdict(float); n_meter = 0; t0 = time.time()
        accum_steps = max(1, int(getattr(cfg, "GRAD_ACCUM_STEPS", 1)))
        # With gradient accumulation, step 1 may log before an optimizer update.
        # Keep the learning logic unchanged; only make logging safe until the first clip/step.
        last_grad = float("nan")
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader, start=1):
            images = batch["images"].to(cfg.DEVICE, non_blocking=True)
            images = optimize_image_tensor_for_cuda(images)
            domain_ids = torch.zeros((images.shape[0],), dtype=torch.long, device=cfg.DEVICE)
            batch = dict(batch); batch["domain_ids"] = domain_ids.detach().cpu()
            with autocast(device_type="cuda", dtype=get_amp_dtype(), enabled=(cfg.DEVICE == "cuda")):
                outputs = model(images, domain_ids=domain_ids)
                loss, stats = str_loss(outputs, batch, epoch)
                loss_for_backward = loss / float(accum_steps)
            scaler.scale(loss_for_backward).backward()
            if step % accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                last_grad = float(clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM))
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
            for k, v in stats.items():
                meter[k] += float(v)
            n_meter += 1
            if step == 1 or step % log_every == 0 or step == len(train_loader):
                log(f"[{current_dataset_tag()}-Crop][ep {epoch}/{cfg.EPOCHS_TT_CROP_WARMUP} step {step}/{len(train_loader)}] loss={stats['loss']:.4f} ctc={stats['ctc']:.4f} ce={stats['ce']:.4f} domain={stats['domain']:.4f} sem={stats['sem']:.4f} grad={last_grad:.2f}")
                if cfg.PRINT_GPU_EVERY_LOG:
                    log(gpu_mem(f"{current_dataset_tag()}-Crop"))
                if float(_cpu_memory_snapshot().get("available_gb", 99.0) or 99.0) < float(getattr(cfg, "RAM_GUARD_MIN_AVAILABLE_GB", 3.0)):
                    memory_safety_barrier(f"{current_dataset_tag()}-Crop-step{step}-ram-guard", aggressive=True)
            try:
                del images, domain_ids, outputs, loss, loss_for_backward
            except Exception:
                pass
        memory_safety_barrier(f"{current_dataset_tag()}-Crop-before-eval", aggressive=True)
        ev = evaluate_tt_crop_recognizer(model, test_loader, tokenizer, crop_dir / f"crop_eval_epoch_{epoch:03d}.json", epoch=epoch)
        ev_rect = None
        rectified_eval_is_duplicate = (rect_test_loader is not None) and tuple(getattr(cfg, 'TT_CROP_MODES', ('rectified',))) == ('rectified',)
        if rectified_eval_is_duplicate:
            ev_rect = dict(ev)
        elif rect_test_loader is not None:
            ev_rect = evaluate_tt_crop_recognizer(model, rect_test_loader, tokenizer, crop_dir / f"crop_eval_rectified_epoch_{epoch:03d}.json", epoch=epoch)
        row = {
            "epoch": epoch,
            "train_loss": meter["loss"] / max(1, n_meter),
            "WA": ev.get("WA", 0.0),
            "CRR": ev.get("CRR", 0.0),
            "edit_le1": ev.get("edit_distance_le1_acc", 0.0),
            "empty_rate": ev.get("empty_pred_rate", 0.0),
            "rectified_WA": ev_rect.get("WA", None) if ev_rect is not None else None,
            "rectified_CRR": ev_rect.get("CRR", None) if ev_rect is not None else None,
            "rectified_edit_le1": ev_rect.get("edit_distance_le1_acc", None) if ev_rect is not None else None,
            "rectified_empty_rate": ev_rect.get("empty_pred_rate", None) if ev_rect is not None else None,
            "rectified_ctc_WA": ev_rect.get("decode_summary", {}).get("ctc_WRR", None) if ev_rect is not None else None,
            "rectified_ce_WA": ev_rect.get("decode_summary", {}).get("ce_WRR", None) if ev_rect is not None else None,
            "rectified_selected_src": json.dumps(ev_rect.get("selected_source_counts", {}), ensure_ascii=False) if ev_rect is not None else None,
            "all_ctc_WA": ev.get("decode_summary", {}).get("ctc_WRR", None),
            "all_ce_WA": ev.get("decode_summary", {}).get("ce_WRR", None),
            "selected_src": json.dumps(ev.get("selected_source_counts", {}), ensure_ascii=False),
            "minutes": (time.time() - t0) / 60.0,
        }
        metrics.append(row); write_csv(metrics, crop_dir / "crop_training_metrics.csv")
        rect_part = ""
        if row.get("rectified_WA") is not None:
            rect_part = (
                f" RECT_WA={float(row['rectified_WA']):.3f} RECT_E1={float(row['rectified_edit_le1']):.3f}"
                f" CTC={float(row['rectified_ctc_WA'] or 0.0):.3f} CE={float(row['rectified_ce_WA'] or 0.0):.3f} src={row.get('rectified_selected_src')}"
            )
        log(f"[{current_dataset_tag()}-Crop][epoch {epoch}] WA={row['WA']:.3f} CRR={row['CRR']:.3f} edit<=1={row['edit_le1']:.3f}{rect_part} empty={row['empty_rate']:.3f} time={row['minutes']:.1f}min")
        best_metric_name = str(getattr(cfg, "TT_CROP_BEST_METRIC", "rectified_WA"))
        score = row.get(best_metric_name, None)
        if score is None:
            score = row["WA"]
        if float(score) > best_wa:
            best_wa = float(score)
            save_ckpt(crop_dir / "best.pt", model, optimizer, scaler, epoch, best_wa, {"eval": ev, "eval_rectified": ev_rect, "best_metric": best_metric_name})
        save_ckpt(crop_dir / "last.pt", model, optimizer, scaler, epoch, best_wa, {"eval": ev})
        clean_memory()
    final = metrics[-1] if metrics else {"WA": 0.0, "edit_le1": 0.0}
    summary = {
        "stage": f"{current_dataset_pretty()} crop recognizer warm-up",
        "status": "healthy_candidate" if final.get("WA", 0.0) >= cfg.TT_CROP_MIN_WA_HEALTH else "crop_recognizer_below_target",
        "best_checkpoint_metric": str(getattr(cfg, "TT_CROP_BEST_METRIC", "rectified_WA")),
        "best_checkpoint_score": best_wa,
        "final": final,
        "note": "Auxiliary diagnostic only; not headline Total-Text result. Rectified-only score is used for checkpointing because full-image E2E recognizes rectified predicted polygons.",
    }
    # Quick configuration dashboard for the exact OCR path used by full-image E2E.
    best_row = max(metrics, key=lambda r: float(r.get(str(getattr(cfg, "TT_CROP_BEST_METRIC", "rectified_WA")), r.get("WA", 0.0)) or 0.0)) if metrics else {}
    summary["readiness_dashboard"] = {
        "best_rectified_WA": float(best_row.get("rectified_WA", 0.0) or 0.0),
        "best_rectified_edit_le1": float(best_row.get("rectified_edit_le1", 0.0) or 0.0),
        "target_rectified_WA": float(cfg.TT_CROP_TARGET_RECT_WA),
        "target_rectified_edit_le1": float(cfg.TT_CROP_TARGET_RECT_E1),
        "ctc_led_decode": str(cfg.TT_CROP_DECODE_MODE),
        "passes_fast_crop_gate": bool(
            float(best_row.get("rectified_WA", 0.0) or 0.0) >= float(cfg.TT_CROP_TARGET_RECT_WA)
            and float(best_row.get("rectified_edit_le1", 0.0) or 0.0) >= float(cfg.TT_CROP_TARGET_RECT_E1)
        ),
    }
    log(f"[{current_dataset_tag()}-Crop-Readiness] best_RECT_WA={summary['readiness_dashboard']['best_rectified_WA']:.3f} best_RECT_E1={summary['readiness_dashboard']['best_rectified_edit_le1']:.3f} pass={summary['readiness_dashboard']['passes_fast_crop_gate']}")
    save_json(summary, crop_dir / "crop_recognizer_summary.json")
    if bool(cfg.USE_BEST_TT_CROP_CHECKPOINT_FOR_DB):
        best_crop_info = load_model_weights_for_eval(crop_dir / "best.pt", model, f"best {current_dataset_pretty()} crop recognizer")
        summary["best_checkpoint_loaded_for_db_e2e"] = best_crop_info
        save_json(summary, crop_dir / "crop_recognizer_summary.json")
    if cfg.FAIL_ON_TT_CROP_WEAK and summary["status"] != "healthy_candidate":
        raise RuntimeError(f"{current_dataset_pretty()} crop recognizer weak: {summary}")
    return model, summary
def _safe_default_font():
    try:
        from PIL import ImageFont
        return ImageFont.load_default()
    except Exception:
        return None
def _resize_contain(im: Image.Image, target: Tuple[int, int], bg=(255, 255, 255)) -> Image.Image:
    tw, th = target
    src = im.convert("RGB")
    scale = min(tw / max(1, src.width), th / max(1, src.height))
    nw, nh = max(1, int(round(src.width * scale))), max(1, int(round(src.height * scale)))
    rs = src.resize((nw, nh), Image.BICUBIC)
    canvas = Image.new("RGB", (tw, th), bg)
    ox = (tw - nw) // 2
    oy = (th - nh) // 2
    canvas.paste(rs, (ox, oy))
    return canvas
def _add_panel_title(im: Image.Image, title: str, subtitle: str = "") -> Image.Image:
    font = _safe_default_font()
    out = Image.new("RGB", (im.width, im.height + 44), (255, 255, 255))
    out.paste(im, (0, 44))
    dr = ImageDraw.Draw(out)
    dr.rectangle((0, 0, out.width, 44), fill=(245, 247, 250))
    dr.text((10, 8), title, fill=(20, 20, 20), font=font)
    if subtitle:
        dr.text((10, 24), subtitle, fill=(90, 90, 90), font=font)
    return out
def _draw_poly_overlay(base: Image.Image, gt_polys: np.ndarray, ignore: np.ndarray, preds: Optional[List[Dict[str, Any]]] = None, gt_texts: Optional[List[str]] = None) -> Image.Image:
    im = base.convert("RGB").copy()
    dr = ImageDraw.Draw(im)
    W, H = im.size
    for i, p in enumerate(gt_polys):
        pts = [(float(x) * W, float(y) * H) for x, y in p]
        color = (40, 190, 80) if not bool(ignore[i]) else (170, 170, 170)
        dr.line(pts + [pts[0]], fill=color, width=3)
        if gt_texts is not None and i < len(gt_texts) and not bool(ignore[i]):
            txt = str(gt_texts[i])[:20]
            if txt:
                x0, y0 = pts[0]
                dr.rectangle((x0, max(0, y0 - 16), x0 + 8 * len(txt) + 6, y0), fill=(255, 255, 255))
                dr.text((x0 + 3, max(0, y0 - 14)), txt, fill=(0, 90, 0), font=_safe_default_font())
    if preds is not None:
        for j, p in enumerate(preds[:15]):
            poly = np.asarray(p.get('poly', []), dtype=np.float32)
            if poly.ndim != 2 or poly.shape[0] < 3:
                continue
            pts = [(float(x) * W, float(y) * H) for x, y in poly]
            dr.line(pts + [pts[0]], fill=(220, 40, 40), width=3)
            txt = str(p.get('text', ''))[:18]
            score = float(p.get('score', 0.0))
            lbl = f"{txt} | {score:.2f}" if txt else f"<NULL> | {score:.2f}"
            x0, y0 = pts[0]
            dr.rectangle((x0, max(0, y0 - 16), x0 + 8 * len(lbl) + 6, y0), fill=(255, 255, 255))
            dr.text((x0 + 3, max(0, y0 - 14)), lbl, fill=(130, 0, 0), font=_safe_default_font())
    return im
def _make_focus_overlay(base: Image.Image, prob: np.ndarray) -> Image.Image:
    arr = np.asarray(np.clip(prob, 0, 1) * 255.0, dtype=np.uint8)
    if cv2 is not None:
        heat = cv2.applyColorMap(arr, cv2.COLORMAP_TURBO)[:, :, ::-1]
    else:
        heat = np.repeat(arr[:, :, None], 3, axis=2)
    heat_im = Image.fromarray(heat).resize(base.size, Image.BICUBIC).convert('RGB')
    return Image.blend(base.convert('RGB'), heat_im, 0.40)
def _make_binary_mask_panel(prob: np.ndarray, threshold: float, target_size: Tuple[int, int]) -> Image.Image:
    mask = (np.asarray(prob) >= float(threshold)).astype(np.uint8) * 255
    if cv2 is not None:
        mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
    else:
        mask = np.repeat(mask[:, :, None], 3, axis=2)
    return Image.fromarray(mask).resize(target_size, Image.NEAREST).convert('RGB')
def _make_crop_gallery(base_img_t: torch.Tensor, preds: List[Dict[str, Any]], max_items: int = 6) -> Image.Image:
    pil = tensor_to_pil_unnorm(base_img_t)
    gallery_w, gallery_h = 760, 360
    canvas = Image.new('RGB', (gallery_w, gallery_h), (255, 255, 255))
    dr = ImageDraw.Draw(canvas)
    font = _safe_default_font()
    dr.rectangle((0, 0, gallery_w, gallery_h), outline=(220, 220, 220), width=2)
    dr.text((10, 8), 'Top OCR crops and decoded text', fill=(20, 20, 20), font=font)
    cell_w, cell_h = 240, 135
    items = preds[:max_items]
    for i, p in enumerate(items):
        r, c = divmod(i, 3)
        x = 10 + c * (cell_w + 8)
        y = 28 + r * (cell_h + 20)
        poly_abs = np.asarray(p['poly'], dtype=np.float32).copy()
        poly_abs[:, 0] *= pil.width - 1
        poly_abs[:, 1] *= pil.height - 1
        poly_abs = expand_poly_abs_for_crop(poly_abs, float(p.get('crop_pad', cfg.TT_CROP_RECTIFY_PAD)), pil.width, pil.height)
        crop = crop_totaltext_by_mode(pil, poly_abs, 'rectified')
        crop = _resize_contain(crop, (cell_w, cell_h), bg=(248, 248, 248))
        canvas.paste(crop, (x, y))
        txt = str(p.get('text', ''))[:28]
        score = float(p.get('score', 0.0))
        dr.rectangle((x, y + cell_h, x + cell_w, y + cell_h + 18), fill=(255, 255, 255))
        cap = txt if txt else "<NULL>"
        dr.text((x + 2, y + cell_h + 2), f"{cap} | score={score:.2f}", fill=(30, 30, 30), font=font)
    if not items:
        dr.text((10, 40), 'No predictions for this sample.', fill=(120, 120, 120), font=font)
    return canvas
def _make_info_panel(dataset_name: str, image_id: str, epoch: int, gt_valid: int, gt_ignore: int, preds: List[Dict[str, Any]], settings: Dict[str, Any], prob: np.ndarray) -> Image.Image:
    W, H = 760, 360
    panel = Image.new('RGB', (W, H), (255, 255, 255))
    dr = ImageDraw.Draw(panel)
    font = _safe_default_font()
    dr.rectangle((0, 0, W, H), outline=(220, 220, 220), width=2)
    y = 10
    lines = [
        f"Dataset: {dataset_name}",
        f"Sample : {image_id}",
        f"Epoch  : {epoch}",
        f"GT valid / ignore : {gt_valid} / {gt_ignore}",
        f"Predictions       : {len(preds)}",
        f"Mean score        : {np.mean([float(p.get('score', 0.0)) for p in preds]) if preds else 0.0:.3f}",
        f"Max prob / mean   : {float(np.max(prob)):.3f} / {float(np.mean(prob)):.3f}",
        '',
        'Selected postprocess',
        f"threshold={settings.get('threshold')}  unshrink={settings.get('unshrink')}",
        f"mode={settings.get('mode')}  min_area={settings.get('min_area')}",
        f"morph={settings.get('morph', 'none')}  crop_pad={settings.get('crop_pad')}",
        '',
        'What this figure shows',
        '• Input image and DB++ focus heatmap',
        '• Ground-truth polygons (green) and ignored regions (gray)',
        '• Predicted polygons (red) with recognized text',
        '• Rectified OCR crops showing reading capability',
    ]
    for ln in lines:
        dr.text((12, y), ln, fill=(25, 25, 25), font=font)
        y += 18
    return panel

def _draw_joint_gt_pred_overlay(base: Image.Image, gt_polys: np.ndarray, ignore: np.ndarray, gt_texts: Optional[List[str]], preds: List[Dict[str, Any]]) -> Image.Image:
    im = base.convert('RGB').copy()
    dr = ImageDraw.Draw(im)
    font = _safe_default_font()
    W, H = im.size
    for i, p in enumerate(gt_polys):
        pts = [(float(x) * W, float(y) * H) for x, y in p]
        ig = bool(ignore[i]) if i < len(ignore) else False
        color = (40, 190, 80) if not ig else (160, 160, 160)
        dr.line(pts + [pts[0]], fill=color, width=3)
        txt = ""
        if gt_texts is not None and i < len(gt_texts):
            txt = str(gt_texts[i]).strip() or "<NULL>"
        if txt:
            x0, y0 = pts[0]
            lbl = f"GT: {txt[:18]}"
            dr.rectangle((x0, max(0, y0 - 16), x0 + 8 * len(lbl) + 6, y0), fill=(255, 255, 255))
            dr.text((x0 + 3, max(0, y0 - 14)), lbl, fill=(0, 100, 0), font=font)
    for p in preds[:20]:
        poly = np.asarray(p.get('poly', []), dtype=np.float32)
        if poly.ndim != 2 or poly.shape[0] < 3:
            continue
        pts = [(float(x) * W, float(y) * H) for x, y in poly]
        dr.line(pts + [pts[0]], fill=(220, 40, 40), width=3)
        txt = str(p.get('text', '')).strip() or '<NULL>'
        score = float(p.get('score', 0.0))
        lbl = f"PR: {txt[:16]} | {score:.2f}"
        x0, y0 = pts[0]
        dr.rectangle((x0, y0, x0 + 8 * len(lbl) + 6, min(H, y0 + 16)), fill=(255, 255, 255))
        dr.text((x0 + 3, y0 + 1), lbl, fill=(130, 0, 0), font=font)
    dr.rectangle((8, 8, 290, 46), fill=(255, 255, 255))
    dr.text((12, 12), "Green = GT text | Red = predicted text", fill=(20, 20, 20), font=font)
    dr.text((12, 28), "Empty prediction is shown as <NULL>", fill=(90, 90, 90), font=font)
    return im

def _save_prediction_text_report(sample_dir: Path, preds: List[Dict[str, Any]]) -> None:
    lines = []
    payload = []
    for i, p in enumerate(preds):
        txt = str(p.get("text", "")).strip()
        score = float(p.get("score", 0.0))
        payload.append({
            "rank": i + 1,
            "text": txt,
            "score": score,
            "crop_pad": float(p.get("crop_pad", 1.0)),
            "poly": np.asarray(p.get("poly", []), dtype=float).tolist(),
        })
        lines.append(f"{i+1:02d}. text={txt!r} score={score:.4f} crop_pad={float(p.get('crop_pad', 1.0)):.2f}")
    (sample_dir / "predictions_readable.txt").write_text("\n".join(lines) if lines else "No predictions")
    (sample_dir / "predictions.json").write_text(json.dumps(payload, indent=2))
def _save_crop_individuals(sample_dir: Path, base_img_t: torch.Tensor, preds: List[Dict[str, Any]], max_items: int = 10) -> None:
    pil = tensor_to_pil_unnorm(base_img_t)
    crop_dir = sample_dir / "ocr_crops"
    crop_dir.mkdir(parents=True, exist_ok=True)
    font = _safe_default_font()
    for i, p in enumerate(preds[:max_items]):
        poly_abs = np.asarray(p['poly'], dtype=np.float32).copy()
        poly_abs[:, 0] *= pil.width - 1
        poly_abs[:, 1] *= pil.height - 1
        poly_abs = expand_poly_abs_for_crop(poly_abs, float(p.get('crop_pad', cfg.TT_CROP_RECTIFY_PAD)), pil.width, pil.height)
        crop = crop_totaltext_by_mode(pil, poly_abs, 'rectified').convert('RGB')
        score = float(p.get('score', 0.0))
        txt = str(p.get('text', '')).strip()
        header_h = 42
        canvas = Image.new('RGB', (crop.width, crop.height + header_h), (255, 255, 255))
        canvas.paste(crop, (0, header_h))
        dr = ImageDraw.Draw(canvas)
        dr.rectangle((0, 0, canvas.width, header_h), fill=(245, 247, 250))
        dr.text((8, 8), txt if txt else '<NULL>', fill=(20, 20, 20), font=font)
        dr.text((8, 24), f"score={score:.3f}", fill=(80, 80, 80), font=font)
        canvas.save(crop_dir / f"crop_{i+1:02d}.png")
def make_totaltext_visual_sheet(model: nn.Module, rec_model: Optional[nn.Module], records: List[Dict[str, Any]], tokenizer: TextTokenizer, settings: Dict[str, Any], out_path: Path, epoch: int) -> None:
    if cv2 is None:
        return
    base_dir = out_path.parent / f"{current_dataset_key()}_epoch_{epoch:03d}_samples"
    base_dir.mkdir(parents=True, exist_ok=True)
    indices = [i for i in cfg.TT_FIXED_VIS_IDS if i < len(records)]
    if not indices:
        return
    subset = [records[i] for i in indices]
    ds = TotalTextFullImageDataset(subset, train=False)
    loader = make_loader(ds, batch_size=1, shuffle=False, collate_fn=collate_tt_full, drop_last=False)
    probs, targets, imgs = collect_db_predictions(model, loader)
    pred_batches = [db_postprocess_prob(p, settings) for p in probs]
    if rec_model is not None:
        pred_batches = recognize_crops_from_pred_polys(rec_model, imgs, pred_batches, tokenizer)
    thumb_items = []
    font = _safe_default_font()
    for row, (record, prob, tgt, img_t, preds) in enumerate(zip(subset, probs, targets, imgs, pred_batches)):
        sample_stub = f"{current_dataset_key()}_ep{epoch:03d}_sample_{indices[row]:04d}"
        sample_dir = base_dir / sample_stub
        sample_dir.mkdir(parents=True, exist_ok=True)
        base = tensor_to_pil_unnorm(img_t).convert('RGB')
        focus = _make_focus_overlay(base, prob)
        binary = _make_binary_mask_panel(prob, float(settings.get('threshold', cfg.TT_DB_DEFAULT_THRESHOLD)), base.size)
        gt_polys = tgt['polys'].detach().cpu().numpy()
        ignore = tgt['ignore'].detach().cpu().numpy().astype(bool)
        gt_texts = tgt.get('texts', [])
        gt_overlay = _draw_poly_overlay(base, gt_polys, ignore, preds=None, gt_texts=gt_texts)
        pred_overlay = _draw_poly_overlay(base, gt_polys, ignore, preds=preds, gt_texts=None)
        joint_overlay = _draw_joint_gt_pred_overlay(base, gt_polys, ignore, gt_texts, preds)
        # Save clean individual panels at native resolution to avoid blur.
        base.save(sample_dir / '01_input.png')
        focus.save(sample_dir / '02_model_focus_overlay.png')
        binary.save(sample_dir / '03_binary_text_map.png')
        gt_overlay.save(sample_dir / '04_ground_truth_overlay.png')
        pred_overlay.save(sample_dir / '05_predictions_with_text.png')
        joint_overlay.save(sample_dir / '07_joint_gt_pred_overlay.png')
        crop_gallery = _make_crop_gallery(img_t, preds, max_items=6)
        crop_gallery.save(sample_dir / '06_ocr_crop_gallery.png')
        _save_crop_individuals(sample_dir, img_t, preds, max_items=10)
        _save_prediction_text_report(sample_dir, preds)
        # concise summary card for paper indexing
        summary_w = max(base.width, 1200)
        summary_h = 190
        summary = Image.new('RGB', (summary_w, summary_h), (255, 255, 255))
        dr = ImageDraw.Draw(summary)
        dr.rectangle((0, 0, summary_w, summary_h), outline=(220, 220, 220), width=2)
        title = f"{current_dataset_pretty()} | sample={Path(record.get('img_path', sample_stub)).name} | epoch={epoch}"
        dr.text((12, 10), title, fill=(20, 20, 20), font=font)
        dr.text((12, 34), f"GT valid={int((~ignore).sum())} ignore={int(ignore.sum())} | predictions={len(preds)} | mean_prob={float(np.mean(prob)):.3f} max_prob={float(np.max(prob)):.3f}", fill=(70, 70, 70), font=font)
        dr.text((12, 58), f"Postprocess: threshold={settings.get('threshold')} unshrink={settings.get('unshrink')} mode={settings.get('mode')} min_area={settings.get('min_area')} morph={settings.get('morph', 'none')} crop_pad={settings.get('crop_pad')}", fill=(70, 70, 70), font=font)
        dr.text((12, 86), 'Top predictions:', fill=(20, 20, 20), font=font)
        y = 108
        for i, p in enumerate(preds[:6]):
            txt = str(p.get('text', '')).strip() or '<NULL>'
            score = float(p.get('score', 0.0))
            dr.text((16, y), f"{i+1}. {txt}   score={score:.3f}", fill=(30, 30, 30), font=font)
            y += 18
        if not preds:
            dr.text((16, y), 'No predictions for this sample.', fill=(120, 120, 120), font=font)
        summary.save(sample_dir / '00_summary_card.png')
        # small thumbnail only for index
        thumb = _resize_contain(joint_overlay, (320, 220), bg=(255, 255, 255))
        thumb_items.append((thumb, sample_stub, preds[:3]))
        memory_safety_barrier(f"{current_dataset_tag()}-vis-sample-{row}")
    # Create a lightweight index sheet that points to the individual files.
    if thumb_items:
        cols = 2
        thumb_w, thumb_h = 320, 220
        cap_h = 70
        gap = 12
        rows = int(math.ceil(len(thumb_items) / cols))
        sheet = Image.new('RGB', (cols * (thumb_w + gap) + gap, rows * (thumb_h + cap_h + gap) + 70), (245, 245, 245))
        dr = ImageDraw.Draw(sheet)
        dr.text((12, 12), f"{current_dataset_pretty()} | visualization index | epoch {epoch}", fill=(20, 20, 20), font=font)
        dr.text((12, 30), f"Open each sample folder under: {base_dir.name}", fill=(90, 90, 90), font=font)
        for idx, (thumb, stub, top_preds) in enumerate(thumb_items):
            r, c = divmod(idx, cols)
            x = gap + c * (thumb_w + gap)
            y = 60 + r * (thumb_h + cap_h + gap)
            sheet.paste(thumb, (x, y))
            dr.rectangle((x, y + thumb_h, x + thumb_w, y + thumb_h + cap_h), fill=(255, 255, 255))
            dr.text((x + 6, y + thumb_h + 4), stub, fill=(40, 40, 40), font=font)
            pred_line = '; '.join([(str(p.get('text', '')).strip() or '<NULL>')[:18] for p in top_preds])
            dr.text((x + 6, y + thumb_h + 24), pred_line, fill=(80, 80, 80), font=font)
        sheet.save(out_path)
    log(f"[vis][{current_dataset_tag()}] saved individual visualizations to {base_dir} and index {out_path}")
def train_totaltext(tokenizer: TextTokenizer, records: Dict[str, Any]) -> Dict[str, Any]:
    section(f"{current_dataset_pretty().upper()} FULL-IMAGE TEXT SPOTTING")
    out_dir = Path(cfg.OUTPUT_ROOT) / current_dataset_run_dirname()
    out_dir.mkdir(parents=True, exist_ok=True)
    if cv2 is None:
        raise RuntimeError(f"cv2/OpenCV is required for {current_dataset_pretty()} DB++ masks and contour postprocessing.")
    crop_model, crop_summary = train_totaltext_crop_recognizer(tokenizer, records, out_dir)
    tt_train_records, tt_val_records = make_totaltext_split(records["tt_train"])
    save_json({"db_train_images": len(tt_train_records), "db_internal_val_images": len(tt_val_records), "official_test_images": len(records["tt_test"]), "note": "Postprocess is selected on train-internal validation, not official test."}, out_dir / "db_split_counts.json")
    train_ds = TotalTextFullImageDataset(tt_train_records, train=True, img_size=cfg.TOTALTEXT_IMG_SIZE)
    val_ds = TotalTextFullImageDataset(tt_val_records, train=False, img_size=cfg.TOTALTEXT_IMG_SIZE)
    test_ds = TotalTextFullImageDataset(records["tt_test"], train=False, img_size=cfg.TOTALTEXT_IMG_SIZE)
    tt_sampler = make_totaltext_image_sampler(tt_train_records)
    train_loader = make_loader(train_ds, cfg.TT_DB_BATCH_SIZE, shuffle=(tt_sampler is None), sampler=tt_sampler, collate_fn=collate_tt_full, drop_last=True)
    val_loader = make_loader(val_ds, int(getattr(cfg, "TT_DB_EVAL_BATCH_SIZE", cfg.TT_DB_BATCH_SIZE)), shuffle=False, collate_fn=collate_tt_full, drop_last=False)
    test_loader = make_loader(test_ds, int(getattr(cfg, "TT_DB_EVAL_BATCH_SIZE", cfg.TT_DB_BATCH_SIZE)), shuffle=False, collate_fn=collate_tt_full, drop_last=False)
    first = next(iter(train_loader))
    print_first_batch_debug(f"{current_dataset_tag()}-DB", first)
    del first
    clean_memory()
    model = TotalTextDBModel().to(cfg.DEVICE)
    if cfg.DEVICE == "cuda" and bool(getattr(cfg, "USE_CHANNELS_LAST", True)):
        model = model.to(memory_format=torch.channels_last)
    model = maybe_compile_model(model, f"{current_dataset_tag()}-DB")
    total, trainable = count_params(model)
    log(f"[{current_dataset_tag()}-DB] params total={total/1e6:.2f}M trainable={trainable/1e6:.2f}M")
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.") and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": cfg.TT_DB_LR_BACKBONE},
        {"params": head_params, "lr": cfg.TT_DB_LR_HEAD},
    ], weight_decay=cfg.TT_DB_WEIGHT_DECAY)
    scaler = GradScaler(enabled=(cfg.DEVICE == "cuda"))
    best_val_det = -1.0
    best_settings = {"threshold": cfg.TT_DB_DEFAULT_THRESHOLD, "unshrink": cfg.TT_DB_DEFAULT_UNSHRINK, "mode": cfg.TT_DB_DEFAULT_MODE, "min_area": cfg.TT_DB_DEFAULT_MIN_AREA, "max_preds": cfg.TT_DB_MAX_PREDS}
    metrics: List[Dict[str, Any]] = []
    log_every = max(int(getattr(cfg, 'MIN_LOG_STEPS', 10)), len(train_loader) // max(1, cfg.LOGS_PER_EPOCH))
    for epoch in range(1, int(cfg.EPOCHS_TT_DB_ARCFLOW) + 1):
        if bool(getattr(cfg, "TT_ENABLE_MULTI_SCALE_TRAINING", False)):
            sizes = tuple(int(x) for x in getattr(cfg, "TT_TRAIN_IMAGE_SIZES", (cfg.TOTALTEXT_IMG_SIZE,)))
            epoch_img_size = sizes[(epoch - 1) % max(1, len(sizes))]
            train_ds = TotalTextFullImageDataset(tt_train_records, train=True, img_size=epoch_img_size)
            tt_sampler = make_totaltext_image_sampler(tt_train_records)
            train_loader = make_loader(train_ds, cfg.TT_DB_BATCH_SIZE, shuffle=(tt_sampler is None), sampler=tt_sampler, collate_fn=collate_tt_full, drop_last=True)
            log(f"[{current_dataset_tag()}-DB][epoch {epoch}] train image size={epoch_img_size} sampler={'small-text' if tt_sampler is not None else 'shuffle'}")
        model.train(); meter = defaultdict(float); n_meter = 0; t0 = time.time()
        accum_steps = max(1, int(getattr(cfg, "GRAD_ACCUM_STEPS", 1)))
        # Same guard as crop recognizer: accumulation can delay the first optimizer step.
        last_grad = float("nan")
        optimizer.zero_grad(set_to_none=True)
        for step, (images, targets) in enumerate(train_loader, start=1):
            images = images.to(cfg.DEVICE, non_blocking=True)
            images = optimize_image_tensor_for_cuda(images)
            with autocast(device_type="cuda", dtype=get_amp_dtype(), enabled=(cfg.DEVICE == "cuda")):
                out = model(images)
                mask, weight_map = polys_to_db_mask_and_weight(targets, out["db_logits"].shape[-2], out["db_logits"].shape[-1], images.device)
                loss_db_only, stats = db_loss(out["db_logits"], mask, weight_map)
                loss_arc, arc_stats = arcflow_supervision_loss(model, out, targets, train=True)
                loss_bridge, bridge_stats = arcbridge_feature_loss(model, out, targets, train=True, epoch=epoch)
                loss = loss_db_only + loss_arc + float(cfg.ARCBRIDGE_WEIGHT) * loss_bridge
                loss_for_backward = loss / float(accum_steps)
                stats.update(arc_stats)
                stats.update(bridge_stats)
                stats["db_loss_only"] = float(loss_db_only.detach().cpu())
                stats["bridge_w"] = float(cfg.ARCBRIDGE_WEIGHT)
                stats["loss"] = float(loss.detach().cpu())
            scaler.scale(loss_for_backward).backward()
            if step % accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                last_grad = float(clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM))
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
            for k, v in stats.items(): meter[k] += float(v)
            n_meter += 1
            if step == 1 or step % log_every == 0 or step == len(train_loader):
                log(f"[{current_dataset_tag()}-DB+ArcFlow][ep {epoch}/{cfg.EPOCHS_TT_DB_ARCFLOW} step {step}/{len(train_loader)}] loss={stats['loss']:.4f} db={stats.get('db_loss_only',0.0):.4f} arc={stats.get('arc_loss',0.0):.4f} bridge={stats.get('bridge_loss',0.0):.4f} bce={stats['bce']:.4f} dice={stats['dice']:.4f} focal={stats['focal']:.4f} textMean={stats['textness_mean']:.3f} textMax={stats['textness_max']:.3f} pos={stats['pred_pos_pct']:.2f}/{stats['target_pos_pct']:.2f} arcN={stats.get('arc_instances',0)} grad={last_grad:.2f}")
                if cfg.PRINT_GPU_EVERY_LOG:
                    log(gpu_mem(f"{current_dataset_tag()}-DB"))
            try:
                del images, out, mask, weight_map, loss_db_only, loss_arc, loss_bridge, loss, loss_for_backward
            except Exception:
                pass
        avg = {k: v / max(1, n_meter) for k, v in meter.items()}
        try:
            del images, out, mask, weight_map, loss_db_only, loss_arc, loss_bridge, loss, loss_for_backward
        except Exception:
            pass
        optimizer.zero_grad(set_to_none=True)
        clean_memory()
        memory_safety_barrier(f"{current_dataset_tag()}-DB-before-val")
        log(gpu_mem(f"{current_dataset_tag()}-DB-before-val"))
        val_probs, val_targets, val_imgs = collect_db_predictions(model, val_loader)
        final_db_epoch = epoch == int(cfg.EPOCHS_TT_DB_ARCFLOW)
        force_sweep_epochs = set(int(x) for x in getattr(cfg, "TT_DB_FORCE_SWEEP_EPOCHS", ()))
        sweep_every = max(1, int(getattr(cfg, "TT_DB_POSTPROCESS_SWEEP_EVERY", 1)))
        run_sweep = (epoch == 1) or final_db_epoch or (epoch in force_sweep_epochs) or (epoch % sweep_every == 0)
        if run_sweep:
            use_e2e_select = bool(getattr(cfg, "TT_DB_ENABLE_E2E_SELECTION", True)) and (
                (bool(getattr(cfg, "TT_DB_E2E_SELECTION_FINAL_ONLY", True)) and final_db_epoch)
                or ((not bool(getattr(cfg, "TT_DB_E2E_SELECTION_FINAL_ONLY", True))) and epoch % max(1, int(getattr(cfg, "TT_DB_E2E_SELECTION_EVERY", 999))) == 0)
            )
            if use_e2e_select:
                epoch_settings, _ = sweep_geometry_alignment_e2e(
                    val_probs, val_targets, val_imgs, crop_model, tokenizer,
                    out_dir / "postprocess_validation", prefix=f"epoch_{epoch:03d}"
                )
                postprocess_monitor = "SWEEP+E2E_TOPK"
            else:
                epoch_settings, _ = sweep_db_postprocess(val_probs, val_targets, out_dir / "postprocess_validation", prefix=f"epoch_{epoch:03d}")
                postprocess_monitor = "SWEEP"
        else:
            epoch_settings = dict(best_settings)
            postprocess_monitor = f"CACHED(best@DetF={best_val_det:.3f})"
        gc.collect()
        if cfg.DEVICE == "cuda":
            torch.cuda.empty_cache()
        # Validation telemetry:
        # 1) Select postprocess using detection only on the train-internal validation split.
        # 2) OCR/E2E is expensive, so run it only every TT_DB_E2E_EVAL_INTERVAL epochs
        #    and on the final epoch. This E2E value is NOT used for selecting
        #    threshold/unshrink/mode/min_area. Official test E2E still runs once at the end.
        val_pred_batches = [db_postprocess_prob(p, epoch_settings) for p in val_probs]
        val_ev_det_only = evaluate_polygon_batches(val_pred_batches, val_targets, out_dir / f"val_det_epoch_{epoch:03d}.json")
        run_val_e2e = (int(cfg.TT_DB_E2E_EVAL_INTERVAL) > 0 and epoch % int(cfg.TT_DB_E2E_EVAL_INTERVAL) == 0) or (bool(cfg.TT_DB_E2E_EVAL_FINAL_EPOCH) and final_db_epoch)
        if run_val_e2e:
            val_pred_rec_batches = recognize_crops_from_pred_polys(crop_model, val_imgs, val_pred_batches, tokenizer)
            val_ev_e2e = evaluate_polygon_batches(val_pred_rec_batches, val_targets, out_dir / f"val_e2e_epoch_{epoch:03d}.json")
        else:
            val_ev_e2e = {
                "skipped": True,
                "reason": f"validation E2E is throttled; interval={cfg.TT_DB_E2E_EVAL_INTERVAL}",
                "epoch": epoch,
            }
        # Official test is not used for setting selection; final official test is run once after DB training.
        current_select_score = float(epoch_settings.get("alignment_selection_score", epoch_settings.get("selection_score", epoch_settings.get("val_det_f", 0.0))))
        if current_select_score > best_val_det:
            best_val_det = current_select_score
            best_settings = dict(epoch_settings)
            save_ckpt(out_dir / "db_best.pt", model, optimizer, scaler, epoch, best_val_det, {"best_postprocess": best_settings, "val_det": val_ev_det_only, "val_e2e": val_ev_e2e})
        save_ckpt(out_dir / "db_last.pt", model, optimizer, scaler, epoch, best_val_det, {"best_postprocess": best_settings, "val_det": val_ev_det_only, "val_e2e": val_ev_e2e})
        row = {
            "epoch": epoch,
            **avg,
            "val_det_f": val_ev_det_only.get("det_f", 0.0),
            "val_det_precision": val_ev_det_only.get("det_precision", 0.0),
            "val_det_recall": val_ev_det_only.get("det_recall", 0.0),
            "val_e2e_ran": not bool(val_ev_e2e.get("skipped", False)),
            "val_e2e_none_f": val_ev_e2e.get("e2e_none_f", None),
            "val_e2e_none_precision": val_ev_e2e.get("e2e_none_precision", None),
            "val_e2e_none_recall": val_ev_e2e.get("e2e_none_recall", None),
            "val_e2e_tp": val_ev_e2e.get("counts", {}).get("e2e_none_tp", None) if not val_ev_e2e.get("skipped", False) else None,
            "val_e2e_fp": val_ev_e2e.get("counts", {}).get("e2e_none_fp", None) if not val_ev_e2e.get("skipped", False) else None,
            "val_e2e_fn": val_ev_e2e.get("counts", {}).get("e2e_none_fn", None) if not val_ev_e2e.get("skipped", False) else None,
            "val_avg_preds_per_image": val_ev_det_only.get("avg_preds_per_image", 0.0),
            "best_threshold": best_settings.get("threshold"),
            "best_unshrink": best_settings.get("unshrink"),
            "best_mode": best_settings.get("mode"),
            "best_min_area": best_settings.get("min_area"),
            "postprocess_monitor": postprocess_monitor,
            "selection_score_used": float(epoch_settings.get("alignment_selection_score", epoch_settings.get("selection_score", epoch_settings.get("val_det_f", 0.0)))),
            "sweep_ran": bool(run_sweep),
            "minutes": (time.time() - t0) / 60.0,
        }
        metrics.append(row); write_csv(metrics, out_dir / "db_training_metrics.csv")
        if row["val_e2e_ran"]:
            e2e_part = (
                f"E2E={float(row['val_e2e_none_f']):.3f} "
                f"E2E_P={float(row['val_e2e_none_precision']):.3f} "
                f"E2E_R={float(row['val_e2e_none_recall']):.3f} "
                f"E2E_TP/FP/FN={row['val_e2e_tp']}/{row['val_e2e_fp']}/{row['val_e2e_fn']} "
            )
        else:
            e2e_part = f"E2E=SKIP(interval={cfg.TT_DB_E2E_EVAL_INTERVAL}) "
        log(
            f"[{current_dataset_tag()}-DB][epoch {epoch}] "
            f"VAL DetF={row['val_det_f']:.3f} P={row['val_det_precision']:.3f} R={row['val_det_recall']:.3f} "
            f"{e2e_part}"
            f"preds/img={row['val_avg_preds_per_image']:.2f} pp={row['postprocess_monitor']} best={best_settings} time={row['minutes']:.1f}min"
        )
        try:
            del val_probs, val_targets, val_imgs, val_pred_batches, val_pred_rec_batches
        except Exception:
            pass
        memory_safety_barrier(f"{current_dataset_tag()}-DB-after-val-epoch{epoch}", aggressive=False)
        if bool(getattr(cfg, "ENABLE_PAPER_VISUALS", True)) and (epoch in set(cfg.VIS_EPOCHS) or epoch == int(cfg.EPOCHS_TT_DB_ARCFLOW)):
            make_totaltext_visual_sheet(model, crop_model, records["tt_test"], tokenizer, best_settings, out_dir / "paper_visualizations" / f"{current_dataset_key()}_fixed_vis_epoch_{epoch:03d}.png", epoch=epoch)
        if cfg.FAIL_FAST_TOTALTEXT_LOW_DETF and epoch >= int(cfg.TOTALTEXT_DETF_HEALTH_EPOCH) and float(row["val_det_f"]) < float(cfg.TOTALTEXT_MIN_DETF_AFTER_HEALTH_EPOCH):
            raise RuntimeError(f"{current_dataset_pretty()} DB++ health gate failed: epoch {epoch} val Det-F={row['val_det_f']:.3f} < {cfg.TOTALTEXT_MIN_DETF_AFTER_HEALTH_EPOCH}. Inspect DB masks/postprocess visuals.")
        clean_memory()
    # Final official test evaluation using the best train-internal validation-selected
    # postprocess and, by default, the best validation-det DB checkpoint. This avoids
    # late-epoch detector drift in long runs without touching official test tuning.
    best_db_info: Dict[str, Any] = {"loaded": False, "reason": "disabled"}
    if bool(cfg.USE_BEST_DB_CHECKPOINT_FOR_FINAL_TEST):
        best_db_info = load_model_weights_for_eval(out_dir / "db_best.pt", model, "best DB++ detector")
    pred_crop_adapt_report = adapt_crop_recognizer_on_predicted_crops(model, crop_model, tokenizer, tt_train_records, tt_val_records, best_settings, out_dir)
    test_ev = eval_db_with_recognizer(model, crop_model, test_loader, tokenizer, best_settings, out_dir / f"{current_dataset_key()}_official_test_det_e2e.json", with_ocr=True)
    test_ev["best_db_checkpoint_for_final_test"] = best_db_info
    save_json(test_ev, out_dir / f"{current_dataset_key()}_official_test_det_e2e.json")
    log(
        f"[{current_dataset_tag()}-OFFICIAL-TEST][final] "
        f"DetF={float(test_ev.get('det_f', 0.0)):.3f} "
        f"P={float(test_ev.get('det_precision', 0.0)):.3f} "
        f"R={float(test_ev.get('det_recall', 0.0)):.3f} "
        f"E2E={float(test_ev.get('e2e_none_f', 0.0)):.3f} "
        f"E2E_P={float(test_ev.get('e2e_none_precision', 0.0)):.3f} "
        f"E2E_R={float(test_ev.get('e2e_none_recall', 0.0)):.3f} "
        f"preds/img={float(test_ev.get('avg_preds_per_image', 0.0)):.2f}"
    )
    # Geometry gap report is paper-facing and enabled by default. V7.17 fuses it
    # into the official eval pass to avoid running the detector over the test set twice.
    if bool(getattr(cfg, "ENABLE_GEOMETRY_GAP_EVAL", False)) and isinstance(test_ev, dict) and isinstance(test_ev.get("gt_vs_pred_geometry_gap"), dict):
        gap_report = test_ev["gt_vs_pred_geometry_gap"]
        log(f"[{current_dataset_tag()}-Gap] using fused official-eval gap report")
    elif bool(getattr(cfg, "ENABLE_GEOMETRY_GAP_EVAL", False)):
        gap_report = make_gt_vs_pred_geometry_gap(model, crop_model, test_loader, tokenizer, best_settings, out_dir)
    else:
        gap_report = {
            "skipped": True,
            "reason": "disabled_by_user_via_disable_gap_eval",
            "gt_geometry_e2e_none_f": 0.0,
            "predicted_geometry_e2e_none_f": 0.0,
            "geometry_to_recognition_gap": 0.0,
        }
        save_json(gap_report, out_dir / "gt_vs_pred_geometry_gap_report.json")
        log(f"[{current_dataset_tag()}-Gap][skip] disabled_by_user_via_disable_gap_eval")
    log(
        f"[{current_dataset_tag()}-Dashboard] official_DetF={float(test_ev.get('det_f', 0.0)):.3f} "
        f"official_E2E={float(test_ev.get('e2e_none_f', 0.0)):.3f} "
        f"GT_E2E={_fmt_metric(gap_report.get('gt_geometry_e2e_none_f'))} "
        f"Pred_E2E={_fmt_metric(gap_report.get('predicted_geometry_e2e_none_f'))} "
        f"settings={best_settings}"
    )
    official_bundle = test_ev.get("official_reporting", {}) if isinstance(test_ev, dict) else {}
    full_metrics = official_bundle.get("full", {}) if isinstance(official_bundle, dict) else {}
    strong_proxy = official_bundle.get("strong_proxy", {}) if isinstance(official_bundle, dict) else {}
    weak_proxy = official_bundle.get("weak_proxy", {}) if isinstance(official_bundle, dict) else {}
    table_rows = [{"split": "official_test", "det_p": test_ev.get("det_precision"), "det_r": test_ev.get("det_recall"), "det_f": test_ev.get("det_f"), "e2e_none_p": test_ev.get("e2e_none_precision"), "e2e_none_r": test_ev.get("e2e_none_recall"), "e2e_none_f": test_ev.get("e2e_none_f"), "e2e_full_f": full_metrics.get("e2e_none_f"), "icdar_strong_proxy_f": strong_proxy.get("e2e_none_f"), "icdar_weak_proxy_f": weak_proxy.get("e2e_none_f"), "avg_preds_per_image": test_ev.get("avg_preds_per_image"), "threshold": best_settings.get("threshold"), "unshrink": best_settings.get("unshrink"), "mode": best_settings.get("mode"), "min_area": best_settings.get("min_area"), "crop_width": cfg.OCR_FIXED_W}]
    write_csv(table_rows, out_dir / f"{current_dataset_key()}_det_e2e_table.csv")
    det_f_final = _safe_float_metric(test_ev.get("det_f", 0.0), 0.0)
    e2e_final = _safe_float_metric(test_ev.get("e2e_none_f", 0.0), 0.0)
    gt_e2e_final = _safe_float_metric(gap_report.get("gt_geometry_e2e_none_f", 0.0), 0.0)
    paper_metric_configuration_candidate = (
        det_f_final >= float(cfg.TOTALTEXT_TARGET_DETF)
        and e2e_final >= float(cfg.TOTALTEXT_TARGET_E2E)
        and gt_e2e_final >= float(cfg.TOTALTEXT_TARGET_GT_E2E)
    )
    architecture_configuration_candidate = (
        det_f_final >= float(cfg.TOTALTEXT_ARCH_TARGET_DETF)
        and e2e_final >= float(cfg.TOTALTEXT_ARCH_TARGET_E2E)
        and gt_e2e_final >= float(cfg.TOTALTEXT_ARCH_TARGET_GT_E2E)
    )
    configuration_candidate = bool(paper_metric_configuration_candidate)
    if paper_metric_configuration_candidate:
        status = "paper_metric_configuration_candidate"
    elif architecture_configuration_candidate:
        status = "architecture_configuration_candidate_needs_full_35_50_scale"
    elif det_f_final >= float(cfg.TOTALTEXT_TARGET_DETF_HEALTH) and e2e_final > float(cfg.TOTALTEXT_TARGET_E2E_HEALTH):
        status = "healthy_candidate_needs_more_ocr_or_final_scale"
    else:
        status = "needs_more_training_or_geometry_review"
    summary = {
        "model": f"{current_dataset_pretty()} DB++ full-image spotting",
        "status": status,
        "configuration_candidate": configuration_candidate,
        "paper_metric_configuration_candidate": bool(paper_metric_configuration_candidate),
        "architecture_configuration_candidate": bool(architecture_configuration_candidate),
        "configuration_targets": {
            "paper_metric_det_f": cfg.TOTALTEXT_TARGET_DETF,
            "paper_metric_e2e_none_f": cfg.TOTALTEXT_TARGET_E2E,
            "paper_metric_gt_geometry_e2e_none_f": cfg.TOTALTEXT_TARGET_GT_E2E,
            "architecture_det_f": cfg.TOTALTEXT_ARCH_TARGET_DETF,
            "architecture_e2e_none_f": cfg.TOTALTEXT_ARCH_TARGET_E2E,
            "architecture_gt_geometry_e2e_none_f": cfg.TOTALTEXT_ARCH_TARGET_GT_E2E,
        },
        "crop_summary": crop_summary,
        "best_postprocess_train_internal_val": best_settings,
        "official_test": test_ev,
        "predicted_crop_adaptation": pred_crop_adapt_report,
        "gt_vs_pred_geometry_gap": gap_report,
        "protocol_note": f"{current_dataset_pretty()} headline is full-image Det/E2E under its dataset-specific protocol. Crop recognition is auxiliary only.",
        "checkpoint_paths": {
            "totaltext_db_best": str(out_dir / "db_best.pt"),
            "totaltext_db_last": str(out_dir / "db_last.pt"),
            "totaltext_crop_best": str(out_dir / "tt_crop_recognizer" / "best.pt"),
            "totaltext_crop_last": str(out_dir / "tt_crop_recognizer" / "last.pt"),
        },
        "small_medium_large_recall": test_ev.get("small_medium_large_recall", {}),
    }
    write_final_research_eval_bundle(summary, out_dir)
    save_json(summary, out_dir / f"{current_dataset_key()}_readiness_summary.json")
    write_totaltext_summary_md(summary, out_dir / f"{current_dataset_key()}_readiness_summary.md")
    try:
        del train_loader, val_loader, test_loader, train_ds, val_ds, test_ds, optimizer, scaler
    except Exception:
        pass
    memory_safety_barrier(f"{current_dataset_tag()}-end-of-dataset", aggressive=True)
    return summary
def _denormalize_letterbox_poly(poly_norm: np.ndarray, meta: Dict[str, Any]) -> np.ndarray:
    pts = np.asarray(poly_norm, dtype=np.float32).reshape(-1, 2).copy()
    out_size = float(meta.get("out_size", cfg.TOTALTEXT_IMG_SIZE))
    scale = float(meta.get("scale", 1.0))
    pad_x = float(meta.get("pad_x", 0.0))
    pad_y = float(meta.get("pad_y", 0.0))
    orig_w = float(meta.get("orig_w", out_size))
    orig_h = float(meta.get("orig_h", out_size))
    pts[:, 0] = (pts[:, 0] * max(1.0, out_size - 1.0) - pad_x) / max(scale, 1e-6)
    pts[:, 1] = (pts[:, 1] * max(1.0, out_size - 1.0) - pad_y) / max(scale, 1e-6)
    pts[:, 0] = np.clip(pts[:, 0], 0, max(1.0, orig_w - 1.0))
    pts[:, 1] = np.clip(pts[:, 1], 0, max(1.0, orig_h - 1.0))
    return pts.astype(np.float32)
def build_predicted_crop_adaptation_records(model: nn.Module, source_records: List[Dict[str, Any]], settings: Dict[str, Any], out_dir: Path, split_name: str) -> List[Dict[str, Any]]:
    """Build predicted-crop OCR adaptation records by matching predictions to GT.

    V7.18 paper-full RAM-stable throughput mode keeps this stage enabled by default, but makes the
    builder genuinely streaming. The old implementation called collect_db_predictions
    and retained probabilities, target dicts, and image tensors for the whole subset.
    That was fine for small debug subsets but unsafe for final paper runs. This version
    processes one DataLoader batch at a time, writes compact progress reports, and only
    keeps the final matched crop records needed by TotalTextCropDataset.

    Detector/model logic is unchanged: predicted polygons are generated by the same
    DB++ postprocess and optional ArcFlow refinement, then matched to non-ignore,
    non-text_ignore GT by IoU >= PRED_CROP_ADAPT_IOU.
    """
    max_images = int(getattr(cfg, "PRED_CROP_ADAPT_MAX_IMAGES", 0))
    subset = list(source_records[:max_images]) if max_images > 0 else list(source_records)
    if not subset:
        save_json({"split": split_name, "source_images": 0, "records": 0, "matched_crops": 0, "reason": "empty source split"}, out_dir / f"pred_crop_adapt_{split_name}_build_report.json")
        return []

    model.eval()
    ds = TotalTextFullImageDataset(subset, train=False, img_size=cfg.TOTALTEXT_IMG_SIZE)
    builder_bs = max(1, int(getattr(cfg, "PRED_CROP_ADAPT_BUILDER_BATCH_SIZE", getattr(cfg, "TT_DB_EVAL_BATCH_SIZE", 1))))
    loader = make_loader(ds, builder_bs, shuffle=False, collate_fn=collate_tt_full, drop_last=False)
    match_iou = float(getattr(cfg, "PRED_CROP_ADAPT_IOU", 0.50))
    records: List[Dict[str, Any]] = []
    progress_rows: List[Dict[str, Any]] = []
    total_matches = 0
    total_preds = 0

    log(f"[{current_dataset_tag()}-PredCropAdapt:{split_name}] streaming builder start images={len(subset)} batches={len(loader)} batch_size={builder_bs} max_images={max_images}")
    memory_safety_barrier(f"{current_dataset_tag()}-predcrop-{split_name}-start")
    try:
        with torch.inference_mode():
            for bi, (images, targets) in enumerate(loader, start=1):
                images_cpu = [x.detach().cpu() for x in images]
                probs = _forward_db_probs_for_batch(model, images)
                pred_batches = [db_postprocess_prob(p, settings) for p in probs]
                if bool(getattr(cfg, "ARCFLOW_REFINE_PRED_POLYS_AT_EVAL", True)) and hasattr(unwrap_model(model), "arcflow_head"):
                    pred_batches = refine_pred_batches_with_arcflow(model, images_cpu, pred_batches)

                batch_records = 0
                batch_matches = 0
                batch_preds = 0
                for preds, tgt in zip(pred_batches, targets):
                    batch_preds += len(preds)
                    gt_polys = tgt["polys"].detach().cpu().numpy()
                    gt_texts = [normalize_for_eval(t, ascii_lower=True) for t in tgt.get("texts", [])]
                    ignore = tgt["ignore"].detach().cpu().numpy().astype(bool)
                    text_ignore_t = tgt.get("text_ignore", torch.zeros((len(gt_polys),), dtype=torch.bool))
                    text_ignore = text_ignore_t.detach().cpu().numpy().astype(bool) if isinstance(text_ignore_t, torch.Tensor) else np.asarray(text_ignore_t, dtype=bool)
                    gt_boxes = [poly_to_bbox(g) for g in gt_polys]
                    matched_gt: set = set()
                    instances: List[Dict[str, Any]] = []

                    for pi in sorted(range(len(preds)), key=lambda i: float(preds[i].get("score", 0.0)), reverse=True):
                        p = preds[pi]
                        p_box = p.get("box", poly_to_bbox(p["poly"]))
                        best_iou, best_gi = 0.0, None
                        for gi in range(len(gt_polys)):
                            if gi in matched_gt or bool(ignore[gi]) or (gi < len(text_ignore) and bool(text_ignore[gi])):
                                continue
                            txt = gt_texts[gi] if gi < len(gt_texts) else ""
                            if not txt:
                                continue
                            if not _bbox_could_overlap(p_box, gt_boxes[gi]):
                                continue
                            iou = polygon_iou_mask(p["poly"], gt_polys[gi])
                            if iou > best_iou:
                                best_iou, best_gi = iou, gi
                        if best_gi is not None and best_iou >= match_iou:
                            matched_gt.add(best_gi)
                            meta = tgt.get("meta", {})
                            poly_abs = _denormalize_letterbox_poly(np.asarray(p["poly"], dtype=np.float32), meta)
                            instances.append({
                                "polygon": poly_abs.tolist(),
                                "text": gt_texts[best_gi],
                                "ignore": False,
                                "text_ignore": False,
                                "source": f"{current_dataset_key()}_predcrop_adapt",
                                "matched_iou": float(best_iou),
                                "pred_score": float(p.get("score", 0.0)),
                            })
                            batch_matches += 1
                    if instances:
                        records.append({"image": tgt.get("path", ""), "instances": instances})
                        batch_records += 1

                total_matches += batch_matches
                total_preds += batch_preds
                log_every = max(1, int(getattr(cfg, "PRED_CROP_ADAPT_BUILD_LOG_EVERY", getattr(cfg, "STREAM_EVAL_LOG_EVERY", 25))))
                if bi == 1 or bi == len(loader) or bi % log_every == 0:
                    row = {
                        "batch": bi,
                        "images_seen": min(bi * builder_bs, len(subset)),
                        "records_so_far": len(records),
                        "matched_crops_so_far": total_matches,
                        "preds_so_far": total_preds,
                        "batch_records": batch_records,
                        "batch_matches": batch_matches,
                        "batch_preds": batch_preds,
                    }
                    progress_rows.append(row)
                    write_csv(progress_rows, out_dir / f"pred_crop_adapt_{split_name}_build_progress.csv")
                    log(f"[{current_dataset_tag()}-PredCropAdapt:{split_name}] batch {bi}/{len(loader)} records={len(records)} matches={total_matches} preds={total_preds}")
                    clean_every = max(1, int(getattr(cfg, "PRED_CROP_ADAPT_BUILD_CLEAN_EVERY", 200)))
                    if bi == len(loader) or bi % clean_every == 0:
                        memory_safety_barrier(f"{current_dataset_tag()}-predcrop-{split_name}-b{bi}")
                del images_cpu, probs, pred_batches, images, targets
    finally:
        model.train()
        clean_memory()

    report = {
        "split": split_name,
        "source_images": len(subset),
        "source_images_policy": "all" if max_images <= 0 else f"first_{max_images}",
        "records": len(records),
        "matched_crops": total_matches,
        "total_predictions_considered": total_preds,
        "iou_threshold": match_iou,
        "streaming_builder": True,
        "ram_safe": True,
    }
    save_json(report, out_dir / f"pred_crop_adapt_{split_name}_build_report.json")
    log(f"[{current_dataset_tag()}-PredCropAdapt:{split_name}] done records={len(records)} matches={total_matches} preds={total_preds}")
    memory_safety_barrier(f"{current_dataset_tag()}-predcrop-{split_name}-done", aggressive=True)
    return records
def adapt_crop_recognizer_on_predicted_crops(model: nn.Module, crop_model: nn.Module, tokenizer: TextTokenizer, train_records: List[Dict[str, Any]], val_records: List[Dict[str, Any]], settings: Dict[str, Any], out_dir: Path) -> Dict[str, Any]:
    """Validation-gated predicted-crop OCR adaptation.
    Keeps detector frozen and only adapts the crop recognizer to predicted polygon crops.
    Restores the previous recognizer if validation on predicted crops does not improve.
    """
    adapt_dir = out_dir / "predicted_crop_adaptation"
    adapt_dir.mkdir(parents=True, exist_ok=True)
    report: Dict[str, Any] = {"enabled": bool(getattr(cfg, "ENABLE_PRED_CROP_ADAPT", True)), "used": False, "research_full_default": True}
    if not bool(getattr(cfg, "ENABLE_PRED_CROP_ADAPT", True)) or int(getattr(cfg, "PRED_CROP_ADAPT_EPOCHS", 0)) <= 0:
        report["reason"] = "disabled_by_user_or_zero_pred_crop_adapt_epochs"
        save_json(report, adapt_dir / "pred_crop_adaptation_report.json")
        log(f"[{current_dataset_tag()}-PredCropAdapt][skip] {report['reason']}")
        return report
    log(f"[{current_dataset_tag()}-PredCropAdapt] building matched predicted crops")
    train_adapt_records = build_predicted_crop_adaptation_records(model, train_records, settings, adapt_dir, "train")
    val_adapt_records = build_predicted_crop_adaptation_records(model, val_records, settings, adapt_dir, "val")
    train_ds = TotalTextCropDataset(train_adapt_records, tokenizer, train=True)
    val_ds = TotalTextCropDataset(val_adapt_records, tokenizer, train=False)
    train_record_count = len(train_adapt_records)
    val_record_count = len(val_adapt_records)
    report.update({"train_records": train_record_count, "val_records": val_record_count, "train_crops": len(train_ds), "val_crops": len(val_ds)})
    del train_adapt_records, val_adapt_records
    memory_safety_barrier(f"{current_dataset_tag()}-predcrop-datasets-built", aggressive=True)
    if len(train_ds) < 32 or len(val_ds) < 8:
        report["reason"] = "not enough matched predicted crops"
        save_json(report, adapt_dir / "pred_crop_adaptation_report.json")
        log(f"[{current_dataset_tag()}-PredCropAdapt][skip] {report}")
        return report
    pred_crop_train_bs = max(1, int(getattr(cfg, "PRED_CROP_ADAPT_TRAIN_BATCH_CAP", int(cfg.TT_CROP_BATCH_SIZE))))
    train_loader = make_loader(train_ds, pred_crop_train_bs, shuffle=True, collate_fn=collate_totaltext_ocr, drop_last=True)
    val_loader = make_loader(val_ds, int(cfg.TT_CROP_EVAL_BATCH_SIZE), shuffle=False, collate_fn=collate_totaltext_ocr, drop_last=False)
    log(f"[{current_dataset_tag()}-PredCropAdapt] train_batch={pred_crop_train_bs} val_batch={int(cfg.TT_CROP_EVAL_BATCH_SIZE)} crops={len(train_ds)}/{len(val_ds)}")
    before = evaluate_tt_crop_recognizer(crop_model, val_loader, tokenizer, adapt_dir / "before_adapt_eval.json", epoch=0)
    before_score = float(before.get("WA", 0.0))
    backup_path = adapt_dir / "_recognizer_backup_before_predcrop.pt"
    backup_state = {k: v.detach().cpu() for k, v in unwrap_model(crop_model).state_dict().items()}
    torch.save(backup_state, backup_path)
    del backup_state
    memory_safety_barrier(f"{current_dataset_tag()}-predcrop-backup-saved", aggressive=True)
    optimizer = torch.optim.AdamW(parameter_groups(crop_model, cfg.LR_BACKBONE * float(getattr(cfg, "PRED_CROP_ADAPT_LR_FACTOR", 0.5)), cfg.LR_STR * float(getattr(cfg, "PRED_CROP_ADAPT_LR_FACTOR", 0.5))), weight_decay=cfg.WEIGHT_DECAY)
    scaler = GradScaler(enabled=(cfg.DEVICE == "cuda"))
    metrics: List[Dict[str, Any]] = []
    epochs = int(getattr(cfg, "PRED_CROP_ADAPT_EPOCHS", 2))
    for epoch in range(1, epochs + 1):
        crop_model.train(); meter = defaultdict(float); n_meter = 0; t0 = time.time()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader, start=1):
            images = batch["images"].to(cfg.DEVICE, non_blocking=True)
            images = optimize_image_tensor_for_cuda(images)
            domain_ids = torch.zeros((images.shape[0],), dtype=torch.long, device=cfg.DEVICE)
            batch = dict(batch); batch["domain_ids"] = domain_ids.detach().cpu()
            with autocast(device_type="cuda", dtype=get_amp_dtype(), enabled=(cfg.DEVICE == "cuda")):
                outputs = crop_model(images, domain_ids=domain_ids)
                loss, stats = str_loss(outputs, batch, max(1, cfg.EPOCHS_TT_CROP_WARMUP + epoch))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad = float(clip_grad_norm_(crop_model.parameters(), cfg.MAX_GRAD_NORM))
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            for k, v in stats.items(): meter[k] += float(v)
            n_meter += 1
        ev = evaluate_tt_crop_recognizer(crop_model, val_loader, tokenizer, adapt_dir / f"adapt_eval_epoch_{epoch:03d}.json", epoch=epoch)
        row = {"epoch": epoch, "train_loss": meter["loss"] / max(1, n_meter), "WA": ev.get("WA", 0.0), "CRR": ev.get("CRR", 0.0), "edit_le1": ev.get("edit_distance_le1_acc", 0.0), "empty_rate": ev.get("empty_pred_rate", 0.0), "minutes": (time.time() - t0) / 60.0}
        metrics.append(row); write_csv(metrics, adapt_dir / "pred_crop_adaptation_metrics.csv")
        log(f"[{current_dataset_tag()}-PredCropAdapt][epoch {epoch}/{epochs}] WA={row['WA']:.3f} E1={row['edit_le1']:.3f} empty={row['empty_rate']:.3f} time={row['minutes']:.1f}min")
    after = evaluate_tt_crop_recognizer(crop_model, val_loader, tokenizer, adapt_dir / "after_adapt_eval.json", epoch=epochs)
    after_score = float(after.get("WA", 0.0))
    # Accept only a real improvement. A tie such as 0.000 -> 0.000 should not overwrite
    # the recognizer, because it can make an already weak 1-epoch smoke run worse.
    accepted = bool(after_score > before_score + 1e-6)
    if not accepted:
        restored_state = torch.load(backup_path, map_location=cfg.DEVICE)
        unwrap_model(crop_model).load_state_dict(restored_state, strict=True)
        del restored_state
        report["reason"] = "validation predicted-crop WA did not improve; restored previous recognizer"
    else:
        save_ckpt(adapt_dir / "accepted_pred_crop_adapt.pt", crop_model, optimizer, scaler, epochs, after_score, {"before": before, "after": after})
        report["reason"] = "accepted"
    report.update({"used": accepted, "before_WA": before_score, "after_WA": after_score, "accepted": accepted, "metrics": metrics})
    save_json(report, adapt_dir / "pred_crop_adaptation_report.json")
    log(f"[{current_dataset_tag()}-PredCropAdapt] accepted={accepted} before_WA={before_score:.3f} after_WA={after_score:.3f}")
    clean_memory()
    return report
def make_gt_vs_pred_geometry_gap(model: nn.Module, rec_model: nn.Module, loader: DataLoader, tokenizer: TextTokenizer, settings: Dict[str, Any], out_dir: Path) -> Dict[str, Any]:
    """True RAM-safe GT-vs-pred geometry gap monitor.

    V7.10 still accumulated every batch's prediction dictionaries/targets and could
    grow RAM on CTW/ICDAR. V7.12 aggregates only TP/FP/FN counters batch-by-batch,
    while preserving the same polygon/E2E matching logic through evaluate_polygon_batches.
    """
    model.eval()
    rec_model.eval()
    pred_acc: Dict[str, Any] = {}
    gt_acc: Dict[str, Any] = {}
    progress_rows: List[Dict[str, Any]] = []
    log(f"[{current_dataset_tag()}-GapStream] start TRUE streaming batches={len(loader)}")
    try:
        with torch.inference_mode():
            for bi, (images, targets) in enumerate(loader, start=1):
                images_cpu = [x.detach().cpu() for x in images]
                probs = _forward_db_probs_for_batch(model, images)
                pred_batches = [db_postprocess_prob(p, settings) for p in probs]
                if bool(getattr(cfg, "ARCFLOW_REFINE_PRED_POLYS_AT_EVAL", True)) and hasattr(unwrap_model(model), "arcflow_head"):
                    pred_batches = refine_pred_batches_with_arcflow(model, images_cpu, pred_batches)
                pred_rec_batches = recognize_crops_from_pred_polys(rec_model, images_cpu, pred_batches, tokenizer)

                gt_batches: List[List[Dict[str, Any]]] = []
                for tgt in targets:
                    polys = tgt["polys"].detach().cpu().numpy()
                    ignore = tgt["ignore"].detach().cpu().numpy().astype(bool)
                    batch_gt = []
                    for p, ign in zip(polys, ignore):
                        if ign:
                            continue
                        batch_gt.append({"poly": p.astype(np.float32), "box": poly_to_bbox(p), "score": 1.0, "text": ""})
                    gt_batches.append(batch_gt)
                gt_rec_batches = recognize_crops_from_pred_polys(rec_model, images_cpu, gt_batches, tokenizer)

                pred_ev_b = evaluate_polygon_batches(pred_rec_batches, targets, None)
                gt_ev_b = evaluate_polygon_batches(gt_rec_batches, targets, None)
                _merge_eval_counts(pred_acc, pred_ev_b, len(targets), pred_rec_batches)
                _merge_eval_counts(gt_acc, gt_ev_b, len(targets), gt_rec_batches)

                if bi == 1 or bi == len(loader) or bi % max(1, int(getattr(cfg, "STREAM_EVAL_LOG_EVERY", 50))) == 0:
                    pred_now = _finalize_eval_acc(pred_acc)
                    gt_now = _finalize_eval_acc(gt_acc)
                    progress_rows.append({
                        "batch": bi,
                        "images": int(pred_acc.get("images", 0)),
                        "gt_geometry_e2e_none_f": gt_now.get("e2e_none_f", 0.0),
                        "predicted_geometry_e2e_none_f": pred_now.get("e2e_none_f", 0.0),
                        "gap": float(gt_now.get("e2e_none_f", 0.0)) - float(pred_now.get("e2e_none_f", 0.0)),
                        "pred_det_f": pred_now.get("det_f", 0.0),
                    })
                    log(
                        f"[{current_dataset_tag()}-GapStream] batch {bi}/{len(loader)} images={pred_acc.get('images',0)} "
                        f"GT_E2E={float(gt_now.get('e2e_none_f',0.0)):.3f} "
                        f"Pred_E2E={float(pred_now.get('e2e_none_f',0.0)):.3f} "
                        f"Gap={float(gt_now.get('e2e_none_f',0.0))-float(pred_now.get('e2e_none_f',0.0)):.3f}"
                    )
                del images_cpu, probs, pred_batches, pred_rec_batches, gt_batches, gt_rec_batches, images, targets
                if bi % 25 == 0:
                    memory_safety_barrier(f"{current_dataset_tag()}-gap-stream-b{bi}")
    except Exception as e:
        report = {
            "streaming_gap_eval": True,
            "true_low_ram_streaming": True,
            "failed": True,
            "error": repr(e),
            "gt_partial": _finalize_eval_acc(gt_acc),
            "pred_partial": _finalize_eval_acc(pred_acc),
            "progress_rows": progress_rows,
            "interpretation": "Gap monitor failed before completion. Partial counters are retained; training/eval can continue unless STRICT_GAP_EVAL=True.",
        }
        save_json(report, out_dir / "gt_vs_pred_geometry_gap_report.json")
        write_csv(progress_rows, out_dir / "gt_vs_pred_geometry_gap_progress.csv")
        log(f"[{current_dataset_tag()}-Gap][warn] failed but continuing: {repr(e)}")
        if bool(getattr(cfg, "STRICT_GAP_EVAL", False)):
            raise
        model.train()
        rec_model.train()
        return report

    pred_ev = _finalize_eval_acc(pred_acc)
    gt_ev = _finalize_eval_acc(gt_acc)
    report = {
        "gt_geometry_e2e_none_f": gt_ev.get("e2e_none_f", 0.0),
        "predicted_geometry_e2e_none_f": pred_ev.get("e2e_none_f", 0.0),
        "gt_geometry_det_f": gt_ev.get("det_f", 0.0),
        "predicted_geometry_det_f": pred_ev.get("det_f", 0.0),
        "geometry_to_recognition_gap": float(gt_ev.get("e2e_none_f", 0.0)) - float(pred_ev.get("e2e_none_f", 0.0)),
        "streaming_gap_eval": True,
        "true_low_ram_streaming": True,
        "counts": {"gt": gt_ev.get("counts", {}), "predicted": pred_ev.get("counts", {})},
        "progress_rows": progress_rows,
        "interpretation": "High GT-geometry E2E with low predicted E2E means predicted geometry/alignment is the bottleneck.",
    }
    save_json(report, out_dir / "gt_vs_pred_geometry_gap_report.json")
    write_csv(progress_rows, out_dir / "gt_vs_pred_geometry_gap_progress.csv")
    log(f"[{current_dataset_tag()}-Gap] GT_E2E={report['gt_geometry_e2e_none_f']:.3f} Pred_E2E={report['predicted_geometry_e2e_none_f']:.3f} Gap={report['geometry_to_recognition_gap']:.3f}")
    memory_safety_barrier(f"{current_dataset_tag()}-gap-stream-done")
    model.train()
    rec_model.train()
    return report


def write_final_research_eval_bundle(summary: Dict[str, Any], out_dir: Path) -> None:
    """Write final paper-facing metric tables in one stable format per dataset."""
    dataset = current_dataset_key()
    pretty = current_dataset_pretty()
    ev = summary.get("official_test", {}) if isinstance(summary, dict) else {}
    crop = summary.get("crop_summary", {}).get("final", {}) if isinstance(summary.get("crop_summary", {}), dict) else {}
    gap = summary.get("gt_vs_pred_geometry_gap", {}) if isinstance(summary, dict) else {}
    pp = summary.get("best_postprocess_train_internal_val", {}) if isinstance(summary, dict) else {}
    off = ev.get("official_reporting", {}) if isinstance(ev, dict) else {}
    full = off.get("full", {}) if isinstance(off, dict) else {}
    strong_proxy = off.get("strong_proxy", {}) if isinstance(off, dict) else {}
    weak_proxy = off.get("weak_proxy", {}) if isinstance(off, dict) else {}
    generic_proxy = off.get("generic_proxy", {}) if isinstance(off, dict) else {}
    research_dir = out_dir / "official_research_eval"
    research_dir.mkdir(parents=True, exist_ok=True)
    det_row = {
        "dataset": pretty, "dataset_key": dataset, "extra_data": "None / real-only", "det_p": ev.get("det_precision"),
        "det_r": ev.get("det_recall"), "det_hmean": ev.get("det_f"), "avg_preds_per_image": ev.get("avg_preds_per_image"),
    }
    e2e_row = {
        "dataset": pretty, "dataset_key": dataset, "extra_data": "None / real-only",
        "none": ev.get("e2e_none_f"), "none_p": ev.get("e2e_none_precision"), "none_r": ev.get("e2e_none_recall"),
        "strong": None, "weak": None, "generic": None, "full": None,
        "strong_proxy": None, "weak_proxy": None, "generic_proxy": None,
        "protocol_note": "",
    }
    if dataset in {"total_text", "ctw1500"}:
        e2e_row["full"] = full.get("e2e_none_f")
        e2e_row["protocol_note"] = "Reports None and Full lexicon settings."
    elif dataset == "icdar2015":
        e2e_row["strong"] = None
        e2e_row["weak"] = None
        e2e_row["generic"] = None
        e2e_row["strong_proxy"] = strong_proxy.get("e2e_none_f")
        e2e_row["weak_proxy"] = weak_proxy.get("e2e_none_f")
        e2e_row["generic_proxy"] = generic_proxy.get("e2e_none_f")
        e2e_row["protocol_note"] = "Official ICDAR S/W/G require release lexicon files; proxies are diagnostic only."
    diagnostic = {
        "dataset": dataset,
        "crop_WA": crop.get("WA"), "crop_CRR": crop.get("CRR"), "crop_edit_le1": crop.get("edit_le1"),
        "gt_geometry_e2e": gap.get("gt_geometry_e2e_none_f"), "predicted_geometry_e2e": gap.get("predicted_geometry_e2e_none_f"),
        "geometry_gap": gap.get("geometry_to_recognition_gap"), "best_postprocess": pp,
        "pred_crop_adaptation": summary.get("predicted_crop_adaptation", {}),
        "peak_vram_note": "See run log and db/crop metric CSV telemetry for per-stage VRAM.",
    }
    write_csv([det_row], research_dir / f"{dataset}_paper_detection_table.csv")
    write_csv([e2e_row], research_dir / f"{dataset}_paper_e2e_table.csv")
    save_json({"detection": det_row, "e2e": e2e_row, "diagnostic": diagnostic, "raw_official_test": ev, "official_reporting": off}, research_dir / f"{dataset}_final_research_metrics.json")
    md = [
        f"# {pretty} Final Research Evaluation",
        "",
        "## Detection",
        "",
        "| Dataset | Extra Data | Det-P | Det-R | Det-H |",
        "|---|---|---:|---:|---:|",
        f"| {pretty} | None / real-only | {_fmt_metric(det_row.get('det_p'))} | {_fmt_metric(det_row.get('det_r'))} | {_fmt_metric(det_row.get('det_hmean'))} |",
        "",
        "## End-to-End Spotting",
        "",
        "| Dataset | None | Strong | Weak | Generic | Full | Notes |",
        "|---|---:|---:|---:|---:|---:|---|",
        f"| {pretty} | {_fmt_metric(e2e_row.get('none'))} | {_fmt_metric(e2e_row.get('strong'))} | {_fmt_metric(e2e_row.get('weak'))} | {_fmt_metric(e2e_row.get('generic'))} | {_fmt_metric(e2e_row.get('full'))} | {e2e_row.get('protocol_note')} |",
        "",
        "## Diagnostics",
        "",
        f"- Crop WA: `{_fmt_metric(diagnostic.get('crop_WA'))}`",
        f"- GT-geometry E2E: `{_fmt_metric(diagnostic.get('gt_geometry_e2e'))}`",
        f"- Predicted-geometry E2E: `{_fmt_metric(diagnostic.get('predicted_geometry_e2e'))}`",
        f"- Geometry gap: `{_fmt_metric(diagnostic.get('geometry_gap'))}`",
        f"- Best postprocess: `{pp}`",
    ]
    (research_dir / f"{dataset}_final_paper_summary.md").write_text("\n".join(md), encoding="utf-8")
    log(f"[{current_dataset_tag()}-ResearchEval] wrote {research_dir}")
def write_totaltext_summary_md(summary: Dict[str, Any], path: Path) -> None:
    ev = summary.get("official_test", {})
    crop = summary.get("crop_summary", {})
    post = summary.get("best_postprocess_train_internal_val", {})
    pretty = summary.get("dataset_pretty") or current_dataset_pretty()
    protocol = summary.get("protocol_report", {})
    lines = [
        f"# ArcFlowText {pretty} Full-Image Summary",
        "",
        f"**Status:** `{summary.get('status')}`",
        "",
        f"## {pretty} Final Full-Image Test Metrics",
        "",
        "| Metric | Value |",
        "|---|---:|",
        f"| Det-P | {ev.get('det_precision')} |",
        f"| Det-R | {ev.get('det_recall')} |",
        f"| Det-F | {ev.get('det_f')} |",
        f"| E2E None-P | {ev.get('e2e_none_precision')} |",
        f"| E2E None-R | {ev.get('e2e_none_recall')} |",
        f"| E2E None-F | {ev.get('e2e_none_f')} |",
        f"| Avg preds/image | {ev.get('avg_preds_per_image')} |",
        "",
        "## Auxiliary Crop Recognizer Diagnostic",
        "",
        f"- Best crop metric: `{crop.get('best_checkpoint_score')}`",
        f"- Final crop status: `{crop.get('status')}`",
        f"- This is diagnostic only, not the headline {pretty} benchmark.",
        "",
        "## Train-Internal Postprocess Selection",
        "",
        f"- threshold: `{post.get('threshold')}`",
        f"- unshrink: `{post.get('unshrink')}`",
        f"- mode: `{post.get('mode')}`",
        f"- min_area: `{post.get('min_area')}`",
        "",
        "## Protocol",
        "",
        f"- Detection: {protocol.get('detection', 'Polygon/quad IoU > 0.5, Precision/Recall/Hmean')}",
        f"- E2E: {protocol.get('e2e_none', 'IoU > 0.5 and exact normalized transcription match, lexicon-free')}",
        "- Postprocess selection is done on a train-internal validation split, not the official test split.",
    ]
    path.write_text("\n".join(lines), encoding="utf-8")
# =============================================================================
# 9. MAIN ENTRYPOINT
# =============================================================================
def prepare_output_root() -> Path:
    out_root = Path(cfg.OUTPUT_ROOT)
    if out_root.exists() and cfg.CLEAN_OUTPUT_ROOT and not cfg.RESUME:
        log(f"[clean] removing stale output root: {out_root}")
        shutil.rmtree(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    save_json(asdict(cfg), out_root / "totaltext_config.json")
    return out_root
def write_final_run_summary(out_root: Path, totaltext_summary: Optional[Dict[str, Any]] = None) -> None:
    summary = {
        "run": "ArcFlowText Total-Text full-image run",
        "arch_version": cfg.ARCH_VERSION,
        "dataset": "Total-Text only",
        "backbone": cfg.BACKBONE_NAME,
        "schedule": {
            "crop_warmup_epochs": cfg.EPOCHS_TT_CROP_WARMUP,
            "db_arcflow_arcbridge_epochs": cfg.EPOCHS_TT_DB_ARCFLOW,
        },
        "totaltext": totaltext_summary or "FAILED_OR_SKIPPED",
        "notes": [
            "This executable contains only the Total-Text training and evaluation path.",
            "Crop recognition remains an auxiliary diagnostic; headline metrics are full-image Total-Text Det/E2E.",
        ],
    }
    save_json(summary, out_root / "totaltext_summary.json")
    md = [
        "# ArcFlowText Total-Text Camera-Ready Summary",
        "",
        f"**Arch version:** `{cfg.ARCH_VERSION}`",
        f"**Backbone:** `{cfg.BACKBONE_NAME}`",
        f"**Crop warm-up epochs:** `{cfg.EPOCHS_TT_CROP_WARMUP}`",
        f"**DB++/ArcFlow/ArcBridge epochs:** `{cfg.EPOCHS_TT_DB_ARCFLOW}`",
        "",
        "Only the Total-Text pipeline was run. The official headline remains full-image Det/E2E, not crop WA.",
        "",
        f"**Total-Text status:** `{(totaltext_summary or {}).get('status')}`",
    ]
    (out_root / "totaltext_summary.md").write_text("\n".join(md), encoding="utf-8")
def sync_outputs_to_drive(out_root: Path) -> None:
    dst = Path(cfg.DRIVE_SYNC_ROOT)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(out_root, dst)
    log(f"[sync] copied outputs to {dst}")
def _safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name)).strip("_") or "run"
def _fmt_metric(value: Any) -> str:
    if value is None:
        return "NA"
    try:
        return f"{float(value):.3f}"
    except Exception:
        return str(value)

def _safe_float_metric(value: Any, default: float = 0.0) -> float:
    if value is None:
        return float(default)
    try:
        if isinstance(value, str) and value.strip().upper() in {"", "NA", "NONE", "NULL"}:
            return float(default)
        return float(value)
    except Exception:
        return float(default)
def print_backbone_result_table(comparison: List[Dict[str, Any]]) -> None:
    """Print the final backbone comparison directly into the notebook console."""
    section("TOTAL-TEXT BACKBONE RESULT COMPARISON")
    if not comparison:
        log("No completed backbone runs were available for comparison.")
        return
    headers = [
        "Backbone", "Status", "Det-F", "E2E None-F",
        "GT-Geom E2E", "Pred-Geom E2E", "Geom Gap", "Output Root",
    ]
    rows = []
    for row in comparison:
        rows.append([
            str(row.get("backbone", "NA")),
            str(row.get("status", "NA")),
            _fmt_metric(row.get("det_f")),
            _fmt_metric(row.get("e2e_none_f")),
            _fmt_metric(row.get("gt_geometry_e2e_none_f")),
            _fmt_metric(row.get("pred_geometry_e2e_none_f")),
            _fmt_metric(row.get("geometry_gap")),
            str(row.get("output_root", "NA")),
        ])
    widths = [len(h) for h in headers]
    for row in rows:
        widths = [max(w, len(str(v))) for w, v in zip(widths, row)]
    def line(parts: Sequence[str]) -> str:
        return " | ".join(str(v).ljust(w) for v, w in zip(parts, widths))
    print(line(headers), flush=True)
    print("-+-".join("-" * w for w in widths), flush=True)
    for row in rows:
        print(line(row), flush=True)
    try:
        ranked = sorted(
            comparison,
            key=lambda r: float(r.get("e2e_none_f") if r.get("e2e_none_f") is not None else -1.0),
            reverse=True,
        )
        best = ranked[0]
        log(
            f"[comparison] best_by_e2e_none={best.get('backbone')} "
            f"DetF={_fmt_metric(best.get('det_f'))} "
            f"E2E={_fmt_metric(best.get('e2e_none_f'))} "
            f"GeomGap={_fmt_metric(best.get('geometry_gap'))}"
        )
    except Exception:
        pass
def write_backbone_comparison_markdown(comparison: List[Dict[str, Any]], path: Path) -> None:
    """Save the same comparison in a paper-review-friendly Markdown table."""
    lines = [
        "# Total-Text Backbone Comparison",
        "",
        "| Backbone | Status | Det-F | E2E None-F | GT-Geometry E2E | Pred-Geometry E2E | Geometry Gap | Output Root |",
        "|---|---:|---:|---:|---:|---:|---:|---|",
    ]
    for row in comparison:
        lines.append(
            "| {backbone} | {status} | {det_f} | {e2e} | {gt} | {pred} | {gap} | `{root}` |".format(
                backbone=row.get("backbone", "NA"),
                status=row.get("status", "NA"),
                det_f=_fmt_metric(row.get("det_f")),
                e2e=_fmt_metric(row.get("e2e_none_f")),
                gt=_fmt_metric(row.get("gt_geometry_e2e_none_f")),
                pred=_fmt_metric(row.get("pred_geometry_e2e_none_f")),
                gap=_fmt_metric(row.get("geometry_gap")),
                root=row.get("output_root", "NA"),
            )
        )
    path.write_text("\n".join(lines), encoding="utf-8")
def backbone_requires_vitae_setup(name: str) -> bool:
    name_l = str(name).lower().replace("-", "_")
    return name_l in {"vitaev2_s", "vitaev2s", "vitae_v2_s", "vitae2_s", "vitae_s", "vitae", "vitaev2_s_224", "vitaev2_s_224_in1k"} or str(name) == "ViTAEv2_S"
def preflight_backbone_if_needed(name: str, amp_dtype: str = "float16") -> None:
    """Validate ViTAEv2-S integration before long training starts."""
    if not backbone_requires_vitae_setup(name):
        return
    section(f"BACKBONE PREFLIGHT: {name}")
    old_name = cfg.BACKBONE_NAME
    old_amp = cfg.AMP_DTYPE
    try:
        cfg.BACKBONE_NAME = name
        cfg.AMP_DTYPE = amp_dtype
        device = get_device()
        model = TotalTextBackbone(name, cfg.PRETRAIN_IMAGENET).to(device).eval()
        test_shapes = [(1, 3, 224, 224), (1, 3, cfg.TOTALTEXT_CROP_H, cfg.TOTALTEXT_CROP_W)]
        with torch.inference_mode():
            for shape in test_shapes:
                x = torch.randn(*shape, device=device)
                feats = model(x)
                shape_map = {k: list(v.shape) for k, v in feats.items()}
                log(f"[preflight] {name} input={list(shape)} feature_shapes={shape_map}")
        del model
        clean_memory()
        log(f"[preflight] {name} official ViTAEv2-S adapter is ready for training.")
    finally:
        cfg.BACKBONE_NAME = old_name
        cfg.AMP_DTYPE = old_amp
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="ArcFlowText Total-Text full-image text spotting")
    parser.add_argument("--dataset_tar", type=str, default=cfg.DRIVE_TAR, help="Path to ArcFlowText_DATASET.tar")
    parser.add_argument("--output_root", type=str, default=cfg.OUTPUT_ROOT, help="Base local output root")
    parser.add_argument("--drive_sync_root", type=str, default=cfg.DRIVE_SYNC_ROOT, help="Base Drive sync root")
    parser.add_argument("--backbone", type=str, default=cfg.BACKBONE_NAME, help="Backbone for a single run, e.g. resnet50 or vitae/vitaev2_s")
    parser.add_argument("--compare_backbones", type=str, default="", help="Comma-separated backbone list, e.g. resnet50,vitaev2_s")
    parser.add_argument("--crop_epochs", type=int, default=cfg.EPOCHS_TT_CROP_WARMUP, help="Total-Text crop warm-up epochs")
    parser.add_argument("--db_epochs", type=int, default=cfg.EPOCHS_TT_DB_ARCFLOW, help="DB++ + ArcFlowAlign + ArcBridge epochs")
    parser.add_argument("--no_sync", action="store_true", help="Disable final copy to Drive")
    parser.add_argument("--resume", action="store_true", help="Resume from existing output root when checkpoints are present")
    parser.add_argument("--no_clean", action="store_true", help="Do not delete an existing output root before running")
    parser.add_argument("--num_workers", type=int, default=cfg.NUM_WORKERS)
    parser.add_argument("--amp_dtype", type=str, default=cfg.AMP_DTYPE, choices=["float16", "bfloat16", "float32"])
    parser.add_argument("--no_auto_t4_profile", action="store_true", help="Disable automatic low-RAM Tesla T4 runtime clamps. Not recommended for Simple T4 final runs.")
    parser.add_argument("--no_backbone_preflight", action="store_true", help="Skip early ViTAEv2-S adapter validation before training")
    return parser.parse_args()
def run_totaltext_once(backbone: str, base_output_root: str, base_drive_root: str, args: argparse.Namespace) -> Dict[str, Any]:
    cfg.DRIVE_TAR = args.dataset_tar
    cfg.BACKBONE_NAME = backbone
    cfg.EPOCHS_TT_CROP_WARMUP = int(args.crop_epochs)
    cfg.EPOCHS_TT_DB_ARCFLOW = int(args.db_epochs)
    cfg.SYNC_TO_DRIVE_AT_END = not bool(args.no_sync)
    cfg.RESUME = bool(args.resume)
    cfg.CLEAN_OUTPUT_ROOT = not bool(args.no_clean)
    cfg.TT_DB_ENABLE_E2E_SELECTION = not bool(getattr(args, "disable_e2e_selection", False))
    cfg.NUM_WORKERS = int(args.num_workers)
    cfg.AMP_DTYPE = str(args.amp_dtype)
    run_suffix = _safe_name(backbone)
    cfg.OUTPUT_ROOT = str(Path(base_output_root) / f"totaltext_{run_suffix}")
    cfg.DRIVE_SYNC_ROOT = str(Path(base_drive_root) / f"totaltext_{run_suffix}")
    set_seed(cfg.SEED)
    print_hardware()
    section("TOTAL-TEXT CAMERA-READY CONTROL PANEL")
    log(f"Dataset tar             : {cfg.DRIVE_TAR}")
    log(f"Backbone                : {cfg.BACKBONE_NAME}")
    log(f"Run mode                : Total-Text-only full-image pipeline")
    log(f"Crop warm-up epochs     : {cfg.EPOCHS_TT_CROP_WARMUP}")
    log(f"DB/ArcFlow epochs       : {cfg.EPOCHS_TT_DB_ARCFLOW}")
    log(f"ArcFlowAlign enabled    : {cfg.ENABLE_ARCFLOW_ALIGN}")
    log(f"ArcBridge enabled       : {cfg.ENABLE_ARCBRIDGE}, weight={cfg.ARCBRIDGE_WEIGHT}, delay={cfg.ARCBRIDGE_DELAY_EPOCHS}")
    log("Rule: Total-Text only; no official-test postprocess tuning.")
    out_root = prepare_output_root()
    local_root = copy_and_extract_dataset()
    paths = discover_dataset_paths(local_root)
    records = build_all_real_records(paths)
    sanity_counts = sanity_check_records(records)
    save_json(sanity_counts, out_root / "dataset_counts_totaltext.json")
    save_json(asdict(cfg), out_root / "totaltext_runtime_config.json")
    tokenizer, _ = build_tokenizer_and_domain_ids(records, out_root)
    totaltext_summary = train_totaltext(tokenizer, records)
    write_final_run_summary(out_root, totaltext_summary=totaltext_summary)
    if cfg.SYNC_TO_DRIVE_AT_END:
        sync_outputs_to_drive(out_root)
    section("DONE: TOTAL-TEXT CAMERA-READY RUN")
    log(f"Summary: {out_root / 'totaltext_summary.md'}")
    log(f"Total-Text details: {Path(cfg.OUTPUT_ROOT) / 'totaltext_full_image_run' / 'totaltext_readiness_summary.md'}")
    return totaltext_summary
def main() -> None:
    args = parse_args()
    base_output_root = args.output_root
    base_drive_root = args.drive_sync_root
    if args.compare_backbones.strip():
        backbones = [b.strip() for b in args.compare_backbones.split(",") if b.strip()]
    else:
        backbones = [args.backbone]
    if not bool(args.no_backbone_preflight):
        for bb in backbones:
            preflight_backbone_if_needed(bb, amp_dtype=args.amp_dtype)
        set_seed(cfg.SEED)
    comparison: List[Dict[str, Any]] = []
    for bb in backbones:
        section(f"STARTING BACKBONE RUN: {bb}")
        clean_memory()
        summary = run_totaltext_once(bb, base_output_root, base_drive_root, args)
        official = summary.get("official_test", {}) if isinstance(summary, dict) else {}
        gap = summary.get("gt_vs_pred_geometry_gap", {}) if isinstance(summary, dict) else {}
        comparison.append({
            "backbone": bb,
            "status": summary.get("status") if isinstance(summary, dict) else "unknown",
            "det_f": official.get("det_f"),
            "e2e_none_f": official.get("e2e_none_f"),
            "gt_geometry_e2e_none_f": gap.get("gt_geometry_e2e_none_f"),
            "pred_geometry_e2e_none_f": gap.get("pred_geometry_e2e_none_f"),
            "geometry_gap": gap.get("geometry_gap"),
            "output_root": s.get("output_root", ""),
        })
        clean_memory()
    base = Path(base_output_root)
    base.mkdir(parents=True, exist_ok=True)
    save_json(comparison, base / "backbone_comparison_totaltext.json")
    write_csv(comparison, base / "backbone_comparison_totaltext.csv")
    write_backbone_comparison_markdown(comparison, base / "backbone_comparison_totaltext.md")
    print_backbone_result_table(comparison)
    log(f"[compare] wrote {base / 'backbone_comparison_totaltext.csv'}")
# =============================================================================
# 10. MULTI-DATASET SEQUENTIAL RUNNER OVERRIDE
# =============================================================================
#
# This block deliberately overrides only the dataset discovery, manifest building,
# tokenizer construction, and main execution logic. The architecture classes above
# remain untouched: ResNet-50 backbone, FPN P2/P3/P4, DB++ head, ArcFlowAlign,
# ArcBridge, and the CTC/CE recognizer are reused exactly as defined earlier.
from dataclasses import dataclass as _dataclass
@_dataclass
class DatasetRunControl:
    key: str
    pretty: str
    train_flag: bool
    crop_warmup_epochs: int
    main_train_epochs: int
    max_text_len: int
    full_image_batch_size: int
    eval_batch_size: int
    grad_accum_steps: int
    crop_batch_size: int
    crop_eval_batch_size: int
    crop_width: int
    img_size: int
    internal_val_images: int
    arc_points: int = 25
    postprocess_family: str = "curved_word"
    gt_crop_pad: float = 1.06
# -------------------------------------------------------------------------
# User-facing control panel. Edit this in Colab.
# -------------------------------------------------------------------------
RUN_CONTROL: Dict[str, DatasetRunControl] = {
    "total_text": DatasetRunControl(
        key="total_text",
        pretty="Total-Text",
        train_flag=True,
        crop_warmup_epochs=10,
        main_train_epochs=10,
        max_text_len=34,
        full_image_batch_size=3,
        eval_batch_size=24,
        grad_accum_steps=1,
        crop_batch_size=160,
        crop_eval_batch_size=256,
        crop_width=320,
        img_size=640,
        internal_val_images=50,
        arc_points=48,
        postprocess_family="curved_word",
        gt_crop_pad=1.06,
    ),
    "icdar2015": DatasetRunControl(
        key="icdar2015",
        pretty="ICDAR2015",
        train_flag=True,
        crop_warmup_epochs=10,
        main_train_epochs=10,
        max_text_len=34,
        full_image_batch_size=3,
        eval_batch_size=24,
        grad_accum_steps=1,
        crop_batch_size=160,
        crop_eval_batch_size=256,
        crop_width=320,
        img_size=640,
        internal_val_images=50,
        arc_points=32,
        postprocess_family="quad_word",
        gt_crop_pad=1.06,
    ),
    "ctw1500": DatasetRunControl(
        key="ctw1500",
        pretty="CTW1500",
        train_flag=True,
        crop_warmup_epochs=10,
        main_train_epochs=10,
        max_text_len=100,
        full_image_batch_size=2,
        eval_batch_size=16,
        grad_accum_steps=2,
        # V7.18: CTW keeps effective full-image batch via accumulation; OCR crops are wide,
        # so keep crop batches stable and use RAM-safe loaders instead of aggressive prefetch.
        crop_batch_size=64,
        crop_eval_batch_size=128,
        crop_width=640,
        img_size=640,
        internal_val_images=50,
        arc_points=64,
        postprocess_family="curved_line",
        gt_crop_pad=1.22,
    ),
}
def _norm_path_text(p: Path) -> str:
    return str(p).lower().replace("\\", "/").replace("_", "-").replace(" ", "-")
def _contains_any(p: Path, words: Sequence[str]) -> bool:
    s = _norm_path_text(p)
    return all(w.lower().replace("_", "-") in s for w in words)
def _dir_has_images(p: Path) -> bool:
    if not p.is_dir():
        return False
    try:
        return any(x.is_file() and x.suffix.lower() in IMAGE_EXTS for x in p.iterdir())
    except Exception:
        return False
def _dir_has_txts(p: Path) -> bool:
    if not p.is_dir():
        return False
    try:
        return any(x.is_file() and x.suffix.lower() == ".txt" for x in p.iterdir())
    except Exception:
        return False
def _choose_one(label: str, candidates: List[Path], required: bool = True) -> Optional[Path]:
    uniq = []
    seen = set()
    for p in candidates:
        rp = str(p.resolve()) if p.exists() else str(p)
        if rp not in seen:
            uniq.append(p); seen.add(rp)
    if not uniq:
        if required:
            raise FileNotFoundError(f"Could not discover required path for {label}")
        log(f"[path][optional-missing] {label}")
        return None
    uniq = sorted(uniq, key=lambda x: (len(str(x)), str(x)))
    log(f"[path] {label:24s}: {uniq[0]}")
    if len(uniq) > 1:
        log(f"[path][info] {label}: {len(uniq)} candidates found; using shortest deterministic match.")
    return uniq[0]
def discover_dataset_paths(local_root: Path) -> Dict[str, Path]:
    """Discover Total-Text, ICDAR2015, and CTW1500 paths in the extracted tar."""
    section("MULTI-DATASET DISCOVERY")
    paths: Dict[str, Path] = {}
    # Total-Text
    tt_train_img = _choose_one("total_text.train_img", [
        p for p in recursive_find(local_root, lambda q: q.is_dir() and q.name.lower() == "train" and _looks_like_totaltext_path(q))
        if _dir_has_images(p)
    ])
    tt_test_img = _choose_one("total_text.test_img", [
        p for p in recursive_find(local_root, lambda q: q.is_dir() and q.name.lower() == "test" and _looks_like_totaltext_path(q))
        if _dir_has_images(p)
    ])
    tt_train_poly = _choose_one("total_text.train_poly", recursive_find(
        local_root,
        lambda q: q.is_dir() and q.name.lower() == "train"
        and "groundtruth_polygonal_annotation" in str(q).lower()
        and _looks_like_totaltext_path(q)
    ))
    tt_test_poly = _choose_one("total_text.test_poly", recursive_find(
        local_root,
        lambda q: q.is_dir() and q.name.lower() == "test"
        and "groundtruth_polygonal_annotation" in str(q).lower()
        and _looks_like_totaltext_path(q)
    ))
    paths.update({
        "total_text_train_img": tt_train_img,
        "total_text_test_img": tt_test_img,
        "total_text_train_ann": tt_train_poly,
        "total_text_test_ann": tt_test_poly,
    })
    # ICDAR2015
    icdar_roots = [p for p in recursive_find(local_root, lambda q: q.is_dir() and ("icdar2015" in _norm_path_text(q) or "icdar-2015" in _norm_path_text(q)))]
    icdar_scope = icdar_roots if icdar_roots else [local_root]
    def _search_icdar(pred):
        out = []
        for root in icdar_scope:
            out.extend(recursive_find(root, pred))
        return out
    ic_train_img = _choose_one("icdar2015.train_img", [
        p for p in _search_icdar(lambda q: q.is_dir() and _dir_has_images(q) and ("training-images" in _norm_path_text(q) or "ch4-training-images" in _norm_path_text(q) or q.name.lower() in {"training", "train"}))
        if "test" not in _norm_path_text(p)
    ])
    ic_test_img = _choose_one("icdar2015.test_img", [
        p for p in _search_icdar(lambda q: q.is_dir() and _dir_has_images(q) and ("test-images" in _norm_path_text(q) or "ch4-test-images" in _norm_path_text(q) or q.name.lower() in {"test", "testing"}))
    ])
    ic_train_ann = _choose_one("icdar2015.train_ann", [
        p for p in _search_icdar(lambda q: q.is_dir() and _dir_has_txts(q) and (
            "training-localization-transcription-gt" in _norm_path_text(q)
            or "ch4-training-localization" in _norm_path_text(q)
            or ("annotations" in _norm_path_text(q) and ("training" in _norm_path_text(q) or "train" in _norm_path_text(q)))
        ))
        if "test" not in _norm_path_text(p)
    ])
    ic_test_ann = _choose_one("icdar2015.test_ann", [
        p for p in _search_icdar(lambda q: q.is_dir() and _dir_has_txts(q) and (
            # Official/local ICDAR2015 test GT folder names seen in common releases.
            # Your tar uses: ICDAR2015/ch4_test_localization_transcription_gt/
            "ch4-test-localization" in _norm_path_text(q)
            or "test-localization-transcription-gt" in _norm_path_text(q)
            or "challenge4-test-task1-gt" in _norm_path_text(q)
            or "localization-transcription-gt" in _norm_path_text(q) and "test" in _norm_path_text(q)
            or ("annotations" in _norm_path_text(q) and ("test" in _norm_path_text(q) or "testing" in _norm_path_text(q)))
        ))
    ])
    paths.update({
        "icdar2015_train_img": ic_train_img,
        "icdar2015_test_img": ic_test_img,
        "icdar2015_train_ann": ic_train_ann,
        "icdar2015_test_ann": ic_test_ann,
    })
    # CTW1500
    ctw_roots = [p for p in recursive_find(local_root, lambda q: q.is_dir() and ("ctw1500" in _norm_path_text(q) or "scut-ctw" in _norm_path_text(q)))]
    ctw_scope = ctw_roots if ctw_roots else [local_root]
    def _search_ctw(pred):
        out = []
        for root in ctw_scope:
            out.extend(recursive_find(root, pred))
        return out
    ctw_train_img = _choose_one("ctw1500.train_img", [
        p for p in _search_ctw(lambda q: q.is_dir() and _dir_has_images(q) and ("ctwtrain-text-image" in _norm_path_text(q) or ("train" in _norm_path_text(q) and "image" in _norm_path_text(q))))
        if "test" not in _norm_path_text(p)
    ])
    ctw_test_img = _choose_one("ctw1500.test_img", [
        p for p in _search_ctw(lambda q: q.is_dir() and _dir_has_images(q) and ("ctwtest-text-image" in _norm_path_text(q) or ("test" in _norm_path_text(q) and "image" in _norm_path_text(q))))
    ])
    ctw_train_json = _choose_one("ctw1500.train_json", [
        p for p in _search_ctw(lambda q: q.is_file() and q.suffix.lower() == ".json" and "train" in _norm_path_text(q) and "ctw" in _norm_path_text(q))
    ])
    ctw_test_json = _choose_one("ctw1500.test_json", [
        p for p in _search_ctw(lambda q: q.is_file() and q.suffix.lower() == ".json" and "test" in _norm_path_text(q) and "ctw" in _norm_path_text(q))
    ])
    paths.update({
        "ctw1500_train_img": ctw_train_img,
        "ctw1500_test_img": ctw_test_img,
        "ctw1500_train_ann": ctw_train_json,
        "ctw1500_test_ann": ctw_test_json,
    })
    return paths
def _record_image_size(path: Path) -> Tuple[Optional[int], Optional[int]]:
    try:
        with Image.open(path) as im:
            return int(im.width), int(im.height)
    except Exception:
        return None, None
def _bbox_xywh_from_poly(poly: np.ndarray) -> List[float]:
    p = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    x1, y1 = float(p[:, 0].min()), float(p[:, 1].min())
    x2, y2 = float(p[:, 0].max()), float(p[:, 1].max())
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]
def _finalize_instance(poly: Any, text: Optional[str], ignore: bool, source: str, **extra) -> Optional[Dict[str, Any]]:
    arr = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    if arr.shape[0] < 3 or not np.isfinite(arr).all() or polygon_area(arr) <= 1.0:
        return None
    raw = "" if text is None else nfc(str(text))
    obj: Dict[str, Any] = {
        "polygon": arr,
        "bbox_xywh": _bbox_xywh_from_poly(arr),
        "text": raw,
        "text_raw": raw,
        "text_norm": normalize_for_eval(raw, ascii_lower=True),
        "ignore": bool(ignore),
        # text_ignore means: keep geometry for detection, but exclude this instance
        # from OCR tokenizer/crop training and E2E text scoring. This is needed
        # for CTW1500 English-only decoding/OOV token95 handling.
        "text_ignore": bool(extra.pop("text_ignore", False)),
        "text_ignore_reason": str(extra.pop("text_ignore_reason", "")),
        "source": source,
    }
    obj.update(extra)
    return obj
def parse_icdar2015_annotation(path: Path) -> List[Dict[str, Any]]:
    """Parse ICDAR2015 x1,y1,...,x4,y4,text lines. Uses split(',', 8)."""
    try:
        txt = path.read_text(encoding="utf-8-sig", errors="ignore")
    except Exception:
        txt = path.read_text(encoding="latin-1", errors="ignore")
    out: List[Dict[str, Any]] = []
    malformed = 0
    for line in txt.splitlines():
        line = line.strip().replace("\ufeff", "")
        if not line:
            continue
        parts = line.split(",", 8)
        if len(parts) < 9:
            malformed += 1
            continue
        try:
            vals = [float(x) for x in parts[:8]]
        except Exception:
            malformed += 1
            continue
        text = parts[8].strip()
        poly = np.asarray(vals, dtype=np.float32).reshape(4, 2)
        # ICDAR2015 official ignore marker is exact ###.
        # Do NOT ignore valid readable labels that begin with #, e.g. #05-11.
        obj = _finalize_instance(poly, text, text.strip() == "###", "icdar2015")
        if obj is not None:
            out.append(obj)
    if malformed:
        log(f"[icdar2015][warn] malformed={malformed} file={path}")
    return out
def _build_txt_ann_map(ann_dir: Path) -> Dict[str, Path]:
    mp: Dict[str, Path] = {}
    for p in ann_dir.glob("*.txt"):
        stem = p.stem.lower()
        keys = {stem, stem.replace("gt_", ""), stem.replace("gt_img_", "img_"), stem.replace("gt_img", "img")}
        for k in keys:
            mp[k] = p
    return mp
def build_icdar2015_records(img_dir: Path, ann_dir: Path, split: str) -> List[Dict[str, Any]]:
    ann_map = _build_txt_ann_map(ann_dir)
    images = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
    records: List[Dict[str, Any]] = []
    missing = total_inst = ignore_inst = 0
    for img in images:
        stem = img.stem.lower()
        ann = ann_map.get(stem) or ann_map.get("gt_" + stem)
        if ann is None:
            # ICDAR often uses img_1 / gt_img_1 naming; keep a relaxed fallback.
            candidates = [p for k, p in ann_map.items() if stem == k or stem in k or k in stem]
            ann = candidates[0] if candidates else None
        if ann is None:
            missing += 1
            inst = []
        else:
            inst = parse_icdar2015_annotation(ann)
        w, h = _record_image_size(img)
        total_inst += len(inst)
        ignore_inst += sum(1 for x in inst if x.get("ignore", False))
        records.append({
            "image": str(img),
            "image_path": str(img),
            "image_id": f"{split}:{img.stem}",
            "width": w,
            "height": h,
            "instances": inst,
            "split": split,
            "source": "icdar2015",
        })
    log(f"[icdar2015:{split}] images={len(records)} instances={total_inst} ignore={ignore_inst} missing_ann={missing}")
    return records
def _bezier_curve(points: np.ndarray, n: int = 16) -> np.ndarray:
    pts = np.asarray(points, dtype=np.float32).reshape(4, 2)
    t = np.linspace(0.0, 1.0, n, dtype=np.float32)[:, None]
    return ((1 - t) ** 3) * pts[0] + 3 * ((1 - t) ** 2) * t * pts[1] + 3 * (1 - t) * (t ** 2) * pts[2] + (t ** 3) * pts[3]
def _polygon_area_np(poly: np.ndarray) -> float:
    poly = np.asarray(poly, dtype=np.float32)
    if poly.ndim != 2 or poly.shape[0] < 3 or poly.shape[1] != 2:
        return 0.0
    x = poly[:, 0]
    y = poly[:, 1]
    return float(0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))
def ctw_bezier_to_polygon(bezier_pts: Sequence[float], samples_per_side: int = 16) -> np.ndarray:
    vals = np.asarray(bezier_pts, dtype=np.float32).reshape(-1)
    if vals.size != 16:
        return np.zeros((0, 2), dtype=np.float32)
    pts = vals.reshape(8, 2)
    upper = _bezier_curve(pts[:4], samples_per_side)
    lower = _bezier_curve(pts[4:], samples_per_side)
    # Important CTW1500 fix:
    # In this CTW1500 JSON copy, the second boundary is often already stored
    # in the opposite traversal direction. Blindly using lower[::-1] collapses
    # many valid text-line polygons to near-zero area. Build both candidates
    # and keep the physically valid one with larger shoelace area.
    cand_a = np.concatenate([upper, lower], axis=0)
    cand_b = np.concatenate([upper, lower[::-1]], axis=0)
    poly = cand_a if _polygon_area_np(cand_a) >= _polygon_area_np(cand_b) else cand_b
    return poly.astype(np.float32)
def decode_ctw_rec(rec: Any) -> str:
    """Decode CTW1500/ABCNet-style rec arrays for English/Latin recognition.
    Token policy locked from the dataset audit:
      - 0..94 map to printable ASCII via chr(token + 32)
      - 96+ are padding/end markers
      - 95 is OOV/non-English/non-printable in this copy, rendered as � and
        marked text_ignore later so it is not silently deleted for E2E.
    """
    if rec is None:
        return ""
    chars: List[str] = []
    for raw in list(rec):
        try:
            idx = int(raw)
        except Exception:
            continue
        if idx < 0:
            continue
        if idx == 95:
            chars.append("�")
            continue
        if idx >= 96:
            continue
        ch = chr(idx + 32)
        chars.append(ch)
    return nfc("".join(chars)).strip()
def ctw_rec_has_oov_or_nonenglish_marker(rec: Any) -> bool:
    if rec is None:
        return False
    for raw in list(rec):
        try:
            if int(raw) == 95:
                return True
        except Exception:
            continue
    return False
def build_ctw1500_records(img_dir: Path, json_path: Path, split: str) -> List[Dict[str, Any]]:
    data = read_json(json_path)
    images_by_id = {int(im["id"]): im for im in data.get("images", [])}
    anns_by_img: Dict[int, List[Dict[str, Any]]] = defaultdict(list)
    for ann in data.get("annotations", []):
        try:
            anns_by_img[int(ann["image_id"])].append(ann)
        except Exception:
            continue
    records: List[Dict[str, Any]] = []
    total_inst = ignored = undecoded = missing_img = 0
    for image_id, imrec in sorted(images_by_id.items(), key=lambda x: x[0]):
        img = img_dir / str(imrec.get("file_name", ""))
        if not img.exists():
            # Relaxed fallback because some copies include nested folders.
            matches = list(img_dir.rglob(str(imrec.get("file_name", ""))))
            img = matches[0] if matches else img
        if not img.exists():
            missing_img += 1
            continue
        inst: List[Dict[str, Any]] = []
        for ann in anns_by_img.get(image_id, []):
            poly = ctw_bezier_to_polygon(ann.get("bezier_pts", []), samples_per_side=16)
            rec_arr = ann.get("rec")
            text = ann.get("text") or ann.get("transcription") or ann.get("label") or decode_ctw_rec(rec_arr)
            ctw_text_ignore = False
            ctw_text_ignore_reason = ""
            if not text:
                undecoded += 1
                ctw_text_ignore = True
                ctw_text_ignore_reason = "empty_or_undecoded_ctw_rec"
            elif str(text).strip() == "#":
                ctw_text_ignore = True
                ctw_text_ignore_reason = "ctw_hash_label"
            elif ctw_rec_has_oov_or_nonenglish_marker(rec_arr) or "�" in str(text):
                # English-only recognition lock: keep valid CTW geometry for DB++
                # detection learning, but exclude OOV/non-English token95 labels
                # from tokenizer, crop recognizer, and E2E text scoring.
                ctw_text_ignore = True
                ctw_text_ignore_reason = "ctw_token95_oov_nonenglish"
            obj = _finalize_instance(poly, text, str(text).strip() == "#", "ctw1500", text_ignore=ctw_text_ignore, text_ignore_reason=ctw_text_ignore_reason, bezier_pts=ann.get("bezier_pts"), rec=rec_arr, bbox_xywh=ann.get("bbox"))
            if obj is not None:
                inst.append(obj)
        total_inst += len(inst)
        ignored += sum(1 for x in inst if x.get("ignore", False))
        w = int(imrec.get("width") or 0) or _record_image_size(img)[0]
        h = int(imrec.get("height") or 0) or _record_image_size(img)[1]
        records.append({
            "image": str(img),
            "image_path": str(img),
            "image_id": f"{split}:{image_id}",
            "width": w,
            "height": h,
            "instances": inst,
            "split": split,
            "source": "ctw1500",
        })
    log(f"[ctw1500:{split}] images={len(records)} instances={total_inst} ignore={ignored} undecoded_text={undecoded} missing_img={missing_img}")
    if total_inst > 0 and undecoded == total_inst:
        raise RuntimeError(
            "CTW1500 transcription decoding failed for every instance. "
            "Do not run E2E CTW1500 until rec/text mapping is fixed."
        )
    return records
def build_total_text_records_for_multi(img_dir: Path, poly_dir: Path, split: str) -> List[Dict[str, Any]]:
    records = build_totaltext_records(img_dir, poly_dir, split)
    for rec in records:
        rec["image_path"] = rec.get("image")
        rec["image_id"] = f"{split}:{Path(rec['image']).stem}"
        rec["source"] = "total_text"
        w, h = _record_image_size(Path(rec["image"]))
        rec["width"] = w
        rec["height"] = h
        for obj in rec.get("instances", []):
            obj["source"] = "total_text"
            obj["text_raw"] = obj.get("text", "")
            obj["text_norm"] = normalize_for_eval(obj.get("text", ""), ascii_lower=True)
            obj["bbox_xywh"] = _bbox_xywh_from_poly(np.asarray(obj["polygon"], dtype=np.float32))
    return records
def build_all_real_records(paths: Dict[str, Path]) -> Dict[str, Any]:
    """Build all three dataset manifests in one unified schema."""
    records = {
        "total_text_train": build_total_text_records_for_multi(paths["total_text_train_img"], paths["total_text_train_ann"], "train"),
        "total_text_test": build_total_text_records_for_multi(paths["total_text_test_img"], paths["total_text_test_ann"], "test"),
        "icdar2015_train": build_icdar2015_records(paths["icdar2015_train_img"], paths["icdar2015_train_ann"], "train"),
        "icdar2015_test": build_icdar2015_records(paths["icdar2015_test_img"], paths["icdar2015_test_ann"], "test"),
        "ctw1500_train": build_ctw1500_records(paths["ctw1500_train_img"], paths["ctw1500_train_ann"], "train"),
        "ctw1500_test": build_ctw1500_records(paths["ctw1500_test_img"], paths["ctw1500_test_ann"], "test"),
    }
    return records
def _to_jsonable(x: Any) -> Any:
    """Convert numpy / torch / Path scalar containers into JSON-safe Python objects.
    This is intentionally used only for manifest/report export. Training still keeps
    the original in-memory numpy arrays and tensors where the dataloaders expect them.
    """
    if x is None or isinstance(x, (str, int, float, bool)):
        return x
    if isinstance(x, Path):
        return str(x)
    # numpy arrays/scalars
    if hasattr(x, "tolist"):
        try:
            return x.tolist()
        except Exception:
            pass
    # torch tensors
    if hasattr(x, "detach") and hasattr(x, "cpu"):
        try:
            return x.detach().cpu().tolist()
        except Exception:
            pass
    if isinstance(x, dict):
        return {str(k): _to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_to_jsonable(v) for v in x]
    return str(x)
def _json_safe_record(rec: Dict[str, Any]) -> Dict[str, Any]:
    """Return a compact manifest-safe record for reproducibility.
    v6 exports manifests before training. Some parsers keep polygons as numpy
    arrays for fast geometric operations, so every exported field must be converted
    to builtin Python types before json.dumps.
    """
    out = {
        "image_path": _to_jsonable(rec.get("image_path") or rec.get("image")),
        "image_id": _to_jsonable(rec.get("image_id")),
        "width": _to_jsonable(rec.get("width")),
        "height": _to_jsonable(rec.get("height")),
        "split": _to_jsonable(rec.get("split")),
        "source": _to_jsonable(rec.get("source")),
        "instances": [],
    }
    for obj in rec.get("instances", []):
        out["instances"].append({
            "polygon": _to_jsonable(obj.get("polygon")),
            "bbox_xywh": _to_jsonable(obj.get("bbox_xywh")),
            "bezier_pts": _to_jsonable(obj.get("bezier_pts")),
            "text": _to_jsonable(obj.get("text")),
            "text_raw": _to_jsonable(obj.get("text_raw", obj.get("text"))),
            "text_norm": _to_jsonable(obj.get("text_norm")),
            "ignore": bool(obj.get("ignore", False)),
            "text_ignore": bool(obj.get("text_ignore", False)),
            "text_ignore_reason": _to_jsonable(obj.get("text_ignore_reason", "")),
            "source": _to_jsonable(obj.get("source")),
        })
    return out
def write_dataset_manifests(records: Dict[str, Any], out_dir: Path) -> None:
    """v6 manifest-first export. These files freeze the exact parser output used by training."""
    manifest_dir = out_dir / "manifests"
    manifest_dir.mkdir(parents=True, exist_ok=True)
    for key, recs in sorted(records.items()):
        path = manifest_dir / f"{key}.jsonl"
        with path.open("w", encoding="utf-8") as f:
            for rec in recs:
                f.write(json.dumps(_json_safe_record(rec), ensure_ascii=False) + "\n")
        log(f"[manifest] {key}: {len(recs)} records -> {path}")
def sanity_check_records(records: Dict[str, Any]) -> Dict[str, Any]:
    section("MULTI-DATASET MANIFEST / COUNT CHECK")
    expected = {
        "total_text": {"train_images": 1255, "test_images": 300},
        "icdar2015": {"train_images": 1000, "test_images": 500},
        "ctw1500": {"train_images": 1000, "test_images": 500},
    }
    counts: Dict[str, Any] = {}
    for key in ["total_text", "icdar2015", "ctw1500"]:
        tr = records[f"{key}_train"]
        te = records[f"{key}_test"]
        counts[key] = {
            "train_images": len(tr),
            "test_images": len(te),
            "train_instances": sum(len(r.get("instances", [])) for r in tr),
            "test_instances": sum(len(r.get("instances", [])) for r in te),
            "train_usable_instances": sum(1 for r in tr for o in r.get("instances", []) if is_text_usable_instance(o)),
            "test_usable_instances": sum(1 for r in te for o in r.get("instances", []) if is_text_usable_instance(o)),
            "train_ignored_instances": sum(1 for r in tr for o in r.get("instances", []) if o.get("ignore", False)),
            "test_ignored_instances": sum(1 for r in te for o in r.get("instances", []) if o.get("ignore", False)),
            "train_text_ignored_instances": sum(1 for r in tr for o in r.get("instances", []) if o.get("text_ignore", False)),
            "test_text_ignored_instances": sum(1 for r in te for o in r.get("instances", []) if o.get("text_ignore", False)),
        }
        for split in ["train_images", "test_images"]:
            if counts[key][split] != expected[key][split]:
                msg = f"{key}.{split}={counts[key][split]} expected={expected[key][split]}"
                if cfg.REQUIRE_EXPECTED_DATA_COUNTS:
                    raise RuntimeError("Dataset split count check failed: " + msg)
                log("[count][warn] " + msg)
    log(json.dumps(counts, ensure_ascii=False, indent=2))
    return counts
def build_tokenizer_for_dataset(dataset_key: str, train_records: List[Dict[str, Any]], out_root: Path) -> TextTokenizer:
    section(f"TOKENIZER BUILD: {dataset_key}")
    tok_path = out_root / f"tokenizer_{dataset_key}.json"
    if cfg.RESUME and tok_path.exists():
        tokenizer = TextTokenizer.from_dict(read_json(tok_path))
        log(f"[tokenizer:{dataset_key}] loaded existing vocab={tokenizer.vocab_size}")
        return tokenizer
    texts: List[str] = []
    for rec in train_records:
        for obj in rec.get("instances", []):
            if obj.get("ignore", False) or obj.get("text_ignore", False):
                continue
            word = normalize_for_eval(obj.get("text", ""), ascii_lower=True)
            if word:
                texts.append(word)
    if len(texts) == 0:
        raise RuntimeError(f"No usable recognition labels for {dataset_key}. Check annotation parser/ignore policy.")
    tokenizer = TextTokenizer(min_freq=cfg.VOCAB_MIN_FREQ)
    tokenizer.build(texts, [f"{dataset_key}_latin"] * len(texts))
    save_json(tokenizer.to_dict(), tok_path)
    save_json({f"{dataset_key}_latin": 0}, out_root / f"domain_to_id_{dataset_key}.json")
    log(f"[tokenizer:{dataset_key}] train_words={len(texts)} vocab_size={tokenizer.vocab_size}")
    return tokenizer
def _configure_runtime_for_dataset(dc: DatasetRunControl, base_output_root: str, base_drive_root: str, args: argparse.Namespace) -> None:
    cfg.DRIVE_TAR = args.dataset_tar
    cfg.BACKBONE_NAME = "resnet50"
    cfg.PRETRAIN_IMAGENET = True
    cfg.EPOCHS_TT_CROP_WARMUP = int(dc.crop_warmup_epochs)
    cfg.EPOCHS_TT_DB_ARCFLOW = int(dc.main_train_epochs)
    cfg.MAX_TEXT_LEN = int(dc.max_text_len)
    cfg.ARCFLOW_K_POINTS = int(dc.arc_points)
    cfg.CURRENT_POSTPROCESS_FAMILY = str(dc.postprocess_family)
    cfg.TT_DB_BATCH_SIZE = int(dc.full_image_batch_size)
    cfg.TT_DB_EVAL_BATCH_SIZE = int(dc.eval_batch_size)
    cfg.GRAD_ACCUM_STEPS = int(dc.grad_accum_steps)
    cfg.TT_CROP_BATCH_SIZE = int(dc.crop_batch_size)
    cfg.TT_CROP_EVAL_BATCH_SIZE = int(dc.crop_eval_batch_size)
    cfg.OCR_FIXED_W = int(dc.crop_width)
    cfg.TOTALTEXT_IMG_SIZE = int(dc.img_size)
    cfg.TT_DB_INTERNAL_VAL_IMAGES = int(dc.internal_val_images)
    # V7 final real-data baseline preprocessing/postprocess. Training engine remains unified;
    # only crop width and validation sweep ranges are adjusted to dataset geometry.
    if dc.key == "ctw1500":
        cfg.TT_DB_GEOM_SWEEP_THRESHOLDS = (0.55, 0.60, 0.65, 0.70, 0.75)
        cfg.TT_DB_GEOM_SWEEP_UNSHRINK = (1.55, 1.75, 1.95, 2.15)
        cfg.TT_DB_GEOM_SWEEP_MODES = ("contour", "minrect")
        cfg.TT_DB_GEOM_SWEEP_MIN_AREAS = (32, 48, 64, 96, 128)
        cfg.TT_DB_GEOM_SWEEP_MORPH = ("none", "close")
        cfg.TT_DB_GEOM_SWEEP_CROP_PAD = (1.00, 1.12, 1.22, 1.34)
        cfg.TT_CROP_RECTIFY_PAD = float(dc.gt_crop_pad)
        cfg.TT_CROP_MODES = ("rectified",)
        cfg.TT_CROP_MODE_WEIGHTS = (1.0,)
    elif dc.key == "icdar2015":
        cfg.TT_DB_GEOM_SWEEP_THRESHOLDS = (0.60, 0.65, 0.70, 0.75, 0.80)
        cfg.TT_DB_GEOM_SWEEP_UNSHRINK = (1.20, 1.35, 1.55, 1.75)
        cfg.TT_DB_GEOM_SWEEP_MODES = ("minrect", "contour")
        cfg.TT_DB_GEOM_SWEEP_MIN_AREAS = (16, 24, 32, 48, 64)
        cfg.TT_DB_GEOM_SWEEP_MORPH = ("none",)
        cfg.TT_DB_GEOM_SWEEP_CROP_PAD = (1.00, 1.06, 1.12)
        cfg.TT_CROP_RECTIFY_PAD = float(dc.gt_crop_pad)
        cfg.TT_CROP_MODES = ("rectified",)
        cfg.TT_CROP_MODE_WEIGHTS = (1.0,)
    else:
        cfg.TT_DB_GEOM_SWEEP_THRESHOLDS = (0.60, 0.65, 0.70, 0.75, 0.80)
        cfg.TT_DB_GEOM_SWEEP_UNSHRINK = (1.35, 1.55, 1.75)
        cfg.TT_DB_GEOM_SWEEP_MODES = ("contour", "minrect")
        cfg.TT_DB_GEOM_SWEEP_MIN_AREAS = (32, 48, 64)
        cfg.TT_DB_GEOM_SWEEP_MORPH = ("none", "close")
        cfg.TT_DB_GEOM_SWEEP_CROP_PAD = (1.00, 1.08, 1.16)
        cfg.TT_CROP_RECTIFY_PAD = float(dc.gt_crop_pad)
        cfg.TT_CROP_MODES = ("rectified",)
        cfg.TT_CROP_MODE_WEIGHTS = (1.0,)
    cfg.NUM_WORKERS = int(args.num_workers)
    cfg.AMP_DTYPE = str(args.amp_dtype)
    cfg.SYNC_TO_DRIVE_AT_END = not bool(args.no_sync)
    cfg.RESUME = bool(args.resume)
    cfg.CLEAN_OUTPUT_ROOT = not bool(args.no_clean)
    cfg.TT_DB_ENABLE_E2E_SELECTION = not bool(getattr(args, "disable_e2e_selection", False))
    cfg.DB_IGNORE_MODE = str(getattr(args, "db_ignore_mode", "v4_default"))
    # V7.18 paper-full RAM-stable throughput default: keep predicted-crop adaptation enabled for paper runs.
    # The builder is now batched + RAM-safe, so memory protection happens inside the stage
    # instead of silently disabling a useful modeling/evaluation path.
    cfg.ENABLE_PRED_CROP_ADAPT = not bool(getattr(args, "disable_pred_crop_adapt", False))
    cfg.PRED_CROP_ADAPT_EPOCHS = int(getattr(args, "pred_crop_adapt_epochs", cfg.PRED_CROP_ADAPT_EPOCHS))
    cfg.PRED_CROP_ADAPT_MAX_IMAGES = int(getattr(args, "pred_crop_adapt_max_images", cfg.PRED_CROP_ADAPT_MAX_IMAGES))
    cfg.PRED_CROP_ADAPT_BUILDER_BATCH_SIZE = int(getattr(args, "pred_crop_builder_batch_size", cfg.PRED_CROP_ADAPT_BUILDER_BATCH_SIZE))
    cfg.PRED_CROP_ADAPT_TRAIN_BATCH_CAP = int(getattr(args, "pred_crop_train_batch_cap", cfg.PRED_CROP_ADAPT_TRAIN_BATCH_CAP))
    cfg.ENABLE_PAPER_VISUALS = not bool(getattr(args, "no_paper_visuals", False))
    # Keep full architecture intact. Only run metadata and paths change.
    cfg.OUTPUT_ROOT = str(Path(base_output_root) / dc.key)
    cfg.DRIVE_SYNC_ROOT = str(Path(base_drive_root) / dc.key)
    cfg.ARCH_VERSION = f"arcf_multidataset_resnet50_locked_{dc.key}"
    cfg.CURRENT_DATASET_KEY = dc.key
    cfg.CURRENT_DATASET_PRETTY = dc.pretty
    cfg.CURRENT_DATASET_TAG = {"total_text": "TotalText", "icdar2015": "ICDAR2015", "ctw1500": "CTW1500"}.get(dc.key, dc.key)
    apply_t4_runtime_profile(dc.key)
    # Respect CLI overrides, but cap them to RAM-safe limits for this exact Simple T4.
    # The previous 64-image full-eval/builder setting did not improve wall time and
    # drove CPU RAM below 1GB on CTW. OOM fallback protects CUDA, but it cannot
    # save the Colab host if worker queues exhaust CPU RAM.
    safe_eval_cap = 16 if dc.key == "ctw1500" else 24
    safe_builder_cap = 12 if dc.key == "ctw1500" else 24
    safe_crop_eval_cap = 128 if dc.key == "ctw1500" else 256
    safe_pred_train_cap = 64 if dc.key == "ctw1500" else 192
    if getattr(args, "eval_batch_size", None) is not None:
        cfg.TT_DB_EVAL_BATCH_SIZE = min(max(1, int(args.eval_batch_size)), safe_eval_cap)
    if getattr(args, "pred_crop_builder_batch_size", None) is not None:
        cfg.PRED_CROP_ADAPT_BUILDER_BATCH_SIZE = min(max(1, int(args.pred_crop_builder_batch_size)), safe_builder_cap)
    if getattr(args, "crop_eval_batch_size", None) is not None:
        cfg.TT_CROP_EVAL_BATCH_SIZE = min(max(1, int(args.crop_eval_batch_size)), safe_crop_eval_cap)
    if getattr(args, "pred_crop_train_batch_cap", None) is not None:
        cfg.PRED_CROP_ADAPT_TRAIN_BATCH_CAP = min(max(1, int(args.pred_crop_train_batch_cap)), safe_pred_train_cap)
def _assert_learning_signal(summary: Dict[str, Any], dataset_key: str, out_root: Path) -> Dict[str, Any]:
    """Check that loss decreases at least weakly and final eval is non-empty."""
    run_dir = Path(cfg.OUTPUT_ROOT) / f"{dataset_key}_full_image_run"
    db_csv = run_dir / "db_training_metrics.csv"
    crop_csv = run_dir / "tt_crop_recognizer" / "crop_metrics.csv"
    report: Dict[str, Any] = {"dataset": dataset_key, "db_csv": str(db_csv), "crop_csv": str(crop_csv)}
    try:
        rows = []
        if db_csv.exists():
            with open(db_csv, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                rows = list(reader)
        if rows:
            first_loss = float(rows[0].get("loss", "nan"))
            last_loss = float(rows[-1].get("loss", "nan"))
            report["db_first_loss"] = first_loss
            report["db_last_loss"] = last_loss
            report["db_loss_decreased"] = bool(np.isfinite(first_loss) and np.isfinite(last_loss) and last_loss < first_loss)
        else:
            report["db_loss_decreased"] = False
            report["reason"] = "no db_training_metrics.csv rows"
    except Exception as e:
        report["db_loss_decreased"] = False
        report["exception"] = repr(e)
    official = summary.get("official_test", {}) if isinstance(summary, dict) else {}
    report["det_f"] = official.get("det_f")
    report["e2e_none_f"] = official.get("e2e_none_f")
    report["nonzero_detection"] = bool(float(official.get("det_f", 0.0) or 0.0) > 0.0)
    save_json(report, out_root / f"{dataset_key}_learning_signal_report.json")
    if not report.get("db_loss_decreased", False):
        log(f"[learning][warn] {dataset_key}: DB loss did not strictly decrease over the short run. Inspect {db_csv}.")
    if not report.get("nonzero_detection", False):
        log(f"[learning][warn] {dataset_key}: final Det-F is zero/non-positive in this short run.")
    return report
def run_one_dataset(dc: DatasetRunControl, records_all: Dict[str, Any], base_output_root: str, base_drive_root: str, args: argparse.Namespace) -> Dict[str, Any]:
    section(f"START DATASET RUN: {dc.pretty}")
    clean_memory()
    _configure_runtime_for_dataset(dc, base_output_root, base_drive_root, args)
    set_seed(cfg.SEED)
    out_root = prepare_output_root()
    memory_safety_barrier(f"{dc.key}-after-config", aggressive=True, raise_on_critical=True)
    save_json(runtime_profile_snapshot(), out_root / "runtime_memory_profile.json")
    train_records = records_all[f"{dc.key}_train"]
    test_records = records_all[f"{dc.key}_test"]
    dataset_counts = {
        "dataset": dc.key,
        "pretty": dc.pretty,
        "train_images": len(train_records),
        "test_images": len(test_records),
        "train_instances": sum(len(r.get("instances", [])) for r in train_records),
        "test_instances": sum(len(r.get("instances", [])) for r in test_records),
        "train_usable_instances": sum(1 for r in train_records for o in r.get("instances", []) if is_text_usable_instance(o)),
        "test_usable_instances": sum(1 for r in test_records for o in r.get("instances", []) if is_text_usable_instance(o)),
        "train_detection_positives": sum(1 for r in train_records for o in r.get("instances", []) if not o.get("ignore", False)),
        "test_detection_positives": sum(1 for r in test_records for o in r.get("instances", []) if not o.get("ignore", False)),
        "train_text_ignored_instances": sum(1 for r in train_records for o in r.get("instances", []) if o.get("text_ignore", False)),
        "test_text_ignored_instances": sum(1 for r in test_records for o in r.get("instances", []) if o.get("text_ignore", False)),
        "crop_warmup_epochs": int(dc.crop_warmup_epochs),
        "main_train_epochs": int(dc.main_train_epochs),
        "max_text_len": int(dc.max_text_len),
        "full_image_batch_size": int(dc.full_image_batch_size),
        "eval_batch_size": int(dc.eval_batch_size),
        "grad_accum_steps": int(dc.grad_accum_steps),
        "db_ignore_mode": str(getattr(cfg, "DB_IGNORE_MODE", "v4_default")),
        "crop_batch_size": int(dc.crop_batch_size),
        "crop_width": int(dc.crop_width),
        "img_size": int(dc.img_size),
        "arc_points": int(dc.arc_points),
        "postprocess_family": str(dc.postprocess_family),
        "gt_crop_pad": float(dc.gt_crop_pad),
        "backbone": cfg.BACKBONE_NAME,
    }
    save_json(dataset_counts, out_root / f"{dc.key}_dataset_run_config.json")
    write_dataset_manifests({f"{dc.key}_train": train_records, f"{dc.key}_test": test_records}, out_root)
    tokenizer = build_tokenizer_for_dataset(dc.key, train_records, out_root)
    # Unified training engine: every dataset is presented through the same
    # internal train/test record keys consumed by the locked architecture. Only
    # dataset setup/logging/evaluation policy differs; optimizer, crop warm-up,
    # DB++/ArcFlow/ArcBridge training, AMP, batching, and checkpoint logic are shared.
    run_records = {"tt_train": train_records, "tt_test": test_records}
    summary = train_totaltext(tokenizer, run_records)
    summary["dataset_key"] = dc.key
    summary["dataset_pretty"] = dc.pretty
    summary["dataset_counts"] = dataset_counts
    summary["output_root"] = str(out_root)
    summary["protocol_report"] = {
        "detection": "Polygon/quad IoU > 0.5, Precision/Recall/Hmean",
        "e2e_none": "IoU > 0.5 and exact normalized transcription match, lexicon-free",
        "icdar2015_extra": "Reports None plus S/W diagnostic proxy tables; real official S/W/G requires release lexicon files. ICDAR ignore is exact ### only.",
        "total_text_extra": "Reports None and Full lexicon, where Full is built from all evaluation text instances and ED<=2 correction is applied.",
        "ctw1500_extra": "Reports None and Full lexicon; CTW uses long-line crop width and NED<=0.3 full-lexicon correction.",
    }
    save_json(summary, out_root / f"{dc.key}_final_summary.json")
    learning = _assert_learning_signal(summary, dc.key, out_root)
    summary["learning_signal"] = learning
    if cfg.SYNC_TO_DRIVE_AT_END:
        sync_outputs_to_drive(out_root)
    clean_memory()
    return summary
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="ArcFlowText V7.17 paper-full max-throughput safe multi-dataset runner, architecture locked to ResNet-50")
    parser.add_argument("--dataset_tar", type=str, default=cfg.DRIVE_TAR)
    parser.add_argument("--output_root", type=str, default="/content/arcflow_runs/ArcFlowText_3Dataset_ResNet50_v7")
    parser.add_argument("--drive_sync_root", type=str, default="/content/drive/MyDrive/ArcFlowText/ArcFlowText_3Dataset_ResNet50_v7")
    parser.add_argument("--train_total_text", action=argparse.BooleanOptionalAction, default=RUN_CONTROL["total_text"].train_flag)
    parser.add_argument("--train_icdar2015", action=argparse.BooleanOptionalAction, default=RUN_CONTROL["icdar2015"].train_flag)
    parser.add_argument("--train_ctw1500", action=argparse.BooleanOptionalAction, default=RUN_CONTROL["ctw1500"].train_flag)
    parser.add_argument("--total_text_crop_epochs", type=int, default=RUN_CONTROL["total_text"].crop_warmup_epochs)
    parser.add_argument("--total_text_main_epochs", type=int, default=RUN_CONTROL["total_text"].main_train_epochs)
    parser.add_argument("--icdar2015_crop_epochs", type=int, default=RUN_CONTROL["icdar2015"].crop_warmup_epochs)
    parser.add_argument("--icdar2015_main_epochs", type=int, default=RUN_CONTROL["icdar2015"].main_train_epochs)
    parser.add_argument("--ctw1500_crop_epochs", type=int, default=RUN_CONTROL["ctw1500"].crop_warmup_epochs)
    parser.add_argument("--ctw1500_main_epochs", type=int, default=RUN_CONTROL["ctw1500"].main_train_epochs)
    parser.add_argument("--num_workers", type=int, default=cfg.NUM_WORKERS)
    parser.add_argument("--amp_dtype", type=str, default=cfg.AMP_DTYPE, choices=["float16", "bfloat16", "float32"])
    parser.add_argument("--no_auto_t4_profile", action="store_true", help="Disable automatic low-RAM Tesla T4 runtime clamps. Not recommended for Simple T4 final runs.")
    parser.add_argument("--no_sync", action="store_true")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--no_clean", action="store_true")
    parser.add_argument("--disable_e2e_selection", action="store_true", help="Disable train-internal E2E-aware postprocess selection; falls back to detection-only selection.")
    parser.add_argument("--db_ignore_mode", type=str, default="v4_default", choices=["v4_default", "soft_neutral", "hard_neutral"], help="DB loss ignore-region policy. V7 default restores v4 detector strength.")
    parser.add_argument("--enable_pred_crop_adapt", action="store_true", help="Explicitly keep predicted-crop recognizer adaptation enabled. It is enabled by default in V7.18 paper-full RAM-stable throughput mode.")
    parser.add_argument("--disable_pred_crop_adapt", action="store_true", help="Force-disable predicted-crop recognizer adaptation stage for emergency debugging only.")
    parser.add_argument("--pred_crop_adapt_epochs", type=int, default=cfg.PRED_CROP_ADAPT_EPOCHS)
    parser.add_argument("--pred_crop_adapt_max_images", type=int, default=cfg.PRED_CROP_ADAPT_MAX_IMAGES, help="Cap source images for predicted-crop adaptation; 0 means all selected split images.")
    parser.add_argument("--pred_crop_builder_batch_size", type=int, default=cfg.PRED_CROP_ADAPT_BUILDER_BATCH_SIZE, help="Full-image batch size for building predicted-crop adaptation records. Uses automatic OOM splitting.")
    parser.add_argument("--pred_crop_train_batch_cap", type=int, default=cfg.PRED_CROP_ADAPT_TRAIN_BATCH_CAP, help="Upper cap for predicted-crop adaptation OCR train batch size.")
    parser.add_argument("--no_paper_visuals", action="store_true", help="Disable qualitative paper visualizations for speed/debug runs.")
    parser.add_argument("--disable_gap_eval", action="store_true", help="Disable GT-vs-pred geometry-gap monitoring. V7.17 fuses gap monitoring into the final official eval by default for speed.")
    parser.add_argument("--disable_fused_gap_eval", action="store_true", help="Run gap monitoring as a separate pass instead of fusing it into final official eval. Slower; use only for debugging.")
    parser.add_argument("--strict_gap_eval", action="store_true", help="Raise if gap monitoring fails instead of saving partial diagnostics and continuing.")
    parser.add_argument("--db_batch_size", type=int, default=None, help="Override full-image DB training batch size for all selected datasets. Try 3 on T4; use 2 if unstable.")
    parser.add_argument("--eval_batch_size", type=int, default=None, help="Override full-image DB validation/test batch size. V7.18 RAM-stable default is capped per dataset on Simple T4; CUDA OOM splitting remains active.")
    parser.add_argument("--crop_batch_size", type=int, default=None, help="Override crop-recognizer train batch size for all selected datasets.")
    parser.add_argument("--crop_eval_batch_size", type=int, default=None, help="Override crop-recognizer eval batch size for all selected datasets.")
    return parser.parse_args()
def _selected_run_controls(args: argparse.Namespace) -> List[DatasetRunControl]:
    controls = {
        "total_text": RUN_CONTROL["total_text"],
        "icdar2015": RUN_CONTROL["icdar2015"],
        "ctw1500": RUN_CONTROL["ctw1500"],
    }
    controls["total_text"].train_flag = bool(args.train_total_text)
    controls["total_text"].crop_warmup_epochs = int(args.total_text_crop_epochs)
    controls["total_text"].main_train_epochs = int(args.total_text_main_epochs)
    controls["icdar2015"].train_flag = bool(args.train_icdar2015)
    controls["icdar2015"].crop_warmup_epochs = int(args.icdar2015_crop_epochs)
    controls["icdar2015"].main_train_epochs = int(args.icdar2015_main_epochs)
    controls["ctw1500"].train_flag = bool(args.train_ctw1500)
    controls["ctw1500"].crop_warmup_epochs = int(args.ctw1500_crop_epochs)
    controls["ctw1500"].main_train_epochs = int(args.ctw1500_main_epochs)
    # Optional throughput overrides. These do not alter model logic or metrics.
    if getattr(args, "db_batch_size", None) is not None:
        for dc in controls.values():
            dc.full_image_batch_size = int(args.db_batch_size)
    if getattr(args, "eval_batch_size", None) is not None:
        for dc in controls.values():
            dc.eval_batch_size = int(args.eval_batch_size)
    if getattr(args, "crop_batch_size", None) is not None:
        for dc in controls.values():
            dc.crop_batch_size = int(args.crop_batch_size)
    if getattr(args, "crop_eval_batch_size", None) is not None:
        for dc in controls.values():
            dc.crop_eval_batch_size = int(args.crop_eval_batch_size)
    ordered = [controls["total_text"], controls["icdar2015"], controls["ctw1500"]]
    return [x for x in ordered if x.train_flag]
def write_multidataset_summary(summaries: List[Dict[str, Any]], base_output_root: Path) -> None:
    rows = []
    for s in summaries:
        official = s.get("official_test", {}) if isinstance(s, dict) else {}
        crop = s.get("crop_summary", {}).get("final", {}) if isinstance(s, dict) else {}
        learning = s.get("learning_signal", {}) if isinstance(s, dict) else {}
        off_bundle = official.get("official_reporting", {}) if isinstance(official, dict) else {}
        full_m = off_bundle.get("full", {}) if isinstance(off_bundle, dict) else {}
        strong_proxy = off_bundle.get("strong_proxy", {}) if isinstance(off_bundle, dict) else {}
        weak_proxy = off_bundle.get("weak_proxy", {}) if isinstance(off_bundle, dict) else {}
        rows.append({
            "dataset": s.get("dataset_key"),
            "status": s.get("status"),
            "crop_final_WA": crop.get("WA"),
            "crop_final_edit_le1": crop.get("edit_le1"),
            "det_p": official.get("det_precision"),
            "det_r": official.get("det_recall"),
            "det_f": official.get("det_f"),
            "e2e_none_p": official.get("e2e_none_precision"),
            "e2e_none_r": official.get("e2e_none_recall"),
            "e2e_none_f": official.get("e2e_none_f"),
            "e2e_full_f": full_m.get("e2e_none_f"),
            "gt_geometry_e2e_f": (s.get("gt_vs_pred_geometry_gap", {}) or {}).get("gt_geometry_e2e_none_f") if isinstance(s.get("gt_vs_pred_geometry_gap", {}), dict) else None,
            "pred_geometry_e2e_f": (s.get("gt_vs_pred_geometry_gap", {}) or {}).get("predicted_geometry_e2e_none_f") if isinstance(s.get("gt_vs_pred_geometry_gap", {}), dict) else None,
            "geometry_gap": (s.get("gt_vs_pred_geometry_gap", {}) or {}).get("geometry_to_recognition_gap") if isinstance(s.get("gt_vs_pred_geometry_gap", {}), dict) else None,
            "icdar_strong_proxy_f": strong_proxy.get("e2e_none_f"),
            "icdar_weak_proxy_f": weak_proxy.get("e2e_none_f"),
            "db_loss_decreased": learning.get("db_loss_decreased"),
            "output_root": s.get("output_root", ""),
        })
    base_output_root.mkdir(parents=True, exist_ok=True)
    save_json(summaries, base_output_root / "multidataset_full_summaries.json")
    write_csv(rows, base_output_root / "multidataset_results_table.csv")
    md = ["# ArcFlowText V7 profile-driven multi-dataset ResNet-50 run", ""]
    md.append("| Dataset | Status | Det-F | E2E None-F | Full/ICDAR proxy | GT-E2E | Pred-E2E | Gap | Crop WA | DB loss ↓ |")
    md.append("|---|---|---:|---:|---:|---:|---:|---:|---:|---|")
    for r in rows:
        full_or_proxy = r.get('e2e_full_f') if r.get('e2e_full_f') is not None else (r.get('icdar_weak_proxy_f') if r.get('icdar_weak_proxy_f') is not None else '')
        md.append(
            f"| {r.get('dataset')} | {r.get('status')} | {_fmt_metric(r.get('det_f'))} | "
            f"{_fmt_metric(r.get('e2e_none_f'))} | {_fmt_metric(full_or_proxy)} | "
            f"{_fmt_metric(r.get('gt_geometry_e2e_f'))} | {_fmt_metric(r.get('pred_geometry_e2e_f'))} | {_fmt_metric(r.get('geometry_gap'))} | "
            f"{_fmt_metric(r.get('crop_final_WA'))} | {r.get('db_loss_decreased')} |"
        )
    (base_output_root / "multidataset_results_summary.md").write_text("\n".join(md), encoding="utf-8")
    log(f"[multi] wrote {base_output_root / 'multidataset_results_table.csv'}")
def main() -> None:
    args = parse_args()
    cfg.AUTO_T4_SAFE_PROFILE = not bool(getattr(args, "no_auto_t4_profile", False))
    cfg.ENABLE_GEOMETRY_GAP_EVAL = not bool(getattr(args, "disable_gap_eval", False))
    cfg.FUSE_GAP_WITH_OFFICIAL_EVAL = not bool(getattr(args, "disable_fused_gap_eval", False))
    cfg.STRICT_GAP_EVAL = bool(getattr(args, "strict_gap_eval", False))
    selected = _selected_run_controls(args)
    if not selected:
        raise RuntimeError("No dataset selected. Enable at least one of train_total_text/train_icdar2015/train_ctw1500.")
    print_hardware()
    memory_safety_barrier("main-start", aggressive=True, raise_on_critical=True)
    section("ARCFLOWTEXT V7.18 MULTI-DATASET CONTROL PANEL")
    log(f"Dataset tar             : {args.dataset_tar}")
    log("Architecture lock       : ResNet-50 + FPN P2/P3/P4 + DB++ + ArcFlowAlign + ArcBridge")
    log("Dataset policy          : unified train engine; dataset-specific parser/profile/reporting only")
    log("CTW1500 OCR policy      : English/Latin only; token95/OOV/non-English kept for detection geometry, ignored for OCR/E2E")
    log("v7.18 paper-full RAM-stable throughput : all monitoring/eval enabled + RAM-aware batching + fused official/gap eval + OOM backoff")
    log(f"Selected datasets       : {[x.key for x in selected]}")
    for dc in selected:
        log(
            f"[control] {dc.key}: crop_warmup={dc.crop_warmup_epochs}, "
            f"main_train={dc.main_train_epochs}, batch={dc.full_image_batch_size}, "
            f"accum={dc.grad_accum_steps}, max_text_len={dc.max_text_len}, "
            f"crop_w={dc.crop_width}, arcK={dc.arc_points}, profile={dc.postprocess_family}"
        )
    cfg.DRIVE_TAR = args.dataset_tar
    cfg.LOCAL_ROOT = "/content/data/ArcFlowText_3Dataset_extracted"
    local_root = copy_and_extract_dataset()
    paths = discover_dataset_paths(local_root)
    records_all = build_all_real_records(paths)
    base_output_root = Path(args.output_root)
    base_output_root.mkdir(parents=True, exist_ok=True)
    write_dataset_manifests(records_all, base_output_root)
    counts = sanity_check_records(records_all)
    save_json(counts, base_output_root / "dataset_manifest_counts.json")
    summaries: List[Dict[str, Any]] = []
    for dc in selected:
        try:
            summary = run_one_dataset(dc, records_all, args.output_root, args.drive_sync_root, args)
            summaries.append(summary)
        finally:
            clean_memory()
    write_multidataset_summary(summaries, base_output_root)
    section("DONE: MULTI-DATASET SEQUENTIAL RUN")
    log(f"Summary table: {base_output_root / 'multidataset_results_table.csv'}")
if __name__ == "__main__":
    # Colab notebooks sometimes inject `-f <kernel.json>`. Remove only those
    # kernel arguments while preserving real CLI flags when this file is run with !python.
    if any(str(x).endswith('.json') and '/jupyter' in str(x).lower() for x in sys.argv):
        cleaned = [sys.argv[0]]
        skip_next = False
        for a in sys.argv[1:]:
            if skip_next:
                skip_next = False
                continue
            if a == "-f":
                skip_next = True
                continue
            cleaned.append(a)
        sys.argv = cleaned
    main()



HARDWARE / RUNTIME CHECK
[2026-05-22 20:43:07] Python       : 3.12.13
[2026-05-22 20:43:07] PyTorch      : 2.10.0+cu128
[2026-05-22 20:43:07] Torchvision  : 0.25.0+cu128
[2026-05-22 20:43:07] Device       : cuda
[2026-05-22 20:43:07] GPU          : Tesla T4
[2026-05-22 20:43:07] VRAM         : 14.56 GB
[2026-05-22 20:43:07] Capability   : 7.5
[2026-05-22 20:43:07] AMP dtype    : torch.float16
[2026-05-22 20:43:07] CPU count    : 2
[2026-05-22 20:43:07] Output root  : /content/arcflow_runs/ArcFlowText_TotalText_CameraReady
[2026-05-22 20:43:07] Arch version : totaltext_fullimage_10crop_10db_arcflow_arcbridge_vitae_official
[2026-05-22 20:43:07] [mem-guard][main-start] cleanup freeVRAM=14.46GB
[2026-05-22 20:43:07] [mem-guard:main-start] VRAM allocated=0.00GB reserved=0.00GB free=14.46GB peak=0.00GB / 14.56GB | CPU available=9.96GB used=21.4%

ARCFLOWTEXT V7.18 MULTI-DATASET CONTROL PANEL
[2026-05-22 20:43:07] Dataset tar             : /content/drive/MyDrive/ArcFlowText/ArcFlowText_DATA

/tmp/ipykernel_6595/3748497040.py:760: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest)


[2026-05-22 20:43:54] [extract] complete in 0.2min

MULTI-DATASET DISCOVERY
[2026-05-22 20:43:54] [path] total_text.train_img    : /content/data/ArcFlowText_3Dataset_extracted/ArcFlowText_DATASET/Total-Text/Train
[2026-05-22 20:43:54] [path][info] total_text.train_img: 3 candidates found; using shortest deterministic match.
[2026-05-22 20:43:54] [path] total_text.test_img     : /content/data/ArcFlowText_3Dataset_extracted/ArcFlowText_DATASET/Total-Text/Test
[2026-05-22 20:43:54] [path][info] total_text.test_img: 3 candidates found; using shortest deterministic match.
[2026-05-22 20:43:54] [path] total_text.train_poly   : /content/data/ArcFlowText_3Dataset_extracted/ArcFlowText_DATASET/Total-Text/Annotation/groundtruth_polygonal_annotation/Train
[2026-05-22 20:43:54] [path] total_text.test_poly    : /content/data/ArcFlowText_3Dataset_extracted/ArcFlowText_DATASET/Total-Text/Annotation/groundtruth_polygonal_annotation/Test
[2026-05-22 20:43:54] [path] icdar2015.train_img     : /content/d

100%|██████████| 97.8M/97.8M [00:00<00:00, 168MB/s]


[2026-05-22 20:44:09] [model] backbone_backend=torchvision_resnet selected=resnet50 out_channels={'c2': 256, 'c3': 512, 'c4': 1024, 'c5': 2048}
[2026-05-22 20:44:09] [TotalText-Crop] params total=29.66M trainable=29.66M
[2026-05-22 20:44:14] [model] backbone_feature_shapes backend=torchvision_resnet selected=resnet50 shapes={'c2': [160, 256, 16, 80], 'c3': [160, 512, 8, 40], 'c4': [160, 1024, 4, 20], 'c5': [160, 2048, 2, 10]}
[2026-05-22 20:44:19] [TotalText-Crop][ep 1/10 step 1/58] loss=25.9238 ctc=25.1351 ce=3.9438 domain=0.0000 sem=0.0000 grad=inf
[2026-05-22 20:44:19] [TotalText-Crop] VRAM allocated=0.18GB reserved=0.46GB free=13.95GB peak=5.02GB / 14.56GB | CPU available=8.74GB used=31.0%
[2026-05-22 20:44:40] [TotalText-Crop][ep 1/10 step 10/58] loss=4.8763 ctc=4.2234 ce=3.2645 domain=0.0000 sem=0.0000 grad=5.75
[2026-05-22 20:44:40] [TotalText-Crop] VRAM allocated=0.24GB reserved=3.97GB free=10.44GB peak=5.02GB / 14.56GB | CPU available=8.92GB used=29.6%
[2026-05-22 20:45:05] [T